In [ ]:
#추가분석이 되나요? 강건성분석에 머물지 말고 (1) GX 특허 기업의 과학은 과거를 따라간다는데 특히 어떤 기업의 과학이 더 과거를 따라가는지? (2) 감소의 정체: 단독 연구의 소멸은 어느 세부 분야에서 더 가속화되는지? (3) 해외 대학 협력은 늘었지만 외부 주도로만 늘었다는데 어느나라가 더 가속화하는지? (4)  새 파트너는 새 주제 진입과 함께 나타난다는데 어느분야가 더 그러는지? (5) 협력의 재편은 지리가 아니라 파트너 유형에서 일어났다는데 구체적으로 어떻게 지리효과가 없는지? 분해해서 있다면 어느 효과인지?

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf
import unicodedata, warnings, traceback
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 120)

OUT = "oa_out"; BLK = 5; MINP = 10; SEED = 0; NB = 200; DROP25 = True
tob = lambda s: s.astype(str).str.lower().eq("true")
TOP4 = ["トヨタ自動車", "三菱電機", "日産自動車", "ホンダ"]

PF = pd.read_csv(f"{OUT}/collab_work_features_v3.csv")
for c in ["first_firm", "last_firm", "for_univ", "dom_univ"]: PF[c] = tob(PF[c])
TM = pd.read_csv(f"{OUT}/oa_topic_map.csv", dtype={"topic_id": str})[["topic_id", "subfield"]]
PF = PF.drop(columns=[c for c in ["subfield"] if c in PF.columns]).merge(TM, on="topic_id", how="left")
PF["blk"] = PF["blk"].astype(int)
YMIN = int(PF["year"].min()); YEND = int(PF["year"].max()); BL = sorted(PF["blk"].unique()); b0, b1 = BL[0], BL[-1]
LAB = {b: f"{YMIN + b*BLK}-{YMIN + b*BLK + BLK - 1}" for b in BL}
U = PF.drop_duplicates(["work_id", "corp_norm"]).copy()
U["solo"] = (U["pcls"] == "단독").astype(int)
YLAST = YEND - 1 if DROP25 else YEND           # 2025 색인 미완 -> 추세 모형에서 제외(기본)

def safe(f):
    try: f()
    except Exception as e:
        print(f"\n[{f.__name__} 실패] {type(e).__name__}: {str(e)[:200]}"); traceback.print_exc(limit=2)

def kink_table(ann, key, off, ycol, min_total=100, label=""):
    """연도별 count를 Poisson(offset=ln 분모)으로 추정: ln(비율) = a + b*t + c*max(0, year-2015). c<0이면 2015 이후 가속 하락."""
    rows = []
    for k, g in ann.groupby(key):
        g = g[(g["year"] <= YLAST) & (g[off] > 0)].copy()
        if g[ycol].sum() < min_total or g["year"].nunique() < 15: continue
        g["t"] = g["year"] - 2015; g["post"] = np.maximum(0, g["year"] - 2015)
        try:
            m = smf.glm(f"{ycol} ~ t + post", data=g, family=sm.families.Poisson(), offset=np.log(g[off])).fit(cov_type="HC0")
        except Exception: continue
        rows.append(dict(**{label: k}, n=int(g[ycol].sum()), pre_slope=np.exp(m.params["t"]) - 1,
                         post_slope=np.exp(m.params["t"] + m.params["post"]) - 1, slope_change_pp=100 * (np.exp(m.params["t"] + m.params["post"]) - np.exp(m.params["t"])),
                         p_change=m.pvalues["post"]))
    R = pd.DataFrame(rows)
    if len(R): R["p_holm"] = multipletests(R["p_change"], method="holm")[1]
    return R

# =====================================================================================
# Q1. 어떤 기업의 과학이 더 과거를 따라가는가
# =====================================================================================
def Q1():
    rng = np.random.default_rng(SEED)
    topics = sorted(U["topic_id"].unique()); ix = {t: i for i, t in enumerate(topics)}
    def vec(ts):
        v = np.zeros(len(topics))
        for t in ts: v[ix[t]] += 1
        return v
    def cosv(a, b):
        d = np.linalg.norm(a) * np.linalg.norm(b); return a @ b / d if d > 0 else np.nan
    def half(ts):
        r = []
        for _ in range(100):
            p = rng.permutation(len(ts)); h = len(ts) // 2
            r.append(cosv(vec(ts[p[:h]]), vec(ts[p[h:]])))
        return np.nanmean(r)
    rows = []
    for f, g in U.groupby("corp_norm"):
        t0 = g.loc[g["blk"] == b0, "topic_id"].values; t1 = g.loc[g["blk"] == b1, "topic_id"].values
        if len(t0) < MINP or len(t1) < MINP: continue
        a, z = vec(t0), vec(t1)
        rows.append(dict(corp_norm=f, n0=len(t0), n1=len(t1), cos=cosv(a, z), ceiling=(half(t0) + half(t1)) / 2, persist=z[a > 0].sum() / z.sum()))
    F = pd.DataFrame(rows).set_index("corp_norm")
    F["inertia"] = F["cos"] / F["ceiling"]       # 블록 간 주제 유사도 / 같은 블록 내부 반분 유사도(표본 크기 효과 보정)
    base = U[U["blk"] <= 1]
    ch = base.groupby("corp_norm").agg(solo=("solo", "mean"), fu=("for_univ", "mean"), led=("first_firm", "mean"), fwci=("fwci", "median"), n_base=("work_id", "size"))
    fm = pd.read_csv("firm_master_window.csv").drop_duplicates("corp_norm").set_index("corp_norm")[["industry", "n_win"]]
    F = F.join(ch).join(fm); F["ln_pat"] = np.log(F["n_win"]); F["top4"] = F.index.isin(TOP4).astype(int)
    print(f"[Q1-1] 기업 {len(F)}곳 | 관성지수 = cos(첫블록, 마지막블록) / 블록 내 반분 상한. 1에 가까울수록 과거와 같은 주제 구성")
    print(F[["cos", "ceiling", "inertia", "persist"]].describe().round(3).T.to_string())
    print("\n[Q1-2] 관성 높은 8 / 낮은 8"); cols = ["n0", "n1", "inertia", "persist", "industry"]
    print(F.sort_values("inertia", ascending=False).head(8)[cols].round(3).to_string()); print(F.sort_values("inertia").head(8)[cols].round(3).to_string())
    print("\n[Q1-3] 기업 특성과 관성의 Spearman (기업 수 적어 기술통계, p 해석 주의)")
    rr = []
    for c, nm in [("solo", "첫 두 블록 단독 비중"), ("fu", "해외대학 협력 비중"), ("led", "기업 제1저자 비중"), ("fwci", "FWCI 중앙값"), ("ln_pat", "특허 규모(ln)"), ("top4", "상위4개사"), ("n0", "첫 블록 논문 수")]:
        d = F[[c, "inertia"]].dropna(); r = d.corr(method="spearman").iloc[0, 1]
        from scipy.stats import spearmanr; rr.append(dict(특성=nm, n=len(d), rho=r, p=spearmanr(d[c], d["inertia"])[1]))
    print(pd.DataFrame(rr).round(3).to_string(index=False))
    print("\n[Q1-4] 업종별 평균(2곳 이상)"); g = F.groupby("industry")["inertia"].agg(["mean", "size"]); print(g[g["size"] >= 2].sort_values("mean", ascending=False).round(3).to_string())
    # 기업x분야 패널: 과거 논문 탄력성이 기업 특성에 따라 달라지는가 (PPML, 기업·분야 FE)
    D = pd.read_csv("panel_new_design.csv"); X = base.groupby("corp_norm").agg(solo=("solo", "mean"), fu=("for_univ", "mean"), led=("first_firm", "mean"))
    D = D.merge(X, left_on="corp_norm", right_index=True, how="left").merge(fm[["n_win"]], left_on="corp_norm", right_index=True, how="left")
    D["ln_nwin"] = np.log(D["n_win"]); D["top4"] = D["corp_norm"].isin(TOP4).astype(float)
    print("\n[Q1-5] PPML: n_out ~ ln_pat + ln_pre + ln_pre:x (x 평균중심화, 기업·분야 FE, 기업 군집). 계수<0이면 x가 클수록 과거 의존이 약함")
    out = []
    for x in ["solo", "fu", "led", "ln_nwin", "top4"]:
        d = D.dropna(subset=[x]).copy(); d[x + "_c"] = d[x] - d[x].mean(); d["ix"] = d["ln_pre"] * d[x + "_c"]
        d = d[d.groupby("corp_norm")["n_out"].transform("sum") > 0]
        mod = smf.glm("n_out ~ ln_pat + ln_pre + ix + C(corp_norm) + C(domain)", data=d, family=sm.families.Poisson(), missing="drop")
        m = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.loc[mod.data.row_labels, "corp_norm"])[0]}, maxiter=300)
        out.append(dict(x=x, ln_pre=m.params["ln_pre"], 상호작용=m.params["ix"], se=m.bse["ix"], p=m.pvalues["ix"], 기업수=d["corp_norm"].nunique()))
    O = pd.DataFrame(out); O["p_holm"] = multipletests(O["p"], method="holm")[1]; print(O.round(3).to_string(index=False))
safe(Q1)

# =====================================================================================
# Q2. 단독 연구 소멸이 어느 세부 분야에서 가속되는가
# =====================================================================================
def Q2():
    for key, label, mt in [("domain", "분야", 100), ("subfield", "소분야(OpenAlex)", 150)]:
        ann = U.groupby([key, "year"]).agg(tot=("solo", "size"), solo=("solo", "sum")).reset_index()
        R = kink_table(ann, key, "tot", "solo", min_total=mt, label=label)
        sh = U.groupby([key, "blk"])["solo"].mean().unstack().reindex(columns=BL)
        sh.columns = [f"단독비중 {LAB[b]}" for b in BL]; n0 = U[U["blk"] == b0].groupby(key).size().rename("n_첫블록"); n1 = U[U["blk"] == b1].groupby(key).size().rename("n_마지막")
        R = R.merge(sh, left_on=label, right_index=True).merge(n0, left_on=label, right_index=True, how="left").merge(n1, left_on=label, right_index=True, how="left")
        R["단독비중 변화(%p)"] = 100 * (R[f"단독비중 {LAB[b1]}"] - R[f"단독비중 {LAB[b0]}"])
        if key == "subfield": R = R[(R["n_첫블록"] >= 30) & (R["n_마지막"] >= 30)]
        print(f"\n[Q2-{label}] 단독 논문 '비중'(offset=그 해 전체 논문)의 연 변화율: pre_slope(~2015), post_slope(2016~), 기울기 변화(pp, 음수=2015 이후 가속 하락)")
        print(R.sort_values("slope_change_pp").round(3).to_string(index=False))
    # 기업 구성 통제: 같은 기업 안에서도 가속하는가 (논문 단위 LPM, 기업 FE)
    d0 = U[U["year"] <= YLAST].copy(); d0["t"] = d0["year"] - 2015; d0["post"] = np.maximum(0, d0["year"] - 2015); rows = []
    for dm, g in d0.groupby("domain"):
        mod = smf.ols("solo ~ t + post + C(corp_norm)", data=g, missing="drop"); m = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(g.loc[mod.data.row_labels, "corp_norm"])[0]})
        rows.append(dict(분야=dm, 기업수=g["corp_norm"].nunique(), pre_연pp=100 * m.params["t"], 기울기변화_pp=100 * m.params["post"], p=m.pvalues["post"]))
    R2 = pd.DataFrame(rows); R2["p_holm"] = multipletests(R2["p"], method="holm")[1]
    print("\n[Q2-기업FE] 분야별 단독 확률의 연 변화(%p/년)와 2015 이후 기울기 변화 — 기업 구성 제거")
    print(R2.sort_values("기울기변화_pp").round(3).to_string(index=False))
safe(Q2)

# =====================================================================================
# Q3. 해외 대학 외부 주도 협력은 어느 국가에서 가속되는가
# =====================================================================================
def Q3():
    L = pd.read_csv(f"{OUT}/collab_links_geo.csv")
    L = L[(L["itype"] == "education") & (L["country"] != "JP") & L["country"].notna()].drop_duplicates(["work_id", "corp_norm", "country"]).copy()
    L = L.merge(PF[["work_id", "corp_norm", "first_firm", "last_firm"]], on=["work_id", "corp_norm"], how="left")
    L["ext"] = (~L["last_firm"].fillna(False)).astype(int); L["firm"] = 1 - L["ext"]
    L["blk"] = ((L["year"] - YMIN) // BLK).astype(int)
    tot_year = U.groupby("year").size().rename("tot")
    allp = L.drop_duplicates(["work_id", "corp_norm"])
    print(f"[Q3-0] 해외대학 협력 논문(고유) {len(allp):,}편, (논문x국가) 행 {len(L):,} — 국가별 표는 논문이 여러 국가에 중복 계산됨")
    per = L.groupby(["country", "blk"]).agg(ext=("ext", "sum"), firm=("firm", "sum")).unstack(fill_value=0).reindex(columns=BL, level=1, fill_value=0)
    top = L.groupby("country")["ext"].sum().sort_values(ascending=False); top = top[top >= 50].index[:15]
    rows = []
    for c in top:
        e0, e1 = per.loc[c, ("ext", b0)] / BLK, per.loc[c, ("ext", b1)] / BLK; f0, f1 = per.loc[c, ("firm", b0)] / BLK, per.loc[c, ("firm", b1)] / BLK
        rows.append(dict(국가=c, ext_0=e0, ext_1=e1, 증감=e1 - e0, firm_0=f0, firm_1=f1, 외부주도비중_0=e0 / (e0 + f0) if e0 + f0 else np.nan, 외부주도비중_1=e1 / (e1 + f1) if e1 + f1 else np.nan))
    T = pd.DataFrame(rows); inc = (per["ext"][b1] - per["ext"][b0]).sum() / BLK
    T["총증가 대비 기여"] = T["증감"] / inc
    ann = L.groupby(["country", "year"])["ext"].sum().reset_index(); ann = ann[ann["country"].isin(top)].merge(tot_year, left_on="year", right_index=True)
    grid = pd.MultiIndex.from_product([top, range(YMIN, YEND + 1)], names=["country", "year"]).to_frame(index=False).merge(ann, how="left", on=["country", "year"]).fillna({"ext": 0})
    grid["tot"] = grid["year"].map(tot_year); K = kink_table(grid, "country", "tot", "ext", min_total=30, label="국가")
    T = T.merge(K.rename(columns={"국가": "국가"}), on="국가", how="left")
    print(f"\n[Q3-1] 외부주도(마지막저자 비기업) 해외대학 협력, 연평균 편수. 총 증가량(국가 합) {inc:+.1f}편/년")
    print("  pre_slope/post_slope = 전체 논문 대비 비율의 연 변화율, slope_change_pp<0이면 2015 이후 둔화, >0이면 가속")
    print(T.sort_values("증감", ascending=False).round(3).to_string(index=False))
    allc = L.groupby("year")["ext"].sum(); print("\n[Q3-2] 연도별 외부주도 해외대학 협력(국가 중복 포함):", allc.to_dict())
safe(Q3)

# =====================================================================================
# Q4. 새 파트너 <-> 새 주제 진입은 어느 분야에서 더 강한가
# =====================================================================================
def Q4():
    Q = PF[PF["blk"] > b0].copy(); Q["ln_n"] = np.log(Q.groupby(["corp_norm", "blk"])["work_id"].transform("size"))
    for c in ["nw_any", "entry"]: Q[c] = Q[c].astype(int)
    def one(d):
        X = pd.concat([d[["nw_any", "ln_n"]], pd.get_dummies(d["blk"], prefix="b", drop_first=True)], axis=1).astype(float)
        X = X.loc[:, X.std() > 0]; X = sm.add_constant(X)
        m = sm.Logit(d["entry"].astype(float).values, X.values).fit(disp=0, maxiter=200, cov_type="cluster", cov_kwds={"groups": pd.factorize(d["corp_norm"])[0]})
        i = list(X.columns).index("nw_any"); return m.params[i], m.bse[i], m.pvalues[i]
    def table(key, min_n, min_ev):
        rows = []
        for k, d in Q.groupby(key):
            ev = int(d["entry"].sum()); w1 = d[d["nw_any"] == 1]; w0 = d[d["nw_any"] == 0]
            if len(d) < min_n or ev < min_ev or len(w1) < 20: continue
            try: b, se, p = one(d)
            except Exception: continue
            rows.append(dict(**{key: k}, 논문=len(d), 진입=ev, 진입률_신규=w1["entry"].mean(), 진입률_비신규=w0["entry"].mean(), 차이_pp=100 * (w1["entry"].mean() - w0["entry"].mean()),
                             OR=np.exp(b), OR_lo=np.exp(b - 1.96 * se), OR_hi=np.exp(b + 1.96 * se), p=p))
        R = pd.DataFrame(rows)
        if len(R): R["p_holm"] = multipletests(R["p"], method="holm")[1]
        return R
    R = table("domain", 100, 20); print("[Q4-1] 분야별 신규 파트너 -> 진입 (블록·기업규모 통제 로짓, 기업 군집)"); print(R.sort_values("OR", ascending=False).round(3).to_string(index=False))
    # 이질성 공동검정: 분야 x 신규 파트너 상호작용
    doms = sorted(Q["domain"].unique()); ref = Q["domain"].value_counts().index[0]; oth = [d for d in doms if d != ref]
    X = pd.concat([Q[["nw_any", "ln_n"]], pd.get_dummies(Q["blk"], prefix="b", drop_first=True), pd.get_dummies(Q["domain"], prefix="d").drop(columns=f"d_{ref}")], axis=1).astype(float)
    for d in oth: X[f"nwx_{d}"] = X["nw_any"] * X[f"d_{d}"]
    X = sm.add_constant(X); cols = list(X.columns)
    m = sm.Logit(Q["entry"].astype(float).values, X.values).fit(disp=0, maxiter=300, cov_type="cluster", cov_kwds={"groups": pd.factorize(Q["corp_norm"])[0]})
    Rm = np.zeros((len(oth), len(cols)))
    for j, d in enumerate(oth): Rm[j, cols.index(f"nwx_{d}")] = 1
    w = m.wald_test(Rm, scalar=True); print(f"\n[Q4-2] 분야 간 이질성 공동 Wald 검정(기준 분야 {ref}): p = {float(w.pvalue):.3f}  <- 크면 '분야마다 다르다'는 주장 불가")
    # 기업 FE LPM
    rows = []
    for dm, d in Q.groupby("domain"):
        if len(d) < 100 or d["entry"].sum() < 20: continue
        mod = smf.ols("entry ~ nw_any + ln_n + C(blk) + C(corp_norm)", data=d, missing="drop"); mm = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.loc[mod.data.row_labels, "corp_norm"])[0]})
        rows.append(dict(분야=dm, 기업FE_효과_pp=100 * mm.params["nw_any"], p=mm.pvalues["nw_any"]))
    print("\n[Q4-3] 같은 기업 안에서의 효과(LPM, 기업 FE)"); print(pd.DataFrame(rows).round(3).to_string(index=False))
    R2 = table("subfield", 300, 30); print("\n[Q4-4] 소분야별(논문 300편·진입 30건 이상)"); print(R2.sort_values("OR", ascending=False).round(3).to_string(index=False))
safe(Q4)

# =====================================================================================
# Q5. 지리 효과 분해: 구성 효과 vs 순수 거리 효과
# =====================================================================================
def Q5():
    P = pd.read_csv(f"{OUT}/collab_links_geo.csv")
    nrm = lambda s: s if pd.isna(s) else "".join(ch for ch in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(ch))
    P["p_region"] = P["p_region"].map(nrm).fillna("미상"); P["blk"] = ((P["year"] - YMIN) // BLK).astype(int)
    P["ln_d_hq"] = np.log1p(P["d_hq"]); P["ln_d_min"] = np.log1p(P["d_min"])
    dy = P.sort_values("d_min").drop_duplicates(["corp_norm", "inst_id", "blk"]).copy()      # 기업-기관-블록 1행
    Dm = dy[dy["country"] == "JP"].copy(); Fo = dy[dy["country"] != "JP"].copy()
    Dm["it_reg"] = Dm["itype"] + "|" + Dm["p_region"]; Fo["ctry_it"] = Fo["country"] + "|" + Fo["itype"]
    def kit(df, cell, y, a=b0, z=b1):
        d0, d1 = df[df["blk"] == a], df[df["blk"] == z]
        s0 = d0[cell].value_counts(normalize=True); s1 = d1[cell].value_counts(normalize=True); cells = s0.index.union(s1.index)
        s0, s1 = s0.reindex(cells, fill_value=0), s1.reindex(cells, fill_value=0)
        m0, m1 = d0.groupby(cell)[y].mean().reindex(cells), d1.groupby(cell)[y].mean().reindex(cells)
        m0 = m0.fillna(m1); m1 = m1.fillna(m0)
        return d1[y].mean() - d0[y].mean(), ((s1 - s0) * (m0 + m1) / 2).sum(), (((s0 + s1) / 2) * (m1 - m0)).sum()
    def boot(df, cell, y):
        rng = np.random.default_rng(SEED); g = {k: v.index.values for k, v in df.groupby("corp_norm")}; fs = list(g); r = []
        for _ in range(NB):
            d = df.loc[np.concatenate([g[fs[i]] for i in rng.choice(len(fs), len(fs))])]
            try: r.append(kit(d, cell, y))
            except Exception: pass
        r = np.array(r); return np.percentile(r, [2.5, 97.5], axis=0)
    def run(df, cells, y, title):
        rows = []
        for c in cells:
            dlt, cp, wi = kit(df, c, y); ci = boot(df, c, y)
            rows.append(dict(분해기준=c, 총변화=dlt, 구성효과=cp, 구성_lo=ci[0, 1], 구성_hi=ci[1, 1], 셀내거리변화=wi, 내_lo=ci[0, 2], 내_hi=ci[1, 2]))
        print(f"\n{title}  (ln(1+km) 평균의 {LAB[b0]} -> {LAB[b1]} 변화 = 구성효과 + 셀 내 변화. 95% 구간은 기업 군집 부트스트랩)")
        print(pd.DataFrame(rows).round(3).to_string(index=False))
    for y, nm in [("ln_d_hq", "본사 기준 거리(주 지표)"), ("ln_d_min", "거점/본사 혼합 거리(민감도)")]:
        run(Dm, ["itype", "p_region", "it_reg", "corp_norm"], y, f"[Q5-A] 국내 파트너 | {nm}")
    run(Fo, ["country", "ctry_it", "corp_norm"], "ln_d_hq", "[Q5-B] 해외 파트너 | 본사 기준 거리")
    # 구성 효과의 내용: 파트너 유형 이동이 거리에 준 기여 (유형별 점유율 변화 x 유형별 평균거리)
    s = Dm.groupby(["itype", "blk"]).size().unstack(fill_value=0).reindex(columns=[b0, b1], fill_value=0); s = s / s.sum()
    mm = Dm.groupby("itype")["ln_d_hq"].mean(); T = pd.DataFrame({"점유_0": s[b0], "점유_1": s[b1], "평균ln거리": mm}); T["기여"] = (T["점유_1"] - T["점유_0"]) * (T["평균ln거리"] - Dm["ln_d_hq"].mean())
    print("\n[Q5-C] 국내: 유형별 점유율 변화와 거리 기여(평균 대비)"); print(T.sort_values("기여").round(3).to_string())
    # 회귀 기반 순차 통제: 블록 계수가 어느 통제에서 사라지는가
    print(f"\n[Q5-D] 국내 ln(1+km), 마지막 블록 계수(기준=첫 블록): 통제를 하나씩 추가")
    for nm, f in [("원자료", "ln_d_hq ~ C(blk)"), ("+파트너 유형", "ln_d_hq ~ C(blk) + C(itype)"), ("+지역", "ln_d_hq ~ C(blk) + C(itype) + C(p_region)"),
                  ("+기업 FE", "ln_d_hq ~ C(blk) + C(itype) + C(p_region) + C(corp_norm)")]:
        mod = smf.ols(f, data=Dm, missing="drop"); m = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(Dm.loc[mod.data.row_labels, "corp_norm"])[0]})
        k = f"C(blk)[T.{b1}]"; print(f"  {nm:<10} coef={m.params[k]:+.3f}  se={m.bse[k]:.3f}  p={m.pvalues[k]:.3f}  N={int(m.nobs)}")
    # 파트너 유형 이동이 지리와 무관한가: 유형별 거리 분포
    print("\n[Q5-E] 유형별 국내 거리(km) 중앙값 x 블록"); print(Dm.pivot_table(index="itype", columns="blk", values="d_hq", aggfunc="median").rename(columns=LAB).round(0).to_string())
safe(Q5)

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf

# (a) Q1: 특허 규모 x 과거 논문 상호작용이 상위 4개사 제외 후에도 남는가
D = pd.read_csv("panel_new_design.csv")
fm = pd.read_csv("firm_master_window.csv").drop_duplicates("corp_norm").set_index("corp_norm")[["n_win"]]
D = D.merge(fm, left_on="corp_norm", right_index=True, how="left"); D["ln_nwin"] = np.log(D["n_win"])
def ppml_int(d, label):
    d = d.copy(); d["x"] = d["ln_nwin"] - d["ln_nwin"].mean(); d["ix"] = d["ln_pre"] * d["x"]
    d = d[d.groupby("corp_norm")["n_out"].transform("sum") > 0]
    mod = smf.glm("n_out ~ ln_pat + ln_pre + ix + C(corp_norm) + C(domain)", data=d, family=sm.families.Poisson(), missing="drop")
    m = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.loc[mod.data.row_labels, "corp_norm"])[0]}, maxiter=300)
    print(f"{label}: 상호작용 {m.params['ix']:+.3f} (se {m.bse['ix']:.3f}, p={m.pvalues['ix']:.3f}) | 기업 {d['corp_norm'].nunique()}곳")
ppml_int(D, "전체"); ppml_int(D[~D["corp_norm"].isin(TOP4)], "상위4 제외")
ppml_int(D[D["n_win"] >= 30], "특허 30건 이상만")

# (b) Q3: 꼬임점 위치 민감도 (고유 논문, 마지막 저자 기준, 해외대학 외부주도 전체)
Lf = pd.read_csv("oa_out/collab_links_geo.csv")
Lf = Lf[(Lf["itype"] == "education") & (Lf["country"] != "JP")].drop_duplicates(["work_id", "corp_norm"])
Lf = Lf.merge(PF[["work_id", "corp_norm", "last_firm"]], on=["work_id", "corp_norm"], how="left")
Lf["ext"] = (~Lf["last_firm"].fillna(False)).astype(int)
ann = Lf.groupby("year")["ext"].sum().to_frame("y").join(U.groupby("year").size().rename("tot")).loc[2006:2024]
for K in (2013, 2015, 2018, 2019):
    g = ann.reset_index(); g["t"] = g["year"] - K; g["post"] = np.maximum(0, g["year"] - K)
    m = smf.glm("y ~ t + post", data=g, family=sm.families.Poisson(), offset=np.log(g["tot"])).fit(cov_type="HC0")
    print(f"꼬임점 {K}: pre {np.exp(m.params['t'])-1:+.3f}, 변화 {np.exp(m.params['t']+m.params['post'])-np.exp(m.params['t']):+.3f}, p={m.pvalues['post']:.3f}")

# (c) Q4: 비GX 주제 제외 후 분야별 효과 (BAD 목록은 이전 A2 출력 기준, 직접 수정)
BAD = ["Enzyme Catalysis and Immobilization", "Lipid Membrane Structure and Behavior",
       "Photopolymerization techniques and applications", "nanoparticles nucleation surface interactions", "Solar Radiation and Photovoltaics"]
Q = PF[(PF["blk"] > PF["blk"].min()) & ~PF["topic"].isin(BAD)].copy()
Q["ln_n"] = np.log(Q.groupby(["corp_norm", "blk"])["work_id"].transform("size"))
for dm, d in Q.groupby("domain"):
    if d["entry"].sum() < 20: continue
    X = sm.add_constant(pd.concat([d[["nw_any", "ln_n"]], pd.get_dummies(d["blk"], prefix="b", drop_first=True)], axis=1).astype(float))
    m = sm.Logit(d["entry"].astype(float).values, X.values).fit(disp=0, maxiter=200, cov_type="cluster", cov_kwds={"groups": pd.factorize(d["corp_norm"])[0]})
    print(f"{dm:<9} OR={np.exp(m.params[1]):.2f}  p={m.pvalues[1]:.3f}  진입 {int(d['entry'].sum())}")

In [ ]:
res = []
for f in D["corp_norm"].unique():
    d = D[D["corp_norm"] != f].copy(); d["x"] = d["ln_nwin"] - d["ln_nwin"].mean(); d["ix"] = d["ln_pre"] * d["x"]
    d = d[d.groupby("corp_norm")["n_out"].transform("sum") > 0]
    try:
        mod = smf.glm("n_out ~ ln_pat + ln_pre + ix + C(corp_norm) + C(domain)", data=d, family=sm.families.Poisson(), missing="drop")
        m = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.loc[mod.data.row_labels, "corp_norm"])[0]}, maxiter=300)
        res.append(dict(제외기업=f, 계수=m.params["ix"], p=m.pvalues["ix"]))
    except Exception: pass
R = pd.DataFrame(res).sort_values("계수", ascending=False)
print(R.head(8).round(3).to_string(index=False)); print(R.tail(4).round(3).to_string(index=False))
print("계수 범위:", R["계수"].min().round(3), "~", R["계수"].max().round(3), "| p<0.05 비율:", (R["p"] < 0.05).mean().round(2))

In [ ]:
#

In [ ]:
t = D[D["corp_norm"] == "トヨタ自動車"][["domain", "n_pre", "n_out", "n_pat_new"]].copy()
t["out/pre(연평균 비)"] = (t["n_out"] / 10) / (t["n_pre"] / 5).replace(0, np.nan)
print(t.round(2).to_string(index=False))
# 도요타 칸 내부 탄력성 vs 나머지 기업 (기업 FE 없이 칸 단위 로그-로그)
for lab, d in [("도요타", D[D["corp_norm"] == "トヨタ自動車"]), ("그 외 상위4", D[D["corp_norm"].isin(TOP4[1:])]), ("나머지", D[~D["corp_norm"].isin(TOP4)])]:
    d = d[(d["n_pre"] > 0) & (d["n_out"] > 0)]
    b = np.polyfit(np.log(d["n_pre"]), np.log(d["n_out"]), 1)[0]
    print(f"{lab}: 칸 {len(d)}개, 로그-로그 기울기 {b:.2f}")

In [ ]:
import pandas as pd
T = pd.read_csv("oa_out/topic_gx_tagging.csv")
DRAFT = {
 "core": ["Advancements in Battery Materials", "Fuel Cells and Related Materials", "Advanced Battery Materials and Technologies",
          "Electrocatalysts for Energy Conversion", "Advancements in Solid Oxide Fuel Cells", "Hydrogen Storage and Materials",
          "Electric and Hybrid Vehicle Technologies", "TiO2 Photocatalysis and Solar Cells", "Advanced Battery Technologies Research",
          "Organic Electronics and Photovoltaics", "Advanced battery technologies research", "Silicon and Solar Cell Technologies"],
 "legacy": ["Advanced Combustion Engine Technologies"],
 "enabling": ["Silicon Carbide Semiconductor Technologies", "GaN-based semiconductor devices and materials", "Building Energy and Comfort Optimization",
              "Electric Motor Design and Analysis", "High voltage insulation and dielectric phenomena", "Power System Optimization and Stability",
              "Advanced DC-DC Converters", "Advanced Thermoelectric Materials and Devices", "Refrigeration and Air Conditioning Technologies",
              "Microgrid Control and Optimization", "Wireless Power Transfer Systems"],
 "non_gx": ["Polymer crystallization and properties", "Graphene research and applications", "Mechanical Behavior of Composites",
            "Carbon Nanotubes in Composites", "Vehicle Noise and Vibration Control", "Fiber-reinforced polymer composites",
            "Polymer Science and Applications", "Advanced ceramic materials synthesis", "Advanced Cellulose Research Studies"],
 "other": ["Catalytic Processes in Materials Science", "Vehicle Dynamics and Control Systems", "Membrane Separation Technologies",
           "Mesoporous Materials and Catalysis", "Advanced Photocatalysis Techniques", "Heat Transfer and Boiling Studies",
           "Organic Light-Emitting Diodes Research"]}
m = {t: k for k, v in DRAFT.items() for t in v}
T["gx_class"] = T["topic"].map(m).fillna(T["gx_class"])
print("분류된 주제:", T["topic"].isin(m).sum(), "/ 매핑 목록", len(m), "| 미분류:", T["gx_class"].isna().sum() + (T["gx_class"] == "").sum())
print(T[T["gx_class"].isna() | (T["gx_class"] == "")].sort_values("n_total", ascending=False).head(40)[["topic", "domain", "n_total"]].to_string(index=False))
T.to_csv("oa_out/topic_gx_tagging.csv", index=False, encoding="utf-8-sig")

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import os, re, json, time, warnings, traceback, unicodedata
import numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf
from scipy import stats
from concurrent.futures import ThreadPoolExecutor
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 150)

# ============================ 설정 ============================
OUT = "oa_out"; BLK = 5; MINP = 10; SEED = 0; NB = 200; NB_BRK = 199; DROP25 = True
FETCH_AUTH = True     # [신규] 저자ID·교신저자 수집 (50편/호출, 캐시됨)
FETCH_HIST = True     # [신규] 외부 연구자의 과거 주제 이력 수집 (최대 MAX_HIST_Q회, 캐시됨)
MAX_HIST_Q = 1500
TOP4 = ["トヨタ自動車", "三菱電機", "日産自動車", "ホンダ"]
API_KEY = os.environ.get("OPENALEX_API_KEY"); MAILTO = os.environ.get("OPENALEX_MAILTO")   # 키는 환경변수로만
os.makedirs(f"{OUT}/extra", exist_ok=True)

# ============================ 공통 함수 ============================
WID = lambda s: str(s).split("/")[-1]
def tobin(s):
    if s.dtype == bool: return s.astype(int)
    if pd.api.types.is_numeric_dtype(s): return (s.fillna(0) > 0).astype(int)
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "1.0"]).astype(int)
def safe(f):
    try: f()
    except Exception as e:
        print(f"\n[{f.__name__} 실패] {type(e).__name__}: {str(e)[:250]}"); traceback.print_exc(limit=2)
def save(df, name): df.to_csv(f"{OUT}/extra/{name}.csv", index=False, encoding="utf-8-sig")
def cl_ols(formula, d, key="corp_norm"):
    d = d.reset_index(drop=True); mod = smf.ols(formula, data=d, missing="drop")
    return mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.loc[mod.data.row_labels, key])[0]})
def oa_get(path, params, retries=4):
    import requests
    p = dict(params)
    if API_KEY: p["api_key"] = API_KEY
    if MAILTO: p["mailto"] = MAILTO
    for k in range(retries):
        try:
            r = requests.get("https://api.openalex.org/" + path, params=p, timeout=60)
            if r.status_code == 200: return r.json()
            if r.status_code in (429, 500, 502, 503): time.sleep(2 ** k); continue
            return None
        except Exception: time.sleep(2 ** k)
    return None

# ============================ 데이터 준비 ============================
PF = pd.read_csv(f"{OUT}/collab_work_features_v3.csv")
for c in ["first_firm", "last_firm", "for_univ", "dom_univ"]: PF[c] = tobin(PF[c]).astype(bool)
for c in ["nw_any", "entry"]: PF[c] = tobin(PF[c])
PF["topic_id"] = PF["topic_id"].astype(str)
TM = pd.read_csv(f"{OUT}/oa_topic_map.csv", dtype={"topic_id": str})[["topic_id", "subfield"]]
PF = PF.drop(columns=[c for c in ["subfield"] if c in PF.columns]).merge(TM, on="topic_id", how="left")
PF["blk"] = PF["blk"].astype(int); PF["w"] = PF["work_id"].map(WID)
YMIN = int(PF["year"].min()); YEND = int(PF["year"].max()); BL = sorted(PF["blk"].unique()); b0, b1 = BL[0], BL[-1]
LAB = {b: f"{YMIN + b*BLK}-{YMIN + b*BLK + BLK - 1}" for b in BL}
YLAST = YEND - 1 if DROP25 else YEND
U = PF.drop_duplicates(["work_id", "corp_norm"]).copy(); U["solo"] = (U["pcls"] == "단독").astype(int)

LK = pd.read_csv(f"{OUT}/collab_links_geo.csv")
LK["w"] = LK["work_id"].map(WID); LK["iid"] = LK["inst_id"].astype(str).map(WID)
PART = LK.groupby(["w", "corp_norm"])["iid"].agg(set).to_dict()
LF = LK[(LK["itype"] == "education") & (LK["country"] != "JP") & LK["country"].notna()].drop_duplicates(["w", "corp_norm", "country", "iid"]).copy()
LF = LF.merge(U[["w", "corp_norm", "first_firm", "last_firm"]], on=["w", "corp_norm"], how="left")
LF["ext"] = (~LF["last_firm"].fillna(False).astype(bool)).astype(int); LF["blk"] = ((LF["year"] - YMIN) // BLK).astype(int)
LFw = LF.drop_duplicates(["w", "corp_norm"]).copy()

nrm = lambda s: s if pd.isna(s) else "".join(ch for ch in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(ch))
P = LK.copy(); P["p_region"] = P["p_region"].map(nrm).fillna("미상"); P["blk"] = ((P["year"] - YMIN) // BLK).astype(int)
P["ln_d_hq"] = np.log1p(P["d_hq"]); P["ln_d_min"] = np.log1p(P["d_min"])
dy = P.sort_values("d_min").drop_duplicates(["corp_norm", "inst_id", "blk"]).copy()
Dm = dy[dy["country"] == "JP"].copy(); Fo = dy[dy["country"] != "JP"].copy()
Dm["it_reg"] = Dm["itype"] + "|" + Dm["p_region"]; Fo["ctry_it"] = Fo["country"] + "|" + Fo["itype"]
print(f"[준비] 논문(기업별) {len(U):,} | 블록 {[LAB[b] for b in BL]} | 기업 {U['corp_norm'].nunique()} | 추세 모형 마지막 연도 {YLAST}")

# ============================ 저자 정보 로드 [신규] ============================
AUTH = {}
def load_auth():
    path = f"{OUT}/authorships_cache.jsonl"
    short = lambda v: (str(v).split("/")[-1] if v else None)          # None·빈 문자열 모두 안전하게 처리
    if os.path.exists(path):
        for line in open(path, encoding="utf-8"):
            try: r = json.loads(line); AUTH[r["w"]] = r["a"]
            except Exception: pass
    need = [w for w in U["w"].unique() if w not in AUTH]
    if FETCH_AUTH and need:
        fails = 0; bad = 0
        with open(path, "a", encoding="utf-8") as fh:
            for k in range(0, len(need), 50):
                chunk = need[k:k + 50]
                js = oa_get("works", {"filter": "openalex:" + "|".join(chunk), "select": "id,authorships", "per-page": 50})
                if js is None:
                    fails += 1
                    if fails >= 3 and not AUTH: print("[저자 수집] 연속 실패 -> 중단(네트워크/키 확인)"); return
                    continue
                for r in js.get("results", []):
                    try:
                        w = short(r.get("id"))
                        if not w: continue
                        a = []
                        for x in (r.get("authorships") or []):
                            insts = x.get("institutions") or []
                            a.append([short((x.get("author") or {}).get("id")), x.get("author_position"), bool(x.get("is_corresponding")),
                                      [i.get("type") for i in insts if i.get("type")], [short(i.get("id")) for i in insts if i.get("id")],
                                      x.get("countries") or []])
                        AUTH[w] = a; fh.write(json.dumps({"w": w, "a": a}) + "\n")
                    except Exception:
                        bad += 1
                if (k // 50) % 20 == 0: print(f"  저자 수집 {k + len(chunk)}/{len(need)}")
        if bad: print(f"  [경고] 파싱 실패 논문 {bad}편 (건너뜀)")
    print(f"[저자 정보] 확보 {sum(1 for v in AUTH.values() if v):,}편 / 전체 {U['w'].nunique():,}편")

# =====================================================================
# Q1-1. 편수는 유지하는데 주제는 바꾸는 기업: 편수 변화 vs 주제 재배분
# =====================================================================
def Q1a():
    rng = np.random.default_rng(SEED)
    def half_turn(ts, R=50):
        ts = np.asarray(ts); o = []
        for _ in range(R):
            p = rng.permutation(len(ts)); h = len(ts) // 2
            a = pd.Series(ts[p[:h]]).value_counts(normalize=True); b = pd.Series(ts[p[h:]]).value_counts(normalize=True); ix = a.index.union(b.index)
            o.append(1 - np.minimum(a.reindex(ix, fill_value=0), b.reindex(ix, fill_value=0)).sum())
        return np.mean(o)
    rows = []
    for f, g in U.groupby("corp_norm"):
        t0 = g.loc[g.blk == b0, "topic_id"].values; t1 = g.loc[g.blk == b1, "topic_id"].values
        if len(t0) < MINP or len(t1) < MINP: continue
        c0 = pd.Series(t0).value_counts(); c1 = pd.Series(t1).value_counts(); ts = c0.index.union(c1.index)
        a = c0.reindex(ts, fill_value=0); z = c1.reindex(ts, fill_value=0); both = (a > 0) & (z > 0); n0, n1 = a.sum(), z.sum()
        turn = 1 - np.minimum(a / n0, z / n1).sum(); base = (half_turn(t0) + half_turn(t1)) / 2
        rows.append(dict(corp_norm=f, n0=n0, n1=n1, ln_ratio=np.log(n1 / n0), 유지주제증감=(z - a)[both].sum() / n0, 신규주제=z[a == 0].sum() / n0,
                         철수주제=-a[z == 0].sum() / n0, turn=turn, turn_adj=turn - base))
    F = pd.DataFrame(rows)
    F["유형"] = np.select([(F.ln_ratio >= F.ln_ratio.median()) & (F.turn_adj >= F.turn_adj.median()), (F.ln_ratio >= F.ln_ratio.median()), (F.turn_adj >= F.turn_adj.median())],
                         ["편수↑·주제 재배분↑", "편수↑·주제 유지", "편수↓·주제 재배분↑"], "편수↓·주제 유지")
    print(f"[Q1-1] 기업 {len(F)}곳 | 편수 증감률 = 유지주제증감 + 신규주제 + 철수주제 (첫 블록 대비). turn_adj = 주제 교체도에서 표본잡음 기준선 차감")
    print(F.sort_values("ln_ratio")[["corp_norm", "n0", "n1", "유지주제증감", "신규주제", "철수주제", "turn", "turn_adj", "유형"]].round(3).to_string(index=False))
    print("\n유형별 기업 수:", F["유형"].value_counts().to_dict())
    r, p = stats.spearmanr(F["ln_ratio"], F["turn_adj"]); print(f"Spearman(편수 증감, 주제 교체도) = {r:+.3f} (p={p:.3f})")
    big = F[(F.n0 >= 30) & (F.n1 >= 30)]
    if len(big) >= 8: r2, p2 = stats.spearmanr(big["ln_ratio"], big["turn_adj"]); print(f"n>=30 기업만({len(big)}곳): {r2:+.3f} (p={p2:.3f})")
    save(F, "Q1_1_volume_vs_topic_turnover")
safe(Q1a)

# =====================================================================
# Q1-2. 특허 -> 논문 경로 (특허의 비특허문헌 인용 데이터 필요)
# =====================================================================
def Q1b():
    fn = "patent_npl_links.csv"
    if not os.path.exists(fn):
        print("[Q1-2] 건너뜀: 특허가 인용한 논문 수 파일 필요 -> patent_npl_links.csv (열: corp_norm, domain, n_npl_cited). 이 데이터는 OpenAlex에 없어 PATSTAT/Lens 등이 필요")
        return
    D = pd.read_csv("panel_new_design.csv").merge(pd.read_csv(fn), on=["corp_norm", "domain"], how="left").fillna({"n_npl_cited": 0})
    D["cited"] = (D["n_npl_cited"] > 0).astype(float); D["ix"] = D["ln_pat"] * D["cited"]
    D = D[D.groupby("corp_norm")["n_out"].transform("sum") > 0]
    mod = smf.glm("n_out ~ ln_pre + ln_pat + ix + C(corp_norm) + C(domain)", data=D, family=sm.families.Poisson(), missing="drop")
    m = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(D.loc[mod.data.row_labels, "corp_norm"])[0]}, maxiter=300)
    t = m.t_test("ln_pat + ix = 0")
    print(f"[Q1-2] 특허 효과: 논문 미인용 칸 {m.params['ln_pat']:+.3f} (p={m.pvalues['ln_pat']:.3f}) | 인용 칸 {float(t.effect[0]):+.3f} (p={float(t.pvalue):.3f}) | 차이 {m.params['ix']:+.3f} (p={m.pvalues['ix']:.3f})")
safe(Q1b)

# =====================================================================
# Q1-3. 관성 계수 감쇠 보정 (반분 도구변수)
# =====================================================================
def Q1c():
    D = pd.read_csv("panel_new_design.csv")[["corp_norm", "domain", "n_out"]]; base = U[U.blk == b0]; rng = np.random.default_rng(SEED)
    def split():
        b = base.copy(); b["h"] = rng.integers(0, 2, len(b))
        H = b.groupby(["corp_norm", "domain", "h"]).size().unstack("h", fill_value=0).reindex(columns=[0, 1], fill_value=0); H.columns = ["nA", "nB"]
        return D.merge(H.reset_index(), on=["corp_norm", "domain"], how="left").fillna({"nA": 0, "nB": 0})
    def est(X):
        Z = np.column_stack([pd.get_dummies(X["corp_norm"]).values.astype(float), pd.get_dummies(X["domain"], drop_first=True).values.astype(float)])
        res = lambda v: v - Z @ np.linalg.lstsq(Z, v, rcond=None)[0]
        y = res(np.log1p(X.n_out.values.astype(float))); w = res(np.log1p(2 * X.nB.values)); z = res(np.log1p(2 * X.nA.values))
        return (w @ y) / (w @ w), (z @ y) / (z @ w), np.corrcoef(z, w)[0, 1]
    print("[Q1-3] ln(1+이후 논문) ~ ln(1+첫 블록 논문) (기업·분야 FE). 반분 A를 도구변수로 쓴 IV가 측정오차 보정값. PPML 계수와 정의가 달라 직접 비교하지 말 것")
    for lab, flt in [("전체", lambda X: X), ("상위4 제외", lambda X: X[~X.corp_norm.isin(TOP4)])]:
        pts = np.array([est(flt(split())) for _ in range(30)]); nv, iv, r = pts.mean(axis=0)
        X0 = flt(split()); g = {k: v for k, v in X0.groupby("corp_norm")}; fs = list(g); bs = []
        for _ in range(NB):
            pick = rng.choice(len(fs), len(fs))
            try: bs.append(est(pd.concat([g[fs[i]].assign(corp_norm=str(j)) for j, i in enumerate(pick)]))[1])
            except Exception: pass
        lo, hi = np.percentile(bs, [2.5, 97.5])
        print(f"  {lab:<8} naive {nv:.3f} | IV(보정) {iv:.3f} [{lo:.3f}, {hi:.3f}] | 반분 상관 {r:.3f} (Spearman-Brown 신뢰도 {2*r/(1+r):.3f}) | 감쇠 배율 {iv/nv:.2f}")
safe(Q1c)

# =====================================================================
# Q2-1. 단독 연구 저자는 사라졌나, 공저로 전환했나 [신규: 저자ID]
# =====================================================================
def Q2a():
    if not AUTH: print("[Q2-1] 건너뜀: 저자 정보 없음"); return
    rec = []
    for r in U[["w", "corp_norm", "blk", "solo"]].itertuples(index=False):
        cl = classify(r.w, r.corp_norm)
        if cl: rec += [(r.corp_norm, c[0], r.blk, r.solo) for c in cl if c[4] and c[0]]
    R = pd.DataFrame(rec, columns=["corp", "aid", "blk", "solo"]).drop_duplicates()
    if R.empty: print("[Q2-1] 기업 저자 식별 실패"); return
    S = {b: set() for b in BL}; Cc = {b: set() for b in BL}
    for corp, aid, b, s in R.itertuples(index=False): (S if s == 1 else Cc)[b].add((corp, aid))
    nblk = U.groupby(["corp_norm", "blk"]).size().to_dict(); ok = lambda f, t: nblk.get((f, t), 0) >= MINP and nblk.get((f, t + 1), 0) >= MINP
    rows = []
    for t in BL[:-1]:
        A1 = S[t + 1] | Cc[t + 1]
        for lab, coh in [("단독만", {x for x in S[t] - Cc[t] if ok(x[0], t)}), ("협력만", {x for x in Cc[t] - S[t] if ok(x[0], t)})]:
            n = len(coh)
            if n == 0: continue
            rows.append(dict(블록=LAB[t], 코호트=lab, n=n, 단독유지=len(coh & S[t + 1]) / n, 협력전환=len((coh & Cc[t + 1]) - S[t + 1]) / n, 미관측=len(coh - A1) / n, gone=len(coh - A1)))
    T = pd.DataFrame(rows); print("[Q2-1] 다음 블록에서의 기업 저자 행방 (같은 기업, 두 블록 모두 논문 10편 이상인 기업만). 미관측 = 해당 기업의 GX 논문 코퍼스에 다시 등장하지 않음(이직·주제 이탈 포함)")
    print(T.drop(columns="gone").round(3).to_string(index=False))
    g = T.groupby("코호트").agg(n=("n", "sum"), gone=("gone", "sum")); g["미관측율"] = g["gone"] / g["n"]; print("\n풀링:"); print(g.round(3).to_string())
    if len(g) == 2:
        chi = stats.chi2_contingency([[g.loc[k, "gone"], g.loc[k, "n"] - g.loc[k, "gone"]] for k in g.index])
        print(f"단독만 vs 협력만 미관측율 차이 chi2 p={chi[1]:.4f} (저자 간 군집 미반영, 서술용)")
    save(T, "Q2_1_author_transition")
safe(Q2a)

# =====================================================================
# Q2-2. 단독 vs 협력 논문의 FWCI·주제 신규성 (같은 기업·분야 안)
# =====================================================================
def Q2b():
    d = U[U.year <= YLAST].copy(); d["lf"] = np.log1p(d["fwci"]); d = d.dropna(subset=["lf"])
    m = cl_ols("lf ~ solo + C(corp_norm) + C(domain) + C(year)", d)
    print(f"[Q2-2a] ln(1+FWCI) ~ 단독 | 단독 계수 {m.params['solo']:+.3f} (se {m.bse['solo']:.3f}, p={m.pvalues['solo']:.3f}) N={int(m.nobs)}")
    rows = []
    for dm, g in d.groupby("domain"):
        if len(g) < 150 or g.solo.sum() < 30: continue
        try: mm = cl_ols("lf ~ solo + C(corp_norm) + C(year)", g); rows.append(dict(분야=dm, N=int(mm.nobs), 단독계수=mm.params["solo"], p=mm.pvalues["solo"]))
        except Exception: pass
    R = pd.DataFrame(rows); R["p_holm"] = multipletests(R["p"], method="holm")[1]; print(R.round(3).to_string(index=False))
    d2 = d[d.blk > b0].copy(); d2["ln_n"] = np.log(d2.groupby(["corp_norm", "blk"])["w"].transform("size"))
    m2 = cl_ols("entry ~ solo + ln_n + C(corp_norm) + C(blk) + C(domain)", d2)
    print(f"\n[Q2-2b] 새 주제 진입(LPM, 기업·블록·분야 FE) ~ 단독 | 계수 {100*m2.params['solo']:+.2f}pp (p={m2.pvalues['solo']:.3f}) | 단독 진입률 {100*d2[d2.solo==1].entry.mean():.1f}% vs 협력 {100*d2[d2.solo==0].entry.mean():.1f}%")
    save(R, "Q2_2_fwci_by_domain")
safe(Q2b)

# =====================================================================
# Q2-3 / Q6-3. 단독 비중 변화의 기업별 shift-share (분야별)
# =====================================================================
SS = {}
def shift_share(g):
    a = g[g.blk == b0]; z = g[g.blk == b1]; n0, n1 = len(a), len(z)
    if n0 < 20 or n1 < 20: return None
    w0 = a.groupby("corp_norm").size() / n0; w1 = z.groupby("corp_norm").size() / n1; f = w0.index.union(w1.index)
    w0, w1 = w0.reindex(f, fill_value=0), w1.reindex(f, fill_value=0)
    s0 = a.groupby("corp_norm")["solo"].mean().reindex(f); s1 = z.groupby("corp_norm")["solo"].mean().reindex(f); s0 = s0.fillna(s1); s1 = s1.fillna(s0)
    T = pd.DataFrame({"w0": w0, "w1": w1, "s0": s0, "s1": s1}); T["기업내"] = (T.w0 + T.w1) / 2 * (T.s1 - T.s0); T["기업간"] = (T.w1 - T.w0) * (T.s0 + T.s1) / 2
    T["기여"] = T["기업내"] + T["기업간"]; T["유형"] = np.where(T.w0 == 0, "신규", np.where(T.w1 == 0, "철수", "계속")); return T
def Q2c():
    rows = []
    for dm, g in [("ALL", U)] + list(U.groupby("domain")):
        T = shift_share(g)
        if T is None: continue
        SS[dm] = T; ab = T["기업내"].abs().sort_values(ascending=False); k50 = int((ab.cumsum() / ab.sum() < 0.5).sum() + 1)
        rows.append(dict(분야=dm, 총변화pp=100 * T["기여"].sum(), 기업내pp=100 * T["기업내"].sum(), 기업간pp=100 * T["기업간"].sum(), 기업내_50퍼센트_설명_기업수=k50, 기업수=len(T)))
    R = pd.DataFrame(rows); print("[Q2-3] 단독 비중 변화(첫 블록 -> 마지막) = 기업 내 변화 + 기업 간 점유 이동. 양수=단독 비중 상승"); print(R.round(2).to_string(index=False)); save(R, "Q2_3_shift_share_summary")
    if "VEHICLE" in SS:
        T = SS["VEHICLE"].copy(); T["기여pp"] = 100 * T["기여"]; print("\n[Q2-3] VEHICLE 기여 상·하위 기업"); print(pd.concat([T.sort_values("기여").head(5), T.sort_values("기여").tail(3)])[["w0", "w1", "s0", "s1", "기업내", "기업간", "기여pp", "유형"]].round(3).to_string())
safe(Q2c)

# =====================================================================
# Q3-1. 마지막 저자 vs 교신저자 지표 [신규: 교신저자]
# =====================================================================
def Q3a():
    if not AUTH: print("[Q3-1] 건너뜀: 저자 정보 없음"); return
    rows = []
    for r in LFw.itertuples(index=False):
        cl = classify(r.w, r.corp_norm)
        if not cl: continue
        flag = any(c[2] for c in cl); ce = any(c[2] and c[3] for c in cl); cf = any(c[2] and c[4] for c in cl)
        rows.append((r.w, r.corp_norm, r.blk, flag, ce and not cf, cf, int(r.ext), int(not bool(r.first_firm))))
    C_ = pd.DataFrame(rows, columns=["w", "corp", "blk", "flag", "corr_ext", "corr_firm", "last_ext", "first_ext"])
    cov = C_["flag"].mean() if len(C_) else 0; print(f"[Q3-1] 해외대학 협력 논문 {len(LFw):,}편 중 저자정보 {len(C_):,}편, 교신저자 표기 {cov:.1%}")
    if cov < 0.3: print("  경고: 교신저자 표기율이 낮아 결과 신뢰도 제한")
    S_ = C_[C_.flag].copy(); S_["corr_ext"] = S_["corr_ext"].astype(int)
    T = S_.groupby("blk").agg(n=("w", "size"), 교신_외부=("corr_ext", "mean"), 마지막_외부=("last_ext", "mean"), 제1_외부=("first_ext", "mean")); T.index = [LAB[b] for b in T.index]
    print(T.round(3).to_string()); print("\n교신저자 외부 vs 마지막저자 외부 일치표:"); print(pd.crosstab(S_["corr_ext"], S_["last_ext"]))
    for col, nm in [("corr_ext", "교신저자"), ("last_ext", "마지막저자"), ("first_ext", "제1저자")]:
        a = S_[S_.blk == b0][col].sum() / BLK; b = S_[S_.blk == b1][col].sum() / BLK; a_all = len(S_[S_.blk == b0]) / BLK; b_all = len(S_[S_.blk == b1]) / BLK
        print(f"  {nm} 기준 외부주도 편수/년: {a:.1f} -> {b:.1f} | 전체 증가 {b_all-a_all:+.1f} 중 외부주도 증가 비중 {((b-a)/(b_all-a_all)):.0%}")
    save(T.reset_index(), "Q3_1_corresponding_author")
safe(Q3a)

# =====================================================================
# Q3-2. 해외 대학 협력 증가의 집중도 (중국 등)
# =====================================================================
def Q3b():
    hhi = lambda s: float(((s / s.sum()) ** 2).sum()) if s.sum() > 0 else np.nan
    L = LF[LF.ext == 1].drop_duplicates(["w", "corp_norm", "iid"]); namec = next((c for c in ["inst_name", "display_name", "inst", "name"] if c in L.columns), None)
    for ctry in ["CN", "US", "DE", "ALL"]:
        g = L if ctry == "ALL" else L[L.country == ctry]; rows = []
        for key, lab in [("corp_norm", "기업"), ("iid", "기관")]:
            c0 = g[g.blk == b0].groupby(key).size(); c1 = g[g.blk == b1].groupby(key).size(); rank = list(c1.sort_values(ascending=False).index)
            for k in [0, 1, 3, 5]:
                drop = set(rank[:k]); a = c0[~c0.index.isin(drop)].sum() / BLK; b = c1[~c1.index.isin(drop)].sum() / BLK
                rows.append(dict(축=lab, 상위제거=k, b0_편수년=a, b1_편수년=b, 증가배수=b / a if a > 0 else np.nan))
            rows.append(dict(축=lab + " HHI", 상위제거=0, b0_편수년=hhi(c0), b1_편수년=hhi(c1), 증가배수=np.nan))
        print(f"\n[Q3-2] {ctry}: 외부주도 해외대학 협력, 상위 기여 기업/기관을 제거해도 증가가 남는가"); print(pd.DataFrame(rows).round(3).to_string(index=False))
        if ctry == "CN":
            top = g[g.blk == b1].groupby("iid").size().sort_values(ascending=False).head(5)
            nm = g.drop_duplicates("iid").set_index("iid")[namec] if namec else None
            print("  마지막 블록 상위 5 기관:", {(nm.get(i, i) if nm is not None else i): int(v) for i, v in top.items()})
            print("  마지막 블록 상위 5 기업:", g[g.blk == b1].groupby("corp_norm").size().sort_values(ascending=False).head(5).to_dict())
safe(Q3b)

# =====================================================================
# Q3-3. 외부 주도 협력에서 기업 연구자의 저자 위치
# =====================================================================
def Q3c():
    d = U[U.for_univ].copy()
    d["위치"] = np.select([d.first_firm & d.last_firm, d.first_firm & ~d.last_firm, ~d.first_firm & d.last_firm], ["기업 제1+마지막", "기업 제1(외부 마지막)", "기업 마지막(외부 제1)"], "기업 중간저자만")
    T = d.groupby(["위치", "blk"]).size().unstack(fill_value=0).reindex(columns=[b0, b1], fill_value=0) / BLK; T.columns = [f"편수/년 {LAB[b0]}", f"편수/년 {LAB[b1]}"]
    T["증감"] = T.iloc[:, 1] - T.iloc[:, 0]; T["증가 기여"] = T["증감"] / T["증감"].sum(); S_ = d.groupby(["위치", "blk"]).size().unstack(fill_value=0).reindex(columns=[b0, b1], fill_value=0); S_ = S_ / S_.sum()
    T[f"비중 {LAB[b0]}"] = S_[b0]; T[f"비중 {LAB[b1]}"] = S_[b1]; print("[Q3-3] 해외대학 포함 논문의 기업 저자 위치"); print(T.round(3).to_string())
    print("  중간저자만 비중이 늘면 기업이 시료·데이터 제공 등 보조 역할로 이동했을 가능성(저자 기여 분류는 아님, 서술용)")
safe(Q3c)

# =====================================================================
# Q4-1. 파트너가 먼저인가 [신규: 외부 연구자의 과거 주제 이력]
# =====================================================================
def Q4a():
    if not (AUTH and FETCH_HIST): print("[Q4-1] 건너뜀: 저자 정보 없음 또는 FETCH_HIST=False"); return
    tid = lambda t: str(t) if re.match(r"^T\d+$", str(t)) else ("T" + str(t) if re.match(r"^\d+$", str(t)) else None)
    E = U[(U.blk > b0) & (U.nw_any == 1)]; ev = E[E.entry == 1]; ct = E[E.entry == 0]; samp = []
    for b, g in ev.groupby("blk"):
        pool = ct[ct.blk == b]; k = min(len(g), len(pool))
        if k: samp += [g.sample(k, random_state=SEED).assign(grp=1), pool.sample(k, random_state=SEED).assign(grp=0)]
    if not samp: print("[Q4-1] 표본 없음"); return
    S_ = pd.concat(samp).sample(frac=1, random_state=SEED); hp = f"{OUT}/author_topic_hist.json"
    H = json.load(open(hp)) if os.path.exists(hp) else {}; items = []; need = set()
    for r in S_.itertuples(index=False):
        t = tid(r.topic_id); cl = classify(r.w, r.corp_norm)
        if t is None or not cl: continue
        ex = [c[0] for c in cl if c[3] and c[0]][:5]
        if not ex: continue
        keys = [f"{a}|{t}|{int(r.year)}" for a in ex]; new = {k for k in keys if k not in H}
        if len(need | new) > MAX_HIST_Q: break
        need |= new; items.append((r.w, r.corp_norm, r.blk, r.grp, keys))
    print(f"[Q4-1] 표본 논문 {len(items)}편, 신규 조회 {len(need)}건 (캐시 {len(H)}건)")
    def job(k):
        a, t, y = k.split("|"); time.sleep(0.05)
        js = oa_get("works", {"filter": f"authorships.author.id:{a},topics.id:{t},to_publication_date:{int(y)-1}-12-31", "per-page": 1, "select": "id"})
        return k, (None if js is None else js["meta"]["count"])
    with ThreadPoolExecutor(max_workers=4) as ex_:
        for k, v in ex_.map(job, list(need)):
            if v is not None: H[k] = v
    json.dump(H, open(hp, "w"))
    rows = []
    for w, corp, b, grp, keys in items:
        v = [H[k] for k in keys if k in H]
        if v: rows.append(dict(w=w, corp_norm=corp, blk=b, entry=grp, maxprior=max(v), any1=int(max(v) >= 1), ge3=int(max(v) >= 3)))
    R = pd.DataFrame(rows)
    if R.empty: print("  유효 조회 없음(주제 ID 형식/네트워크 확인)"); return
    T = R.groupby("entry").agg(논문=("w", "size"), 전문가_파트너_비중=("any1", "mean"), 전문가_3편이상=("ge3", "mean"), 평균_ln1p=("maxprior", lambda s: np.log1p(s).mean())); T.index = ["대조(기존 주제)", "진입"]
    print(T.round(3).to_string())
    m = cl_ols("any1 ~ entry + C(blk)", R); print(f"LPM(블록 FE): 진입 - 대조 = {100*m.params['entry']:+.1f}pp (se {100*m.bse['entry']:.1f}, p={m.pvalues['entry']:.3f})")
    print("  해석: 양(+)이면 외부 파트너가 그 주제의 기존 전문가로 합류(파트너 선행) 쪽, 0 근처면 선행성 증거 없음. 저자ID 오귀속·OpenAlex 주제 태깅 노이즈 한계")
    save(R, "Q4_1_partner_prior_expertise")
safe(Q4a)

# =====================================================================
# Q4-2. 진입의 정착: 신규 파트너 동반 진입이 더 오래가는가
# =====================================================================
def Q4b():
    A = PF.groupby(["corp_norm", "topic_id"]).agg(fb=("blk", "min")).reset_index()
    pres = PF.groupby(["corp_norm", "topic_id"])["blk"].apply(lambda s: frozenset(s)).rename("bs").reset_index(); A = A.merge(pres, on=["corp_norm", "topic_id"])
    nw = PF.groupby(["corp_norm", "topic_id", "blk"]).agg(nw=("nw_any", "max"), n_first=("w", "nunique")).reset_index().rename(columns={"blk": "fb"}); A = A.merge(nw, on=["corp_norm", "topic_id", "fb"])
    chk = PF[PF.blk > b0].merge(A[["corp_norm", "topic_id", "fb"]], on=["corp_norm", "topic_id"]); print("[Q4-2] 정의 점검: PF의 entry 플래그 vs 첫 등장 블록(fb==blk) 일치표"); print(pd.crosstab(chk["entry"], chk["blk"] == chk["fb"]))
    E = A[(A.fb > b0) & (A.fb < b1)].copy(); E["next"] = [int((f + 1) in s) for f, s in zip(E.fb, E.bs)]; E["later"] = [int(any(b > f for b in s)) for f, s in zip(E.fb, E.bs)]; E["ln_n"] = np.log(E.n_first)
    print(f"\n진입 사건 {len(E)}건 | 신규 파트너 동반 {int(E.nw.sum())}건"); print(E.groupby("nw")[["next", "later"]].mean().round(3).rename(index={0: "파트너 없음", 1: "신규 파트너 동반"}))
    rows = []
    for lab, d in [("전체", E), ("상위4 제외", E[~E.corp_norm.isin(TOP4)])]:
        for y in ["next", "later"]:
            try: m = cl_ols(f"{y} ~ nw + ln_n + C(fb) + C(corp_norm)", d); rows.append(dict(표본=lab, 결과=y, 효과pp=100 * m.params["nw"], se=100 * m.bse["nw"], p=m.pvalues["nw"], N=int(m.nobs)))
            except Exception: pass
    print("\nLPM(기업·블록 FE): 신규 파트너 동반 진입의 정착 확률 차이"); print(pd.DataFrame(rows).round(3).to_string(index=False))
safe(Q4b)

# =====================================================================
# Q4-3. 진입 기회 집합을 분모로 한 분야 이질성 (이산시간 hazard)
# =====================================================================
def Q4c():
    fbd = PF.groupby(["corp_norm", "domain", "topic_id"]).agg(fb=("blk", "min")).reset_index(); dt = PF.groupby("domain")["topic_id"].nunique()
    cnt = fbd.groupby(["corp_norm", "domain", "fb"]).size().unstack("fb", fill_value=0).reindex(columns=BL, fill_value=0); cum = cnt.cumsum(axis=1).shift(1, axis=1).fillna(0)
    cell = U.groupby(["corp_norm", "domain", "blk"]).agg(n=("w", "size"), nw=("nw_any", "mean")).reset_index(); cell = cell[cell.blk > b0]
    ev = cnt.stack().rename("ev").reset_index().rename(columns={"fb": "blk"}); pr = cum.stack().rename("prior").reset_index().rename(columns={"fb": "blk"})
    cell = cell.merge(ev, on=["corp_norm", "domain", "blk"], how="left").merge(pr, on=["corp_norm", "domain", "blk"], how="left").fillna({"ev": 0, "prior": 0})
    cell["risk"] = cell["domain"].map(dt) - cell["prior"]; cell = cell[cell.risk > 0].copy(); cell["ev"] = np.minimum(cell["ev"], cell["risk"]); cell["prop"] = cell["ev"] / cell["risk"]; cell["ln_n"] = np.log(cell["n"])
    print(f"[Q4-3] 기업x분야x블록 {len(cell)}칸, 진입 {int(cell.ev.sum())}건. 모형: 진입비율 ~ 신규파트너 논문 비중 (위험집합 가중, 이항). 주의: 같은 논문에서 진입과 파트너가 정의되는 기계적 연관은 남아 있음")
    grp = pd.factorize(cell["corp_norm"])[0]; rows = []
    for dm, d in cell.groupby("domain"):
        if d.ev.sum() < 15: continue
        X = sm.add_constant(pd.concat([d[["nw", "ln_n"]], pd.get_dummies(d["blk"], prefix="b", drop_first=True)], axis=1).astype(float)); X = X.loc[:, X.std().fillna(1) > 0] if False else X
        try:
            m = sm.GLM(d["prop"].values, X.values, family=sm.families.Binomial(), freq_weights=d["risk"].values).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d["corp_norm"])[0]})
            rows.append(dict(분야=dm, 칸=len(d), 진입=int(d.ev.sum()), nw계수=m.params[1], OR_per_100pp=np.exp(m.params[1]), p=m.pvalues[1]))
        except Exception: pass
    R = pd.DataFrame(rows); R["p_holm"] = multipletests(R["p"], method="holm")[1]; print(R.round(3).to_string(index=False))
    doms = sorted(cell.domain.unique()); ref = cell.domain.value_counts().index[0]; oth = [x for x in doms if x != ref]
    X = pd.concat([cell[["nw", "ln_n"]], pd.get_dummies(cell["blk"], prefix="b", drop_first=True), pd.get_dummies(cell["domain"], prefix="d").drop(columns=f"d_{ref}")], axis=1).astype(float)
    for x in oth: X[f"nwx_{x}"] = X["nw"] * X[f"d_{x}"]
    X = sm.add_constant(X); cols = list(X.columns)
    m = sm.GLM(cell["prop"].values, X.values, family=sm.families.Binomial(), freq_weights=cell["risk"].values).fit(cov_type="cluster", cov_kwds={"groups": grp})
    Rm = np.zeros((len(oth), len(cols)))
    for j, x in enumerate(oth): Rm[j, cols.index(f"nwx_{x}")] = 1
    print(f"분야 간 이질성 공동 Wald (기준 {ref}): p = {float(m.wald_test(Rm, scalar=True).pvalue):.3f}  (논문 단위 이전 결과 p=0.040과 비교)")
    save(R, "Q4_3_hazard_by_domain")
safe(Q4c)

# =====================================================================
# Q5-1. 국내 거리 구성효과의 기관 수준 분해
# =====================================================================
def Q5a():
    d0, d1 = Dm[Dm.blk == b0], Dm[Dm.blk == b1]; s0 = d0["inst_id"].value_counts(normalize=True); s1 = d1["inst_id"].value_counts(normalize=True); cells = s0.index.union(s1.index)
    s0, s1 = s0.reindex(cells, fill_value=0), s1.reindex(cells, fill_value=0); m0 = d0.groupby("inst_id")["ln_d_hq"].mean().reindex(cells); m1 = d1.groupby("inst_id")["ln_d_hq"].mean().reindex(cells); m0 = m0.fillna(m1); m1 = m1.fillna(m0)
    M = Dm["ln_d_hq"].mean(); contrib = (s1 - s0) * ((m0 + m1) / 2 - M); namec = next((c for c in ["inst_name", "display_name", "inst", "name"] if c in Dm.columns), None)
    meta = Dm.drop_duplicates("inst_id").set_index("inst_id")[["itype", "p_region"] + ([namec] if namec else [])]; T = meta.reindex(cells).assign(점유0=s0, 점유1=s1, 평균ln거리=(m0 + m1) / 2, 기여=contrib).dropna(subset=["기여"])
    print(f"[Q5-1] 국내 기관 수준 구성효과 합계 {contrib.sum():+.3f} (기관 {len(T)}곳). 기여 = 점유 변화 x (기관 평균거리 - 전체 평균)")
    show = [c for c in [namec, "itype", "p_region", "점유0", "점유1", "평균ln거리", "기여"] if c]; print("\n구성효과를 키운 상위 10 기관"); print(T.sort_values("기여", ascending=False).head(10)[show].round(3).to_string())
    print("\n구성효과를 줄인 상위 5 기관"); print(T.sort_values("기여").head(5)[show].round(3).to_string())
    pos = T[T.기여 > 0]["기여"].sort_values(ascending=False); print(f"\n양(+) 기여 상위 5 기관의 비중: {pos.head(5).sum()/pos.sum():.0%}"); print("\n유형별 기여 합:"); print(T.groupby("itype")["기여"].sum().round(3).to_string())
    save(T.reset_index(), "Q5_1_institution_contrib")
safe(Q5a)

# =====================================================================
# Q5-2. 실제 연구 거점 기준 거리 (거점 좌표 파일 필요)
# =====================================================================
def Q5b():
    fn = "firm_sites.csv"
    if not os.path.exists(fn): print("[Q5-2] 건너뜀: 기업 연구소 좌표 파일 필요 -> firm_sites.csv (열: corp_norm, lat, lon; 기업당 여러 행 가능). 파트너 좌표 열(lat/lon 등)이 collab_links_geo.csv에 있어야 함"); return
    S_ = pd.read_csv(fn); cand = [("p_lat", "p_lon"), ("lat", "lon"), ("inst_lat", "inst_lon"), ("latitude", "longitude")]; cols = next((c for c in cand if c[0] in Dm.columns and c[1] in Dm.columns), None)
    if cols is None: print("[Q5-2] 링크 파일에 파트너 좌표 열이 없음"); return
    def hav(la1, lo1, la2, lo2):
        la1, lo1, la2, lo2 = map(np.radians, [la1, lo1, la2, lo2]); a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2; return 6371 * 2 * np.arcsin(np.sqrt(a))
    sites = {k: v[["lat", "lon"]].values for k, v in S_.groupby("corp_norm")}; d = Dm[Dm.blk.isin([b0, b1])].copy(); out = []
    for r in d.itertuples(index=False):
        s = sites.get(r.corp_norm); la, lo = getattr(r, cols[0]), getattr(r, cols[1])
        out.append(np.nanmin(hav(s[:, 0], s[:, 1], la, lo)) if s is not None and pd.notna(la) else np.nan)
    d["d_site"] = out; d["ln_d_site"] = np.log1p(d["d_site"]); d["post"] = (d.blk == b1).astype(int); d = d.dropna(subset=["ln_d_site"])
    for y in ["ln_d_hq", "ln_d_site"]:
        m = cl_ols(f"{y} ~ post + C(itype)", d); print(f"[Q5-2] {y}: 블록 계수 {m.params['post']:+.3f} (se {m.bse['post']:.3f}, p={m.pvalues['post']:.3f}) N={int(m.nobs)} (같은 표본)")
safe(Q5b)

# =====================================================================
# Q5-3. 셀 내 순수 거리 변화의 등가성 검정 (TOST)
# =====================================================================
def Q5c():
    rows = []
    for lab, df, cell in [("국내", Dm, "it_reg"), ("해외", Fo, "ctry_it")]:
        d = df[df.blk.isin([b0, b1])].copy(); d["post"] = (d.blk == b1).astype(int)
        for spec, f in [("셀 FE", f"ln_d_hq ~ post + C({cell})"), ("셀+기업 FE", f"ln_d_hq ~ post + C({cell}) + C(corp_norm)")]:
            try: m = cl_ols(f, d)
            except Exception: continue
            b, se = m.params["post"], m.bse["post"]; row = dict(구분=lab, 모형=spec, 계수=b, se=se, lo90=b - 1.645 * se, hi90=b + 1.645 * se, N=int(m.nobs))
            for dl in [1.1, 1.25, 1.5]:
                dd = np.log(dl); row[f"TOST_p(±{int(round((dl-1)*100))}%)"] = max(1 - stats.norm.cdf((b + dd) / se), stats.norm.cdf((b - dd) / se))
            rows.append(row)
    R = pd.DataFrame(rows); print("[Q5-3] 셀 내 거리 변화의 등가성 검정: TOST p<0.05이면 '±X% 이상의 변화는 배제됨'. 90% 구간이 ±ln(1+X)에 들어가야 함"); print(R.round(3).to_string(index=False)); save(R, "Q5_3_tost")
safe(Q5c)

# =====================================================================
# Q6-1. 꼬임점을 미지수로 둔 구조변화 (sup-LR, 파라메트릭 부트스트랩)
# =====================================================================
def Q6a():
    rng = np.random.default_rng(SEED); Ks = list(range(YMIN + 4, YLAST - 3)); rows = []
    doms = ["ALL"] + [d for d, g in U.groupby("domain") if len(g) >= 300]
    for name in doms:
        g = U if name == "ALL" else U[U.domain == name]; ann = g.groupby("year").agg(tot=("solo", "size"), solo=("solo", "sum")).reset_index(); ann = ann[(ann.year <= YLAST) & (ann.tot > 0)]
        if ann.solo.sum() < 100 or len(ann) < 15: continue
        y = ann.solo.values.astype(float); yr = ann.year.values; off = np.log(ann.tot.values.astype(float)); t = (yr - 2015).astype(float); X0 = sm.add_constant(t)
        fit = lambda X, yy: sm.GLM(yy, X, family=sm.families.Poisson(), offset=off).fit()
        def prof(yy):
            m0 = fit(X0, yy); ph = max(1.0, m0.pearson_chi2 / m0.df_resid); return {K: (m0.deviance - fit(sm.add_constant(np.column_stack([t, np.maximum(0, yr - K)])), yy).deviance) / ph for K in Ks}, m0, ph
        LR, m0, phi = prof(y); Kh = max(LR, key=LR.get); sup = LR[Kh]; mu = m0.mu; sims = []
        for _ in range(NB_BRK):
            yy = rng.poisson(rng.gamma(mu / (phi - 1), phi - 1)) if phi > 1.0001 else rng.poisson(mu)
            try: sims.append(max(prof(yy.astype(float))[0].values()))
            except Exception: pass
        p = (1 + np.sum(np.array(sims) >= sup)) / (1 + len(sims)); mk = fit(sm.add_constant(np.column_stack([t, np.maximum(0, yr - Kh)])), y)
        ci = [K for K in Ks if LR[K] >= sup - 3.84]
        rows.append(dict(분야=name, 추정_꼬임점=Kh, 구간=f"{min(ci)}-{max(ci)}", sup_LR=sup, p_boot=p, 이전_연변화=np.exp(mk.params[1]) - 1, 이후_연변화=np.exp(mk.params[1] + mk.params[2]) - 1, LR_2015=LR.get(2015, np.nan), 과산포=phi))
    R = pd.DataFrame(rows); R["p_holm"] = multipletests(R["p_boot"], method="holm")[1] if len(R) else []
    print("[Q6-1] 단독 비중(offset=전체 논문)의 꼬임점 탐색. 추정_꼬임점이 2015와 크게 다르면 '2015 전환' 서술 수정. 구간=프로파일 LR 근사 95%"); print(R.round(3).to_string(index=False)); save(R, "Q6_1_unknown_breakpoint")
safe(Q6a)

# =====================================================================
# Q6-2. 단독 논문 감소분은 어느 협력 유형으로 흡수됐나
# =====================================================================
def Q6b():
    print("[Q6-2] 논문 유형(pcls) 분포:", U["pcls"].value_counts().to_dict())
    T = U.groupby(["pcls", "blk"]).size().unstack(fill_value=0).reindex(columns=[b0, b1], fill_value=0) / BLK; T.columns = [f"편수/년 {LAB[b0]}", f"편수/년 {LAB[b1]}"]; T["증감"] = T.iloc[:, 1] - T.iloc[:, 0]; print(T.round(1).to_string())
    F = U.groupby(["corp_norm", "pcls", "blk"]).size().unstack("blk", fill_value=0).reindex(columns=[b0, b1], fill_value=0); F["d"] = (F[b1] - F[b0]) / BLK; D = F["d"].unstack("pcls", fill_value=0)
    if "단독" not in D.columns: print("  '단독' 유형 없음"); return
    dn = D[D["단독"] < 0]; others = [c for c in D.columns if c != "단독"]
    print(f"\n단독 감소 기업 {len(dn)}곳: 단독 감소 합 {dn['단독'].sum():.1f}편/년 | 같은 기업의 다른 유형 증감 합:"); print((dn[others].sum()).round(1).to_string())
    print(f"흡수율(다른 유형 증가 합 / 단독 감소 절대값) = {dn[others].sum().sum() / abs(dn['단독'].sum()):.2f}  (1이면 총 편수 불변, 음수면 총량도 감소)")
    print("\n기업 단위 Spearman(단독 증감, 유형별 증감):", {c: round(float(stats.spearmanr(D["단독"], D[c])[0]), 3) for c in others})
    save(D.reset_index(), "Q6_2_class_shift_by_firm")
safe(Q6b)

# =====================================================================
# Q6-3. THERMAL·FUELCELL 단독 비중이 반대로 움직인 이유 (기업 기여)
# =====================================================================
def Q6c():
    for dm in ["THERMAL", "FUELCELL"]:
        if dm not in SS: continue
        T = SS[dm].copy(); T["기여pp"] = 100 * T["기여"]
        print(f"\n[Q6-3] {dm}: 단독 비중 변화 {100*T['기여'].sum():+.1f}pp = 기업내 {100*T['기업내'].sum():+.1f} + 기업간 {100*T['기업간'].sum():+.1f} | 신규 기업 기여 {T[T.유형=='신규']['기여pp'].sum():+.1f}, 철수 기업 기여 {T[T.유형=='철수']['기여pp'].sum():+.1f}")
        print(T.reindex(T["기여pp"].abs().sort_values(ascending=False).index).head(6)[["w0", "w1", "s0", "s1", "기업내", "기업간", "기여pp", "유형"]].round(3).to_string())
safe(Q6c)
print("\n[완료] 표는 oa_out/extra/*.csv 저장")

In [ ]:
#

In [ ]:
import os, getpass
os.environ["OPENALEX_API_KEY"] = getpass.getpass("OpenAlex key: ")
API_KEY = os.environ["OPENALEX_API_KEY"]      # oa_get()이 이 전역 변수를 읽습니다

# (1) 저자 수집이 왜 비었는지 진단
print("AUTH 크기:", len(AUTH), "| 캐시 파일:", os.path.exists(f"{OUT}/authorships_cache.jsonl"), "| FETCH_AUTH:", FETCH_AUTH, "| API키 설정:", bool(API_KEY))
w0 = U["w"].iloc[0]; print("샘플 ID:", w0)
try:
    import requests
    r = requests.get("https://api.openalex.org/works", params={"filter": f"openalex:{w0}", "select": "id,authorships", "per-page": 1, **({"api_key": API_KEY} if API_KEY else {})}, timeout=30)
    print("HTTP", r.status_code, "|", r.text[:200])
except Exception as e: print("요청 예외:", type(e).__name__, str(e)[:200])

# (2) 해외 거리의 셀 내 변동 (TOST 공허 여부)
print(Fo.groupby("country")["ln_d_hq"].std().describe().round(4).to_string())

# (3) Q4: 칸 단위에서 풀링 신규 파트너 효과가 유지되는가
fbd = PF.groupby(["corp_norm", "domain", "topic_id"]).agg(fb=("blk", "min")).reset_index(); dt = PF.groupby("domain")["topic_id"].nunique()
cnt = fbd.groupby(["corp_norm", "domain", "fb"]).size().unstack("fb", fill_value=0).reindex(columns=BL, fill_value=0); cum = cnt.cumsum(axis=1).shift(1, axis=1).fillna(0)
cell = U.groupby(["corp_norm", "domain", "blk"]).agg(n=("w", "size"), nw=("nw_any", "mean")).reset_index(); cell = cell[cell.blk > b0]
cell = cell.merge(cnt.stack().rename("ev").reset_index().rename(columns={"fb": "blk"}), on=["corp_norm", "domain", "blk"], how="left").merge(cum.stack().rename("prior").reset_index().rename(columns={"fb": "blk"}), on=["corp_norm", "domain", "blk"], how="left").fillna({"ev": 0, "prior": 0})
cell["risk"] = cell["domain"].map(dt) - cell["prior"]; cell = cell[cell.risk > 0].copy(); cell["ev"] = np.minimum(cell["ev"], cell["risk"]); cell["prop"] = cell["ev"] / cell["risk"]; cell["ln_n"] = np.log(cell["n"])
for lab, d in [("전체", cell), ("상위4 제외", cell[~cell.corp_norm.isin(TOP4)])]:
    X = pd.concat([d[["nw", "ln_n"]], pd.get_dummies(d["blk"], prefix="b", drop_first=True), pd.get_dummies(d["domain"], prefix="d", drop_first=True)], axis=1).astype(float); X = sm.add_constant(X)
    m = sm.GLM(d["prop"].values, X.values, family=sm.families.Binomial(), freq_weights=d["risk"].values).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d["corp_norm"])[0]})
    print(f"{lab}: 신규 파트너 100pp당 OR {np.exp(m.params[1]):.2f} (p={m.pvalues[1]:.3f}), 칸 {len(d)}, 진입 {int(d.ev.sum())}")

In [ ]:
import inspect; print("수정본 적용:", "short =" in inspect.getsource(load_auth))
AUTH.clear(); load_auth()
print("샘플:", next(iter(AUTH.items())) if AUTH else "비어 있음")
safe(Q2a); safe(Q3a); safe(Q4a)

In [ ]:
def classify(w, corp):
    a = AUTH.get(w)
    if not a: return None
    pset = PART.get((w, corp), set()); out = []
    for aid, pos, corr, types, iids, ctry in a:
        ext = any(i in pset for i in iids); firm = (not ext) and ("company" in types)
        out.append((aid, pos, corr, ext, firm))
    return out

# 소속 정보 커버리지 (이 분류의 신뢰도 상한)
tot = noaff = noid = 0
for lst in AUTH.values():
    for aid, pos, corr, types, iids, ctry in lst:
        tot += 1; noaff += (not iids); noid += (aid is None)
print(f"저자 {tot:,}명 | 소속 없음 {noaff/tot:.1%} | 저자ID 없음 {noid/tot:.1%}")

safe(Q2a); safe(Q3a); safe(Q4a)

In [ ]:
#

In [ ]:
# (a) 새 단독 저자의 유입이 줄었는가 (첫 블록은 모두 '신규'로 잡히므로 제외하고 읽을 것)
rec = []
for r in U[["w", "corp_norm", "blk", "solo"]].itertuples(index=False):
    cl = classify(r.w, r.corp_norm)
    if cl: rec += [(r.corp_norm, c[0], r.blk, r.solo) for c in cl if c[4] and c[0]]
R = pd.DataFrame(rec, columns=["corp", "aid", "blk", "solo"]).drop_duplicates()
Rs = R.merge(R.groupby(["corp", "aid"])["blk"].min().rename("fb").reset_index(), on=["corp", "aid"]); rows = []
for b in BL:
    act = Rs[Rs.blk == b]; new = act[act.fb == b]; u = lambda d: d[["corp", "aid"]].drop_duplicates().shape[0]
    rows.append(dict(블록=LAB[b], 활동저자=u(act), 신규저자=u(new), 단독참여_저자=u(act[act.solo == 1]), 신규_단독참여=u(new[new.solo == 1])))
T = pd.DataFrame(rows); T["신규비율"] = T["신규저자"] / T["활동저자"]; T["신규중_단독참여"] = T["신규_단독참여"] / T["신규저자"]; print(T.round(3).to_string(index=False))

# (b) Q3-1을 비중 기준으로 기업 FE까지 (표기율 변화 영향 제거)
rows = []
for r in LFw.itertuples(index=False):
    cl = classify(r.w, r.corp_norm)
    if not cl or not any(c[2] for c in cl): continue
    ce = any(c[2] and c[3] for c in cl); cf = any(c[2] and c[4] for c in cl)
    rows.append(dict(corp_norm=r.corp_norm, blk=r.blk, corr=int(ce and not cf), last=int(r.ext), first=int(not bool(r.first_firm))))
C_ = pd.DataFrame(rows); C_ = C_[C_.blk.isin([b0, b1])].copy(); C_["post"] = (C_.blk == b1).astype(int)
for y in ["corr", "last", "first"]:
    m = cl_ols(f"{y} ~ post", C_); mf = cl_ols(f"{y} ~ post + C(corp_norm)", C_)
    print(f"{y:<6} 외부주도 비중 변화: {100*m.params['post']:+.1f}pp (se {100*m.bse['post']:.1f}) | 기업 FE {100*mf.params['post']:+.1f}pp (se {100*mf.bse['post']:.1f}) | N={int(m.nobs)}")

# (c) Q4-1: 천장 효과를 피한 연속 지표와 신뢰구간
R4 = pd.read_csv(f"{OUT}/extra/Q4_1_partner_prior_expertise.csv"); R4["ln1p"] = np.log1p(R4["maxprior"])
for y in ["any1", "ge3", "ln1p"]:
    m = cl_ols(f"{y} ~ entry + C(blk)", R4); b, s = m.params["entry"], m.bse["entry"]
    print(f"{y:<5} 진입-대조 {b:+.3f} (se {s:.3f}, p={m.pvalues['entry']:.3f}) 95%CI [{b-1.96*s:+.3f}, {b+1.96*s:+.3f}]")
print(R4.groupby("entry")["maxprior"].describe().round(1).to_string())

In [ ]:
#

In [ ]:
def inflow(Uu):
    rec = []
    for r in Uu[["w", "corp_norm", "blk", "solo"]].itertuples(index=False):
        cl = classify(r.w, r.corp_norm)
        if cl: rec += [(r.corp_norm, c[0], r.blk, r.solo) for c in cl if c[4] and c[0]]
    R = pd.DataFrame(rec, columns=["corp_norm", "aid", "blk", "solo"]).drop_duplicates()
    fb = R.groupby(["corp_norm", "aid"])["blk"].min().rename("fb").reset_index()
    A = R.merge(fb, on=["corp_norm", "aid"]).groupby(["corp_norm", "aid", "blk", "fb"])["solo"].max().reset_index()
    return A[(A.blk == A.fb) & (A.blk > b0)].copy()          # 첫 블록은 전원 신규라 제외

for lab, Uu in [("2025 포함", U), ("2025 제외", U[U.year <= YLAST])]:
    N = inflow(Uu); g = N.groupby("blk")["solo"].agg(["size", "mean"]); g.index = [LAB[b] for b in g.index]
    print(f"\n[{lab}] 신규 저자 수와 단독 참여 비중"); print(g.round(3).to_string())
    m = cl_ols("solo ~ C(blk) + C(corp_norm)", N); k = f"C(blk)[T.{b1}]"
    print(f"  마지막 블록 vs 2011-15 (기업 FE): {100*m.params[k]:+.1f}pp (se {100*m.bse[k]:.1f}, p={m.pvalues[k]:.3f})")

In [ ]:
#GX 분류표 확정(126개 미분류)과 core 한정 재추정을 기반으로 마무리 분석을 해주세요. 임펙트 있는거를 더 세부적으로 파?? 근데 여기서 앞서 수집한 일본  GX 특허기반으로 수집한 관련 논문이라는 주제에 맞게??? 제시해주세요. 우선 일본이라는거 그리고 GX관련기술 특허를 기반으로 논문분석을 들어간거를 유기적이고 논리가 맞으면서 시너지 있게요.

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import os, re, numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multitest import multipletests

T = pd.read_csv(f"{OUT}/topic_gx_tagging.csv")
DRAFT2 = {
 "core": ["solar cell performance optimization", "Catalysts for Methane Reforming", "CO2 Reduction Techniques and Catalysts", "Electric Vehicles and Infrastructure",
          "Supercapacitor Materials and Fabrication", "Solar Radiation and Photovoltaics", "Wind Turbine Control Systems", "Perovskite Materials and Applications",
          "Photovoltaic System Optimization Techniques", "Geothermal Energy Systems and Applications"],
 "enabling": ["Power Systems and Renewable Energy", "Multilevel Inverters and Converters", "Electric Power System Optimization", "Sensorless Control of Electric Motors",
              "Smart Grid Energy Management", "High-Voltage Power Transmission Systems", "Power Systems Fault Detection", "Adsorption and Cooling Systems"],
 "legacy": ["Engine and Fuel Emissions", "Advanced Thermodynamic Systems and Engines"],
 "other": ["Heat Transfer and Optimization", "Membrane Separation and Gas Transport", "Luminescence Properties of Advanced Materials",
           "biodegradable polymer synthesis and properties", "Quantum Dots Synthesis And Properties", "Conducting polymers and applications"],
 "non_gx": ["Organometallic Complex Synthesis and Catalysis", "Polymer Nanocomposites and Properties", "Thermal properties of materials", "Synthesis and properties of polymers",
            "Boron and Carbon Nanomaterials Research", "Advanced Polymer Synthesis and Characterization", "Fluid Dynamics and Heat Transfer", "Lipid Membrane Structure and Behavior",
            "Polymer composites and self-healing", "Block Copolymer Self-Assembly", "Zeolite Catalysis and Synthesis", "Polymer Surface Interaction Studies",
            "Extraction and Separation Processes", "Cellular and Composite Structures"]}
m2 = {t: k for k, v in DRAFT2.items() for t in v}
T["src"] = np.where(T["gx_class"].notna() & (T["gx_class"] != ""), "claude_draft", "")
hit = T["topic"].isin(m2); T.loc[hit, "gx_class"] = T.loc[hit, "topic"].map(m2); T.loc[hit, "src"] = "claude_draft"

# 나머지(관측하지 못한 주제)는 키워드 규칙으로 '임시' 표기 -> 반드시 검토
RULES = [("legacy", r"combustion|diesel|gasoline|\bengines?\b|fuel injection"),
         ("core", r"battery|batteries|fuel cell|hydrogen|solar|photovolta|\bwind\b|electric vehicle|lithium|perovskite|\bco2\b|carbon dioxide|geothermal|biomass|biofuel|energy storage|supercapacitor|electrocatal|solid oxide"),
         ("enabling", r"power system|\bgrid\b|inverter|converter|\bmotors?\b|semiconductor|thermoelectric|insulation|energy management|heat pump|refrigerat|cooling|building energy|power electronics|transformer|power transmission"),
         ("non_gx", r"polymer|composite|nanotube|graphene|ceramic|lipid|enzyme|protein|cellulose|crystal|alloy|\bsteel\b|coating|\bfiber|laser")]
def rule(t):
    for k, rx in RULES:
        if re.search(rx, str(t), flags=re.I): return k
    return "unreviewed"
miss = T["gx_class"].isna() | (T["gx_class"] == "")
T.loc[miss, "gx_class"] = T.loc[miss, "topic"].map(rule); T.loc[miss, "src"] = "rule_provisional"

# 지도교수·전문가 검토 후 gx_review_final.csv (열: topic, gx_class_final)를 두면 그것이 우선
fin = f"{OUT}/gx_review_final.csv"
if os.path.exists(fin):
    F = pd.read_csv(fin)[["topic", "gx_class_final"]].dropna(); T = T.merge(F, on="topic", how="left"); T.loc[T.gx_class_final.notna(), ["gx_class", "src"]] = [None, "final"]
    T["gx_class"] = np.where(T.gx_class_final.notna(), T.gx_class_final, T.gx_class)
T.to_csv(f"{OUT}/topic_gx_tagging.csv", index=False, encoding="utf-8-sig")
T.sort_values("n_total", ascending=False)[["topic", "domain", "n_total", "gx_class", "src"]].assign(gx_class_final="").to_csv(f"{OUT}/gx_review_sheet.csv", index=False, encoding="utf-8-sig")
CLS = T.set_index("topic")["gx_class"].to_dict()
U["cls"] = U["topic"].map(CLS).fillna("unreviewed"); PF["cls"] = PF["topic"].map(CLS).fillna("unreviewed")
cov = U.groupby("cls").size(); print("[분류 커버리지] 논문 기준(기업별 논문 단위)"); print(pd.DataFrame({"논문": cov, "비중": (cov / cov.sum()).round(3)}).to_string())
print("\n[출처별 주제 수 / 논문 비중]"); print(T.groupby("src").agg(주제=("topic", "size"), n_total=("n_total", "sum")).assign(논문비중=lambda d: d.n_total / d.n_total.sum()).round(3).to_string())
print("\n규칙 임시분류 중 논문 많은 상위 30 (우선 검토)"); print(T[T.src == "rule_provisional"].sort_values("n_total", ascending=False).head(30)[["topic", "domain", "n_total", "gx_class"]].to_string(index=False))

In [ ]:
DEFS = {"S1_core": {"core"}, "S2_core+enabling": {"core", "enabling"}, "S3_excl_nonGX": {"core", "enabling", "legacy", "other", "unreviewed"}}
BLK_ = BLK if "BLK" in globals() else 5
def eff(d):
    """신규 파트너 -> 진입: 논문 단위 로짓 OR + 기업 FE LPM"""
    d = d.copy(); d["ln_n"] = np.log(d.groupby(["corp_norm", "blk"])["w"].transform("size")); ev = int(d.entry.sum())
    if len(d) < 80 or ev < 15 or d.nw_any.sum() < 15: return dict(논문=len(d), 진입=ev, OR=np.nan)
    X = sm.add_constant(pd.concat([d[["nw_any", "ln_n"]], pd.get_dummies(d["blk"], prefix="b", drop_first=True)], axis=1).astype(float))
    m = sm.Logit(d["entry"].astype(float).values, X.values).fit(disp=0, maxiter=200, cov_type="cluster", cov_kwds={"groups": pd.factorize(d["corp_norm"])[0]})
    b, s = m.params[1], m.bse[1]; mf = cl_ols("entry ~ nw_any + ln_n + C(blk) + C(corp_norm)", d)
    return dict(논문=len(d), 진입=ev, OR=np.exp(b), lo=np.exp(b - 1.96 * s), hi=np.exp(b + 1.96 * s), p=m.pvalues[1], FE_pp=100 * mf.params["nw_any"], FE_p=mf.pvalues["nw_any"])

# ---- (A) 구성: 5년 블록별 논문 수/년과 분류 비중, 분야별 core 비중
def A():
    t = U.groupby(["blk", "cls"]).size().unstack(fill_value=0); t.index = [LAB[b] for b in t.index]
    print("[A-1] 분류별 논문 수/년"); print((t / BLK_).round(1).to_string()); print("\n[A-2] 분류 비중"); print(t.div(t.sum(1), axis=0).round(3).to_string())
    d = U[U.cls == "core"].groupby(["domain", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0); a = U.groupby(["domain", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0)
    print("\n[A-3] 분야별 core 비중 (분야 정의와 core 분류가 얼마나 겹치는가)"); print((d / a).rename(columns=LAB).round(3).to_string())
    P = U.groupby(["corp_norm", "blk", "cls"]).size().unstack("cls", fill_value=0); P["tot"] = P.sum(1); P["core_share"] = P.get("core", 0) / P["tot"]
    w = P["core_share"].unstack("blk"); n = P["tot"].unstack("blk"); ok = (n[b0] >= 30) & (n[b1] >= 30); R = pd.DataFrame({"n0": n[b0], "n1": n[b1], "core0": w[b0], "core1": w[b1]})[ok]; R["변화pp"] = 100 * (R.core1 - R.core0)
    print("\n[A-4] 기업별 core 비중 (두 블록 모두 30편 이상)"); print(R.sort_values("변화pp").round(3).to_string())
safe(A)

# ---- (B) 단독 연구 감소가 어느 분류에서 일어났나
def B():
    d = U[U.year <= YLAST].copy(); rows = []
    for k, g in d.groupby("cls"):
        if len(g) < 200: continue
        x = g.groupby(["blk"]).agg(n=("solo", "size"), solo=("solo", "sum")); x = x.reindex(BL, fill_value=0)
        try: m = cl_ols("solo ~ C(blk) + C(corp_norm)", g); co = m.params.get(f"C(blk)[T.{b1}]", np.nan); p = m.pvalues.get(f"C(blk)[T.{b1}]", np.nan)
        except Exception: co, p = np.nan, np.nan
        rows.append(dict(분류=k, 단독편수년_첫=x.solo[b0] / BLK_, 단독편수년_끝=x.solo[b1] / BLK_, 단독비중_첫=x.solo[b0] / max(x.n[b0], 1), 단독비중_끝=x.solo[b1] / max(x.n[b1], 1), 기업FE_변화pp=100 * co, p=p))
    R = pd.DataFrame(rows); R["p_holm"] = multipletests(R["p"].fillna(1), method="holm")[1]; print("[B-1] 분류별 단독 연구 (끝 블록 vs 첫 블록, 기업 FE)"); print(R.round(3).to_string(index=False))
    # 신규 기업 저자 유입 (classify/AUTH 사용): 분류별 신규 저자 중 단독 참여 비중
    rec = []
    for r in U[["w", "corp_norm", "blk", "solo", "cls"]].itertuples(index=False):
        cl = classify(r.w, r.corp_norm)
        if cl: rec += [(r.corp_norm, c[0], r.blk, r.solo, r.cls) for c in cl if c[4] and c[0]]
    Rr = pd.DataFrame(rec, columns=["corp_norm", "aid", "blk", "solo", "cls"])
    fb = Rr.groupby(["corp_norm", "aid"])["blk"].min().rename("fb").reset_index(); Rr = Rr.merge(fb, on=["corp_norm", "aid"])
    new = Rr[(Rr.blk == Rr.fb) & (Rr.blk > b0) & (Rr.blk <= (YLAST - YMIN) // BLK_)]
    for lab, sub in [("core", new[new.cls == "core"]), ("core 외", new[new.cls != "core"])]:
        g = sub.groupby(["corp_norm", "aid", "blk"])["solo"].max().reset_index().groupby("blk")["solo"].agg(["size", "mean"]); g.index = [LAB[b] for b in g.index]
        print(f"\n[B-2] {lab} 논문에서의 신규 기업 저자: 수와 단독 참여 비중"); print(g.round(3).to_string())
safe(B)

# ---- (C) 신규 파트너 -> 진입, 정의별 + 분류별
def Cc():
    Q = U[U.blk > b0].copy(); rows = []
    for nm, S in DEFS.items():
        for lab, d in [("전체", Q[Q.cls.isin(S)]), ("상위4 제외", Q[Q.cls.isin(S) & ~Q.corp_norm.isin(TOP4)])]:
            try: rows.append(dict(정의=nm, 표본=lab, **eff(d)))
            except Exception as e: rows.append(dict(정의=nm, 표본=lab, OR=np.nan))
    print("[C-1] 진입 = 해당 분류 주제로의 진입 (논문 단위)"); print(pd.DataFrame(rows).round(3).to_string(index=False))
    rows = []
    for k in ["core", "enabling", "legacy", "other", "non_gx"]:
        try: rows.append(dict(분류=k, **eff(Q[Q.cls == k])))
        except Exception: pass
    R = pd.DataFrame(rows); print("\n[C-2] 분류별 (GX 핵심으로의 이동에서 신규 파트너가 더 동반되는가)"); print(R.round(3).to_string(index=False))
    d = Q[Q.cls.isin(["core", "enabling", "legacy", "non_gx"])].copy(); d["ln_n"] = np.log(d.groupby(["corp_norm", "blk"])["w"].transform("size")); ref = "non_gx"
    X = pd.concat([d[["nw_any", "ln_n"]], pd.get_dummies(d["blk"], prefix="b", drop_first=True), pd.get_dummies(d["cls"], prefix="c").drop(columns=f"c_{ref}")], axis=1).astype(float)
    for c_ in [k for k in d.cls.unique() if k != ref]: X[f"nwx_{c_}"] = X["nw_any"] * X[f"c_{c_}"]
    X = sm.add_constant(X); cols = list(X.columns); m = sm.Logit(d["entry"].astype(float).values, X.values).fit(disp=0, maxiter=300, cov_type="cluster", cov_kwds={"groups": pd.factorize(d["corp_norm"])[0]})
    oth = [k for k in d.cls.unique() if k != ref]; Rm = np.zeros((len(oth), len(cols)))
    for j, c_ in enumerate(oth): Rm[j, cols.index(f"nwx_{c_}")] = 1
    print(f"[C-3] 분류 간 이질성 공동 Wald(기준 {ref}): p = {float(m.wald_test(Rm, scalar=True).pvalue):.3f}")
safe(Cc)

# ---- (D) 특허-과학 정렬: 같은 분야에서 과거 논문과 특허가 core 논문 산출을 설명하는가
def Dd():
    D0 = pd.read_csv("panel_new_design.csv"); best = None
    for k, bs in {"블록1-3": [1, 2, 3], "블록1-2": [1, 2], "블록2-3": [2, 3]}.items():
        chk = U[U.blk.isin(bs)].groupby(["corp_norm", "domain"]).size().rename("chk").reset_index(); mm = D0.merge(chk, on=["corp_norm", "domain"], how="left").fillna({"chk": 0})
        r = (mm["chk"] == mm["n_out"]).mean(); print(f"[D-0] n_out 창 {k}: 일치율 {r:.3f}")
        if best is None or r > best[0]: best = (r, bs, k)
    if best[0] < 0.8: print("  일치율이 낮아 n_out 정의를 재확인해야 함 -> 중단"); return
    bs = best[1]; print(f"  -> {best[2]} 사용")
    def build(S):
        o = U[U.blk.isin(bs) & U.cls.isin(S)].groupby(["corp_norm", "domain"]).size().rename("n_out_s"); p = U[(U.blk == b0) & U.cls.isin(S)].groupby(["corp_norm", "domain"]).size().rename("n_pre_s")
        d = D0.merge(o.reset_index(), on=["corp_norm", "domain"], how="left").merge(p.reset_index(), on=["corp_norm", "domain"], how="left").fillna({"n_out_s": 0, "n_pre_s": 0}); d["ln_pre_s"] = np.log1p(d["n_pre_s"]); return d
    rows = []
    for nm, S in [("전체 논문(기준선)", set(U.cls.unique()))] + list(DEFS.items()):
        d0 = build(S)
        for lab, d in [("전체", d0), ("상위4 제외", d0[~d0.corp_norm.isin(TOP4)])]:
            d = d[d.groupby("corp_norm")["n_out_s"].transform("sum") > 0]
            try:
                mod = smf.glm("n_out_s ~ ln_pat + ln_pre_s + C(corp_norm) + C(domain)", data=d, family=sm.families.Poisson(), missing="drop")
                m = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.loc[mod.data.row_labels, "corp_norm"])[0]}, maxiter=300)
                rows.append(dict(정의=nm, 표본=lab, 기업=d.corp_norm.nunique(), 과거논문탄성=m.params["ln_pre_s"], se1=m.bse["ln_pre_s"], 특허효과=m.params["ln_pat"], se2=m.bse["ln_pat"], p_특허=m.pvalues["ln_pat"]))
            except Exception as e: rows.append(dict(정의=nm, 표본=lab, 과거논문탄성=np.nan))
    print("\n[D-1] PPML n_out_s ~ ln_pat + ln_pre_s (기업·분야 FE). 특허효과가 core에서 더 크면 '특허가 GX 핵심 과학을 끈다', 같으면 정렬 없음"); print(pd.DataFrame(rows).round(3).to_string(index=False))
safe(Dd)

# ---- (E) 일본 구조: 분류별 파트너 유형 구성
def E():
    L = LK.copy(); L["blk"] = ((L["year"] - YMIN) // BLK_).astype(int)
    grp = np.select([(L.country == "JP") & (L.itype == "education"), (L.country == "JP") & L.itype.isin(["facility", "government"]), (L.country == "JP") & (L.itype == "company"), (L.country != "JP") & (L.itype == "education")],
                    ["국내대학", "국내공공연구", "국내기업", "해외대학"], "기타")
    L["g"] = grp; L = L.merge(U[["w", "corp_norm", "cls"]].drop_duplicates(["w", "corp_norm"]), on=["w", "corp_norm"], how="left")
    P = L.drop_duplicates(["w", "corp_norm", "g"]).assign(v=1).pivot_table(index=["w", "corp_norm", "blk", "cls"], columns="g", values="v", fill_value=0).reset_index()
    for k in ["core", "non_gx"]:
        s = P[P.cls == k].groupby("blk")[[c for c in ["국내대학", "국내공공연구", "국내기업", "해외대학"] if c in P.columns]].mean(); s.index = [LAB[b] for b in s.index]
        print(f"\n[E] {k} 논문의 파트너 유형별 포함 비중 (논문당 1개 이상 포함, 중복 가능)"); print(s.round(3).to_string())
safe(E)

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import os, numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# ---- (0) 분류 패치: src 보존, 명백한 항목만. 정의 쟁점은 other로 두고 검토 시트에서 결정
T = pd.read_csv(f"{OUT}/topic_gx_tagging.csv")
PATCH = {"non_gx": ["Gold and Silver Nanoparticles Synthesis and Applications", "Nanoparticle-Based Drug Delivery", "bioluminescence and chemiluminescence research",
                    "Characterization and Applications of Magnetic Nanoparticles", "Nanoparticles: synthesis and applications", "Underwater Vehicles and Communication Systems"],
         "legacy": ["Vehicle emissions and performance"],
         "other": ["Nuclear Engineering Thermal-Hydraulics", "Urban Heat Island Mitigation", "Heat and Mass Transfer in Porous Media", "Heat Transfer Mechanisms",
                   "Catalysis and Oxidation Reactions", "Surface Chemistry and Catalysis", "Thermal Expansion and Ionic Conductivity", "Silicon Nanostructures and Photoluminescence"]}
for k, ts in PATCH.items():
    h = T["topic"].isin(ts); T.loc[h, "gx_class"] = k; T.loc[h, "src"] = "claude_patch"
print("패치 주제:", int((T.src == "claude_patch").sum()), "/", sum(len(v) for v in PATCH.values()))
fin = f"{OUT}/gx_review_final.csv"
if os.path.exists(fin):
    F_ = pd.read_csv(fin)[["topic", "gx_class_final"]].dropna(); mp = dict(zip(F_.topic, F_.gx_class_final)); h = T["topic"].isin(mp)
    T.loc[h, "gx_class"] = T.loc[h, "topic"].map(mp); T.loc[h, "src"] = "final"
T.to_csv(f"{OUT}/topic_gx_tagging.csv", index=False, encoding="utf-8-sig")
CLS = T.set_index("topic")["gx_class"].to_dict()
U["cls"] = U["topic"].map(CLS).fillna("unreviewed"); PF["cls"] = PF["topic"].map(CLS).fillna("unreviewed")

# ---- (1) 연 환산 정정(마지막 블록 2024까지) + 상위4 vs 그 외 + core 비중 shift-share
NY = {b: min(YMIN + b * BLK + BLK - 1, YLAST) - (YMIN + b * BLK) + 1 for b in BL}
Ud = U[U.year <= YLAST].copy()
def per_year(d, col=None):
    x = (d if col is None else d[d[col] == 1]).groupby(["blk", "cls"]).size().unstack(fill_value=0).reindex(BL, fill_value=0)
    return x.div(pd.Series(NY), axis=0)
for nm, col in [("전체 논문", None), ("단독 논문", "solo")]:
    t = per_year(Ud, col); dl = t.loc[b1] - t.loc[b0]
    print(f"\n[1] {nm} 연 환산 (마지막 블록 {NY[b1]}년 기준)"); print(t.rename(index=LAB).round(1).to_string())
    print("  증감:", dl.round(1).to_dict(), "| 순변화", round(dl.sum(), 1), "| core 기여", f"{dl.get('core', 0) / dl.sum():.1%}")
for lab, g in [("상위4", Ud[Ud.corp_norm.isin(TOP4)]), ("그 외", Ud[~Ud.corp_norm.isin(TOP4)])]:
    t = per_year(g); o = pd.DataFrame({"core": t["core"], "전체": t.sum(1)}); o["core비중"] = o.core / o.전체; o.index = [LAB[b] for b in o.index]
    print(f"\n[1-b] {lab}: core 연 편수와 비중"); print(o.round(3).to_string())
a, z = Ud[Ud.blk == b0], Ud[Ud.blk == b1]
w0, w1 = a.corp_norm.value_counts(normalize=True), z.corp_norm.value_counts(normalize=True); f = w0.index.union(w1.index); w0, w1 = w0.reindex(f, fill_value=0), w1.reindex(f, fill_value=0)
s0 = a.assign(c=a.cls == "core").groupby("corp_norm")["c"].mean().reindex(f); s1 = z.assign(c=z.cls == "core").groupby("corp_norm")["c"].mean().reindex(f); s0, s1 = s0.fillna(s1), s1.fillna(s0)
S = pd.DataFrame({"w0": w0, "w1": w1, "s0": s0, "s1": s1}); S["기업내"] = (S.w0 + S.w1) / 2 * (S.s1 - S.s0); S["기업간"] = (S.w1 - S.w0) * (S.s0 + S.s1) / 2
print(f"\n[1-c] core 비중 변화 {100*(S.기업내.sum()+S.기업간.sum()):+.1f}pp = 기업 내 {100*S.기업내.sum():+.1f} + 기업 간 {100*S.기업간.sum():+.1f}")
print(S.reindex(S.기업내.abs().sort_values(ascending=False).index).head(8).round(3).to_string())

# ---- (2) 단독 확률: core와 비core의 변화 차이 (형식 검정)
d = Ud[Ud.blk.isin([b0, b1]) & Ud.cls.isin(["core", "enabling", "legacy", "non_gx", "other"])].copy(); d["post"] = (d.blk == b1).astype(int); d["core"] = (d.cls == "core").astype(int)
m = cl_ols("solo ~ post*core + C(corp_norm) + C(domain)", d)
print(f"\n[2] 단독 확률 ~ post*core: 비core 변화 {100*m.params['post']:+.1f}pp (p={m.pvalues['post']:.3f}) | core 추가차 {100*m.params['post:core']:+.1f}pp (se {100*m.bse['post:core']:.1f}, p={m.pvalues['post:core']:.3f})")

# ---- (3) 파트너 유형: 모든 논문(단독 포함)을 분모로, core x post
PT = {"국내대학": "p_domuniv", "국내공공연구": "p_pub", "국내기업": "p_firm", "해외대학": "p_foruniv"}
L = LK.copy(); L["g"] = np.select([(L.country == "JP") & (L.itype == "education"), (L.country == "JP") & L.itype.isin(["facility", "government"]),
                                   (L.country == "JP") & (L.itype == "company"), (L.country != "JP") & (L.itype == "education")], list(PT), "기타")
Pw = L[L.g != "기타"].drop_duplicates(["w", "corp_norm", "g"]).assign(v=1).pivot_table(index=["w", "corp_norm"], columns="g", values="v", fill_value=0).rename(columns=PT).reset_index()
X = Ud.merge(Pw, on=["w", "corp_norm"], how="left")
for c in PT.values(): X[c] = X[c].fillna(0) if c in X.columns else 0.0
Xd = X[X.blk.isin([b0, b1]) & X.cls.isin(["core", "enabling", "legacy", "non_gx", "other"])].copy(); Xd["post"] = (Xd.blk == b1).astype(int); Xd["core"] = (Xd.cls == "core").astype(int)
print("\n[3-a] 파트너 포함 비중 (단독 논문 포함 전체 분모)"); print(Xd.groupby(["core", "blk"])[list(PT.values())].mean().rename(index=LAB, level=1).round(3).to_string())
rows = []
for c in PT.values():
    mm = cl_ols(f"{c} ~ post*core + C(corp_norm)", Xd); rows.append(dict(파트너=c, 비core_변화pp=100 * mm.params["post"], core_추가pp=100 * mm.params["post:core"], se=100 * mm.bse["post:core"], p=mm.pvalues["post:core"]))
R = pd.DataFrame(rows); R["p_holm"] = multipletests(R.p, method="holm")[1]; print("\n[3-b] core 추가 변화 (기업 FE)"); print(R.round(3).to_string(index=False))

# ---- (4) 수정본: concat 뒤 reset_index(drop=True)
D0 = pd.read_csv("panel_new_design.csv"); bs = [2, 3]
dom_core = sorted(U.loc[U.cls == "core", "domain"].unique()); print("[4] core가 있는 분야:", dom_core)
def cell(Sset, lab):
    o = U[U.blk.isin(bs) & U.cls.isin(Sset)].groupby(["corp_norm", "domain"]).size().rename("y").reset_index()
    p = U[(U.blk == b0) & U.cls.isin(Sset)].groupby(["corp_norm", "domain"]).size().rename("pre").reset_index()
    d = D0[["corp_norm", "domain", "ln_pat"]].merge(o, how="left", on=["corp_norm", "domain"]).merge(p, how="left", on=["corp_norm", "domain"]).fillna({"y": 0, "pre": 0})
    d["ln_pre"] = np.log1p(d.pre); d["isc"] = lab; return d
St = pd.concat([cell({"core"}, 1), cell(set(U.cls.unique()) - {"core"}, 0)], ignore_index=True)
St = St[St.domain.isin(dom_core)].copy()
St["ix_pre"] = St.ln_pre * St.isc; St["ix_pat"] = St.ln_pat * St.isc
St = St[St.groupby("corp_norm")["y"].transform("sum") > 0].reset_index(drop=True)     # 인덱스 재설정이 핵심
print(f"  표본 {len(St)}행 (core {int(St.isc.sum())} / 비core {int((1 - St.isc).sum())}), 기업 {St.corp_norm.nunique()}")
mod = smf.glm("y ~ ln_pat + ln_pre + isc + ix_pre + ix_pat + C(corp_norm) + C(domain)", data=St, family=sm.families.Poisson(), missing="drop")
mp_ = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(St.loc[mod.data.row_labels, "corp_norm"])[0]}, maxiter=300)
for k, nm in [("ln_pre", "비core 과거 탄성"), ("ix_pre", "core - 비core 탄성 차이"), ("ln_pat", "비core 특허효과"), ("ix_pat", "core - 비core 특허효과 차이")]:
    print(f"  {nm:<22} {mp_.params[k]:+.3f} (se {mp_.bse[k]:.3f}, p={mp_.pvalues[k]:.3f})")

# ---- (5) core 진입의 기업 집중
Q = U[U.blk > b0]; e = Q[(Q.cls == "core") & (Q.entry == 1)]
print(f"\n[5] core 진입 논문 {len(e)} | 상위4 비중 {e.corp_norm.isin(TOP4).mean():.1%} | core 논문 중 상위4 비중 {Q[Q.cls == 'core'].corp_norm.isin(TOP4).mean():.1%}")

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf

Ud = U[U.year <= YLAST].copy(); FIRMS = ["トヨタ自動車", "日産自動車", "ホンダ", "三菱電機"]

# (a) core 논문의 기업 집중도
rows = []
for b in BL:
    c = Ud[(Ud.blk == b) & (Ud.cls == "core")].groupby("corp_norm").size().sort_values(ascending=False); s = c / c.sum()
    rows.append(dict(블록=LAB[b], core편수년=c.sum() / NY[b], 기업수=len(c), 상위1=s.iloc[0], 상위3=s.iloc[:3].sum(), 상위4=s.reindex(TOP4).fillna(0).sum(), HHI=(s ** 2).sum()))
print("[a] core 논문의 기업 집중"); print(pd.DataFrame(rows).round(3).to_string(index=False))

# (b) 기업별 core 연 편수 (상위 4개사 개별, 변화 상·하위)
t = Ud[Ud.cls == "core"].groupby(["corp_norm", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0).div(pd.Series(NY), axis=1)
t.columns = [LAB[b] for b in BL]; t["변화"] = t.iloc[:, -1] - t.iloc[:, 0]
print("\n[b-1] 상위 4개사 개별"); print(t.reindex([f for f in FIRMS if f in t.index]).round(1).to_string())
print("\n[b-2] 변화 하위 8"); print(t.sort_values("변화").head(8).round(1).to_string()); print("\n[b-3] 변화 상위 5"); print(t.sort_values("변화").tail(5).round(1).to_string())

# (c) 도요타·닛산·혼다 core 증가가 어느 분야에서 나왔나 (연 환산)
g = Ud[(Ud.cls == "core") & Ud.corp_norm.isin(FIRMS[:3])].groupby(["corp_norm", "domain", "blk"]).size().unstack(fill_value=0).reindex(columns=[b0, b1], fill_value=0)
g[b0] = g[b0] / NY[b0]; g[b1] = g[b1] / NY[b1]; g.columns = [LAB[b0], LAB[b1]]; g["변화"] = g.iloc[:, 1] - g.iloc[:, 0]
print("\n[c] 자동차 3사의 core 연 편수: 분야별"); print(g.round(1).to_string())

# (d) [4] 교란 점검: core와 비core가 함께 있는 분야(MATERIAL, THERMAL, VEHICLE)만, 분야 x core 교호 포함
S2 = St[St.domain.isin(["MATERIAL", "THERMAL", "VEHICLE"])].copy()
S2 = S2[S2.groupby("corp_norm")["y"].transform("sum") > 0].reset_index(drop=True)
mod = smf.glm("y ~ ln_pat + ln_pre + ix_pre + ix_pat + C(corp_norm) + C(domain)*isc", data=S2, family=sm.families.Poisson(), missing="drop")
m4 = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(S2.loc[mod.data.row_labels, "corp_norm"])[0]}, maxiter=300)
print(f"\n[d] 표본 {len(S2)}행, 기업 {S2.corp_norm.nunique()}")
for k, nm in [("ln_pre", "비core 과거 탄성"), ("ix_pre", "core - 비core 탄성 차이"), ("ln_pat", "비core 특허효과"), ("ix_pat", "core - 비core 특허효과 차이")]:
    print(f"  {nm:<22} {m4.params[k]:+.3f} (se {m4.bse[k]:.3f}, p={m4.pvalues[k]:.3f})")

# (e) [2] core 순변화의 신뢰구간, [3] 분야 FE 추가
d2 = Ud[Ud.blk.isin([b0, b1]) & Ud.cls.isin(["core", "enabling", "legacy", "non_gx", "other"])].copy(); d2["post"] = (d2.blk == b1).astype(int); d2["core"] = (d2.cls == "core").astype(int)
m2 = cl_ols("solo ~ post*core + C(corp_norm) + C(domain)", d2); nm_ = list(m2.params.index); r = np.zeros(len(nm_)); r[nm_.index("post")] = 1; r[nm_.index("post:core")] = 1; tt = m2.t_test(r)
print(f"\n[e-1] core 단독 순변화 {100*float(tt.effect[0]):+.1f}pp (se {100*float(np.ravel(tt.sd)[0]):.1f}, p={float(np.ravel(tt.pvalue)[0]):.3f})")
Xd["top4"] = Xd.corp_norm.isin(TOP4).astype(int)
for c_ in ["p_foruniv", "p_firm", "p_domuniv"]:
    for lab, dd in [("전체", Xd), ("MATERIAL·THERMAL·VEHICLE", Xd[Xd.domain.isin(["MATERIAL", "THERMAL", "VEHICLE"])])]:
        mm = cl_ols(f"{c_} ~ post*core + C(corp_norm) + C(domain)", dd)
        print(f"[e-2] {c_:<10} {lab:<26} core 추가 {100*mm.params['post:core']:+.1f}pp (se {100*mm.bse['post:core']:.1f}, p={mm.pvalues['post:core']:.3f}) N={int(mm.nobs)}")
print("\n[e-3] 해외대학 비중: 상위4 x core x 블록"); print(Xd.groupby(["top4", "core", "blk"])["p_foruniv"].agg(["mean", "size"]).rename(index=LAB, level=2).round(3).to_string())

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf

MIX = ["MATERIAL", "THERMAL", "VEHICLE"]; TOY = ["トヨタ自動車"]; TN = ["トヨタ自動車", "日産自動車"]
def ppml(d, formula):
    d = d.reset_index(drop=True); mod = smf.glm(formula, data=d, family=sm.families.Poisson(), missing="drop")
    return mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.loc[mod.data.row_labels, "corp_norm"])[0]}, maxiter=300)
BASE = "y ~ ln_pat + ln_pre + ix_pre + ix_pat + C(corp_norm) + C(domain)*isc"
def pr(lab, m): print(f"  {lab:<26} 비core 특허 {m.params['ln_pat']:+.3f} | 차이(ix_pat) {m.params['ix_pat']:+.3f} (se {m.bse['ix_pat']:.3f}, p={m.pvalues['ix_pat']:.3f}) | 탄성차 {m.params['ix_pre']:+.3f} (p={m.pvalues['ix_pre']:.3f})")
def keep(d): return d[d.groupby("corp_norm")["y"].transform("sum") > 0].reset_index(drop=True)

# (a) 셀(기업x분야) 고정효과: ln_pat의 주효과는 흡수되고 core 상호작용만 식별
S3 = S2.copy(); S3["cell"] = S3.corp_norm + "|" + S3.domain; S3 = S3[S3.groupby("cell")["y"].transform("sum") > 0].reset_index(drop=True)
print(f"[a] 셀 FE 표본 {len(S3)}행, 셀 {S3.cell.nunique()}")
m = ppml(S3, "y ~ ln_pre + ix_pre + ix_pat + isc + C(cell)")
print(f"  셀 FE: ix_pat {m.params['ix_pat']:+.3f} (se {m.bse['ix_pat']:.3f}, p={m.pvalues['ix_pat']:.3f})")

# (b) 도요타·닛산·상위4 제외
print("\n[b] 기업 제외")
for lab, ex in [("전체", []), ("도요타 제외", TOY), ("도요타·닛산 제외", TN), ("상위4 제외", TOP4)]:
    try: pr(lab, ppml(keep(S2[~S2.corp_norm.isin(ex)]), BASE))
    except Exception as e: print(f"  {lab}: 실패 {type(e).__name__}")

# (c) 한 기업씩 제외(jackknife)
res = []
for f in S2.corp_norm.unique():
    try: m = ppml(keep(S2[S2.corp_norm != f]), BASE); res.append((f, m.params["ix_pat"], m.pvalues["ix_pat"]))
    except Exception: pass
R = pd.DataFrame(res, columns=["제외기업", "ix_pat", "p"]).sort_values("ix_pat")
print(f"\n[c] jackknife {len(R)}회: 범위 {R.ix_pat.min():+.3f} ~ {R.ix_pat.max():+.3f} | p<0.05 비율 {(R.p < 0.05).mean():.2f}"); print(R.head(5).round(3).to_string(index=False)); print(R.tail(3).round(3).to_string(index=False))

# (d) 정의 점검: 분류 경계 항목 제외, 위약 대조(enabling vs non_gx), core vs non_gx만
CONT = ["Catalysts for Methane Reforming", "CO2 Reduction Techniques and Catalysts", "Carbon dioxide utilization in catalysis"]
def stack(A, B, Udf):
    def cz(Sset, lab):
        o = Udf[Udf.blk.isin(bs) & Udf.cls.isin(Sset)].groupby(["corp_norm", "domain"]).size().rename("y").reset_index()
        p = Udf[(Udf.blk == b0) & Udf.cls.isin(Sset)].groupby(["corp_norm", "domain"]).size().rename("pre").reset_index()
        d = D0[["corp_norm", "domain", "ln_pat"]].merge(o, how="left", on=["corp_norm", "domain"]).merge(p, how="left", on=["corp_norm", "domain"]).fillna({"y": 0, "pre": 0})
        d["ln_pre"] = np.log1p(d.pre); d["isc"] = lab; return d
    S = pd.concat([cz(A, 1), cz(B, 0)], ignore_index=True)
    dm = [x for x in S.domain.unique() if S[(S.domain == x) & (S.isc == 1)].y.sum() >= 30 and S[(S.domain == x) & (S.isc == 0)].y.sum() >= 30]
    S = S[S.domain.isin(dm)].copy(); S["ix_pre"] = S.ln_pre * S.isc; S["ix_pat"] = S.ln_pat * S.isc; return keep(S), dm
U2 = U.copy(); U2.loc[U2.topic.isin(CONT), "cls"] = "contested"
allc = set(U.cls.unique())
print("\n[d] 정의 점검")
for lab, A, B, Udf in [("core vs 나머지(경계 3개 제외)", {"core"}, allc - {"core", "contested"}, U2), ("core vs non_gx만", {"core"}, {"non_gx"}, U),
                      ("위약: enabling vs non_gx", {"enabling"}, {"non_gx"}, U)]:
    try:
        S, dm = stack(A, B, Udf); frm = BASE if len(dm) > 1 else "y ~ ln_pat + ln_pre + ix_pre + ix_pat + C(corp_norm) + isc"
        print(f"  [{lab}] 분야 {dm}, {len(S)}행"); pr("  ", ppml(S, frm))
    except Exception as e: print(f"  {lab}: 실패 {type(e).__name__}: {str(e)[:100]}")

# (e) 해외대학 비중: 모든 블록, 혼합 분야, 분야별 시간 추세 통제
X2 = X[X.cls.isin(["core", "enabling", "legacy", "non_gx", "other"]) & X.domain.isin(MIX) & (X.year <= YLAST)].copy(); X2["core"] = (X2.cls == "core").astype(int); X2["t"] = (X2.year - 2015) / 10
print("\n[e] 혼합 3개 분야, 전 기간, 분야별 시간 추세 통제: core의 시간 기울기 추가분(10년당 pp)")
for c in ["p_foruniv", "p_firm", "p_domuniv", "p_pub"]:
    mm = cl_ols(f"{c} ~ t*core + C(corp_norm) + C(domain)*t", X2); print(f"  {c:<10} {100*mm.params['t:core']:+.1f}pp (se {100*mm.bse['t:core']:.1f}, p={mm.pvalues['t:core']:.3f}) N={int(mm.nobs)}")

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import numpy as np, pandas as pd

Ud = U[U.year <= YLAST].copy()
TOY = "トヨタ自動車"
CONT = ["Catalysts for Methane Reforming", "CO2 Reduction Techniques and Catalysts", "Carbon dioxide utilization in catalysis"]

def core_counts(d):
    """기업x블록 core 연 환산 편수"""
    t = d[d.cls == "core"].groupby(["corp_norm", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0)
    return t.div(pd.Series(NY), axis=1)

def hhi_table(t, exclude=None):
    rows = []
    for b in BL:
        c = t[b][t[b] > 0]
        if exclude: c = c[~c.index.isin(exclude)]
        if c.sum() == 0: continue
        s = c / c.sum()
        rows.append(dict(블록=LAB[b], core편수년=c.sum(), 기업수=len(c), 상위1기업=s.idxmax(), 상위1점유=s.max(),
                         상위3점유=s.nlargest(3).sum(), HHI=(s ** 2).sum(), 유효기업수=1 / (s ** 2).sum()))
    return pd.DataFrame(rows)

T_all = core_counts(Ud)

# ---- (A) 도요타 포함/제외 HHI (유효 기업 수 = 1/HHI, 기업 수 차이를 보정해 읽기 위함)
print("[A-1] 도요타 포함"); print(hhi_table(T_all).round(3).to_string(index=False))
print("\n[A-2] 도요타 제외 (나머지 기업 점유율 재정규화)"); print(hhi_table(T_all, exclude=[TOY]).round(3).to_string(index=False))
print("\n[A-3] 도요타·닛산 제외"); print(hhi_table(T_all, exclude=[TOY, "日産自動車"]).round(3).to_string(index=False))

# ---- (B) 기업 진입·이탈: 첫 블록 vs 마지막 블록, 그리고 블록별 이동
t = T_all.copy(); t.columns = [LAB[b] for b in BL]; f, l = t.columns[0], t.columns[-1]
t["변화"] = t[l] - t[f]
def grp(r):
    if r[f] == 0 and r[l] > 0: return "신규(첫 블록 0)"
    if r[f] > 0 and r[l] == 0: return "소멸(마지막 블록 0)"
    if r[f] > 0 and r[l] > 0:
        ratio = r[l] / r[f]
        return "크게 감소(<50%)" if ratio < 0.5 else ("크게 증가(>200%)" if ratio > 2 else "유지(50~200%)")
    return "core 없음"
t["유형"] = t.apply(grp, axis=1); t = t[t["유형"] != "core 없음"]
S = t.groupby("유형").agg(기업수=("변화", "size"), 첫블록편수=(f, "sum"), 마지막편수=(l, "sum"), 순변화=("변화", "sum")).round(1)
print("\n[B-1] 기업 유형별 core 변화 (첫 블록 → 마지막 블록)"); print(S.to_string())
print("\n[B-2] 소멸·크게 감소 기업"); print(t[t.유형.isin(["소멸(마지막 블록 0)", "크게 감소(<50%)"])].sort_values("변화").round(1).to_string())
print("\n[B-3] 신규·크게 증가 기업"); print(t[t.유형.isin(["신규(첫 블록 0)", "크게 증가(>200%)"])].sort_values("변화", ascending=False).round(1).to_string())

# 블록별 이동(진입/이탈이 단계마다 몇 곳인지)
mv = []
for a, b in zip(BL[:-1], BL[1:]):
    x, y = T_all[a], T_all[b]
    mv.append(dict(구간=f"{LAB[a]}→{LAB[b]}", 신규진입=int(((x == 0) & (y > 0)).sum()), 소멸=int(((x > 0) & (y == 0)).sum()),
                   유지기업=int(((x > 0) & (y > 0)).sum())))
print("\n[B-4] 블록 간 진입·소멸 기업 수"); print(pd.DataFrame(mv).to_string(index=False))

# 이탈 기업의 편수가 enabling/other로 이동했는지 (분류 경계 영향 점검)
firms_down = t[t.유형.isin(["소멸(마지막 블록 0)", "크게 감소(<50%)"])].index
x = Ud[Ud.corp_norm.isin(firms_down)].groupby(["blk", "cls"]).size().unstack(fill_value=0).reindex(BL, fill_value=0)
x = x.div(pd.Series(NY), axis=0); x.index = [LAB[b] for b in x.index]
print("\n[B-5] core 감소·소멸 기업군의 분류별 연 편수 (core 밖으로 옮겨갔는지)"); print(x.round(1).to_string())

# ---- (C) 경계 주제 3개를 core에서 뺀 버전
U2 = Ud.copy(); U2.loc[U2.topic.isin(CONT), "cls"] = "contested"
T_c = core_counts(U2)
print("\n[C-1] 경계 3개 제외 core: 도요타 포함"); print(hhi_table(T_c).round(3).to_string(index=False))
print("\n[C-2] 경계 3개 제외 core: 도요타 제외"); print(hhi_table(T_c, exclude=[TOY]).round(3).to_string(index=False))
cmp = pd.DataFrame({"core(원본)": T_all.sum(), "core(경계제외)": T_c.sum(), "도요타(원본)": T_all.loc[TOY], "도요타(경계제외)": T_c.loc[TOY]}); cmp.index = [LAB[b] for b in cmp.index]
cmp["도요타점유(원본)"] = cmp["도요타(원본)"] / cmp["core(원본)"]; cmp["도요타점유(경계제외)"] = cmp["도요타(경계제외)"] / cmp["core(경계제외)"]
print("\n[C-3] 원본 vs 경계 제외"); print(cmp.round(3).to_string())

# ---- (D) 부트스트랩: 기업 재표집으로 HHI 변화(첫→마지막)의 구간 (도요타 포함/제외)
rng = np.random.default_rng(0); firms = list(T_all.index)
def dh(tbl, excl):
    h = hhi_table(tbl, exclude=excl); return h.HHI.iloc[-1] - h.HHI.iloc[0]
res = {"포함": [], "도요타 제외": []}
for _ in range(500):
    pick = rng.choice(len(firms), len(firms)); tb = T_all.iloc[pick].copy(); tb.index = [f"{i}_{k}" for k, i in enumerate(tb.index)]
    try: res["포함"].append(dh(tb, None))
    except Exception: pass
    keep = [i for i in tb.index if not i.startswith(TOY + "_")]
    try: res["도요타 제외"].append(dh(tb.loc[keep], None))
    except Exception: pass
print("\n[D] HHI 변화(마지막-첫 블록) 기업 재표집 95% 구간 (도요타가 표본에 뽑히면 포함 버전에서 변화가 큼)")
for k, v in res.items():
    if v: print(f"  {k:<8} 점추정 {dh(T_all, None if k=='포함' else [TOY]):+.3f} | 구간 [{np.percentile(v,2.5):+.3f}, {np.percentile(v,97.5):+.3f}] | 반복 {len(v)}")

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import numpy as np, pandas as pd, statsmodels.formula.api as smf

Ud = U[U.year <= YLAST].copy()
TOY, NIS = "トヨタ自動車", "日産自動車"
CONT = ["Catalysts for Methane Reforming", "CO2 Reduction Techniques and Catalysts", "Carbon dioxide utilization in catalysis"]
SOLID = ["Thermal Expansion and Ionic Conductivity"]
NUC = ["Nuclear Engineering Thermal-Hydraulics"]

# 규칙 임시분류 주제 목록 (src 열이 있으면 사용)
try:
    Tt = pd.read_csv(f"{OUT}/topic_gx_tagging.csv"); RULE_T = set(Tt.loc[Tt.src.isin(["rule_provisional"]), "topic"])
except Exception:
    RULE_T = set()

def scen(name):
    d = Ud.copy(); c = d["cls"].copy()
    if name == "기본": pass
    elif name == "엄격(경계3 제외)": c[d.topic.isin(CONT)] = "contested"
    elif name == "확장(+고체전해질+원자력)": c[d.topic.isin(SOLID + NUC)] = "core"
    elif name == "other 전부 core": c[c == "other"] = "core"
    elif name == "규칙임시 제외": d = d[~d.topic.isin(RULE_T) & (d.cls != "unreviewed")]; c = d["cls"]
    d["cls"] = c; return d

def core_tbl(d):
    t = d[d.cls == "core"].groupby(["corp_norm", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0)
    return t.div(pd.Series(NY), axis=1)

def hhi(t, excl=()):
    out = []
    for b in BL:
        c = t[b][t[b] > 0]; c = c[~c.index.isin(excl)]
        s = c / c.sum() if c.sum() > 0 else c
        out.append((s ** 2).sum() if len(s) else np.nan)
    return out

# ---- (1)(2) 분류 시나리오별 핵심 지표
rows = []
for nm in ["기본", "엄격(경계3 제외)", "확장(+고체전해질+원자력)", "other 전부 core", "규칙임시 제외"]:
    d = scen(nm); t = core_tbl(d); tot = t.sum()
    tsh = (t.loc[TOY] / tot) if TOY in t.index else pd.Series(np.nan, index=BL)
    h1, h2, h3 = hhi(t), hhi(t, [TOY]), hhi(t, [TOY, NIS])
    rows.append(dict(시나리오=nm, core_첫=tot[b0], core_정점=tot.max(), core_끝=tot[b1], 도요타점유_첫=tsh[b0], 도요타점유_끝=tsh[b1],
                     HHI포함_첫=h1[0], HHI포함_끝=h1[-1], HHI도요타제외_첫=h2[0], HHI도요타제외_끝=h2[-1], HHI도요타닛산제외_첫=h3[0], HHI도요타닛산제외_끝=h3[-1]))
R = pd.DataFrame(rows).set_index("시나리오")
print("[1-2] 분류 시나리오별 core 집중 (끝 블록은 4년 환산)"); print(R.round(3).T.to_string())
chk = pd.DataFrame({"점유상승": R.도요타점유_끝 > R.도요타점유_첫, "도요타제외HHI상승": R.HHI도요타제외_끝 > R.HHI도요타제외_첫,
                    "도요타닛산제외변화(끝-첫)": (R.HHI도요타닛산제외_끝 - R.HHI도요타닛산제외_첫).round(3)})
print("\n판정표"); print(chk.to_string())

# ---- (3) 닛산 점검: 분야·주제별 core 구성, 증가 기여
n = Ud[(Ud.corp_norm == NIS) & (Ud.cls == "core")]
g = n.groupby(["domain", "blk"]).size().unstack(fill_value=0).reindex(columns=[b0, b1], fill_value=0); g[b0] /= NY[b0]; g[b1] /= NY[b1]; g.columns = [LAB[b0], LAB[b1]]; g["변화"] = g.iloc[:, 1] - g.iloc[:, 0]
print("\n[3-a] 닛산 core 연 편수: 분야별"); print(g.round(1).to_string())
tp = n.groupby(["topic", "blk"]).size().unstack(fill_value=0).reindex(columns=[b0, b1], fill_value=0); tp[b0] /= NY[b0]; tp[b1] /= NY[b1]; tp.columns = [LAB[b0], LAB[b1]]; tp["변화"] = tp.iloc[:, 1] - tp.iloc[:, 0]
print("\n[3-b] 닛산 core 주제별 변화 상·하위 8"); print(pd.concat([tp.sort_values("변화").head(3), tp.sort_values("변화").tail(5)]).round(1).to_string())
tn = core_tbl(Ud); no = tn.drop(index=[TOY], errors="ignore")
print(f"\n[3-c] 도요타 제외 core 증감 {no[b1].sum()-no[b0].sum():+.1f}편/년 중 닛산 {no.loc[NIS,b1]-no.loc[NIS,b0]:+.1f}")
# 닛산 증가 상위 주제(경계 주제 포함 여부)
print("[3-d] 닛산 core 중 경계 3개 주제 비중:", {LAB[b]: round(float(n[(n.blk == b) & n.topic.isin(CONT)].shape[0] / max(n[n.blk == b].shape[0], 1)), 3) for b in [b0, b1]})

# ---- (4) core 감소·소멸 기업군: post x core 형식 검정 (기업 FE)
t = core_tbl(Ud); f_, l_ = t[b0], t[b1]
down = [c for c in t.index if f_[c] > 0 and l_[c] < 0.5 * f_[c]]
print(f"\n[4] 감소·소멸 기업 {len(down)}곳 (첫 블록 >0, 끝 블록 <50%)")
d = Ud[Ud.corp_norm.isin(down) & Ud.blk.isin([b0, b1]) & Ud.cls.isin(["core", "enabling", "legacy", "non_gx", "other"])]
cnt = d.groupby(["corp_norm", "blk", "cls"]).size().rename("n").reset_index()
grid = pd.MultiIndex.from_product([down, [b0, b1], ["core", "enabling", "legacy", "non_gx", "other"]], names=["corp_norm", "blk", "cls"]).to_frame(index=False)
cnt = grid.merge(cnt, how="left").fillna({"n": 0}); cnt["yrs"] = cnt.blk.map(NY); cnt["rate"] = cnt.n / cnt.yrs
cnt["post"] = (cnt.blk == b1).astype(int); cnt["core"] = (cnt.cls == "core").astype(int); cnt["lr"] = np.log1p(cnt.rate)
m = smf.ols("lr ~ post*core + C(corp_norm)", data=cnt).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(cnt.corp_norm)[0]})
print(f"  ln(1+연 편수) ~ post*core (기업 FE, 군집): 비core 변화 {m.params['post']:+.3f} (p={m.pvalues['post']:.3f}) | core 추가 {m.params['post:core']:+.3f} (se {m.bse['post:core']:.3f}, p={m.pvalues['post:core']:.3f})")
by = cnt.groupby(["cls", "blk"])["rate"].sum().unstack().rename(columns=LAB); by["변화율"] = by.iloc[:, 1] / by.iloc[:, 0] - 1
print("  분류별 합계 연 편수와 변화율"); print(by.round(2).to_string())
# 대조: 도요타·닛산 외 '유지/증가' 기업군에서도 같은 검정 (core 특이성 비교)
others = [c for c in t.index if c not in down and c not in (TOY, NIS) and f_[c] > 0]
d2 = Ud[Ud.corp_norm.isin(others) & Ud.blk.isin([b0, b1]) & Ud.cls.isin(["core", "enabling", "legacy", "non_gx", "other"])]
c2 = d2.groupby(["corp_norm", "blk", "cls"]).size().rename("n").reset_index()
g2 = pd.MultiIndex.from_product([others, [b0, b1], ["core", "enabling", "legacy", "non_gx", "other"]], names=["corp_norm", "blk", "cls"]).to_frame(index=False)
c2 = g2.merge(c2, how="left").fillna({"n": 0}); c2["lr"] = np.log1p(c2.n / c2.blk.map(NY)); c2["post"] = (c2.blk == b1).astype(int); c2["core"] = (c2.cls == "core").astype(int)
m2 = smf.ols("lr ~ post*core + C(corp_norm)", data=c2).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(c2.corp_norm)[0]})
print(f"  [대조: 나머지 core 보유 기업 {len(others)}곳] 비core 변화 {m2.params['post']:+.3f} | core 추가 {m2.params['post:core']:+.3f} (p={m2.pvalues['post:core']:.3f})")

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import numpy as np, pandas as pd, statsmodels.formula.api as smf

Ud = U[U.year <= YLAST].copy(); TOY, NIS = "トヨタ自動車", "日産自動車"
CLS5 = ["core", "enabling", "legacy", "non_gx", "other"]

# ---------- (1) 선택 무관 기업군에서 post x core ----------
def post_core(firms, a, z, label):
    d = Ud[Ud.corp_norm.isin(firms) & Ud.blk.isin([a, z]) & Ud.cls.isin(CLS5)]
    cnt = d.groupby(["corp_norm", "blk", "cls"]).size().rename("n").reset_index()
    grid = pd.MultiIndex.from_product([sorted(firms), [a, z], CLS5], names=["corp_norm", "blk", "cls"]).to_frame(index=False)
    c = grid.merge(cnt, how="left").fillna({"n": 0}); c["lr"] = np.log1p(c.n / c.blk.map(NY))
    c["post"] = (c.blk == z).astype(int); c["core"] = (c.cls == "core").astype(int)
    m = smf.ols("lr ~ post*core + C(corp_norm)", data=c).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(c.corp_norm)[0]})
    print(f"  {label:<26} 기업 {len(firms):>2} | 비core {m.params['post']:+.3f} | core 추가 {m.params['post:core']:+.3f} (se {m.bse['post:core']:.3f}, p={m.pvalues['post:core']:.3f})")

n0 = Ud[Ud.blk == b0].groupby("corp_norm").size()
n0 = n0[~n0.index.isin([TOY, NIS])]
terc = pd.qcut(n0.rank(method="first"), 3, labels=["소", "중", "대"])
print("[1] 첫 블록 논문 수 3분위 (도요타·닛산 제외), 구간:", n0.groupby(terc).agg(["min", "max", "size"]).to_dict("index"))
for a, z, nm in [(b0, b1, "첫→끝"), (b0 + 1, b1, "정점 이후(2011-15→끝)")]:
    print(f"\n 창: {nm}")
    post_core(list(n0.index), a, z, "전체(도요타·닛산 제외)")
    for g in ["소", "중", "대"]:
        post_core(list(n0.index[terc == g]), a, z, f"n0 {g}")
try:
    fm = pd.read_csv("firm_master_window.csv").drop_duplicates("corp_norm").set_index("corp_norm")
    ind = fm["industry"].reindex(n0.index); vc = ind.value_counts()
    print("\n 업종별(4곳 이상, 첫→끝)")
    for k in vc[vc >= 4].index: post_core(list(ind[ind == k].index), b0, b1, f"업종 {k}")
except Exception as e: print("업종 분할 생략:", type(e).__name__)

# ---------- (2) 꼬임점 p 재산출 (시간 오래 걸림, ALL·VEHICLE만 보면 됨) ----------
NB_BRK = 999
safe(Q6a)

# ---------- (3) 진입 플래그 불일치 원인 ----------
A = PF.groupby(["corp_norm", "topic_id"]).agg(fb=("blk", "min")).reset_index()
chk = PF[PF.blk > b0].merge(A, on=["corp_norm", "topic_id"]); chk = chk[chk.blk == chk.fb]
bad = chk[chk.entry == 0].copy(); good = chk[chk.entry == 1]
print(f"\n[3] 첫 등장 논문 {len(chk)}건 중 entry=0 {len(bad)}건")
print(" 블록별:", bad.groupby("blk").size().to_dict(), "| entry=1:", good.groupby("blk").size().to_dict())
prev = PF.groupby(["corp_norm", "blk"]).size()
bad["직전블록_논문"] = [prev.get((c, b - 1), 0) for c, b in zip(bad.corp_norm, bad.blk)]
print(" 직전 블록 논문 수 분포(0이면 해당 기업 그 블록 활동 없음):"); print(bad["직전블록_논문"].describe().round(1).to_string())
print(" 직전 블록 논문 0 비중:", round((bad["직전블록_논문"] == 0).mean(), 3), "| 분야별:", bad.groupby("domain").size().to_dict())
print(" 같은 기업의 topic(이름) 기준 선행 등장 여부:", int(bad.apply(lambda r: ((PF.corp_norm == r.corp_norm) & (PF.topic == r.topic) & (PF.blk < r.blk)).any(), axis=1).sum()), "건")

# ---------- (4) 島津製作所 제외 민감도 (해외 대학 협력, 기업 저자 위치) ----------
def pos_table(d, lab):
    d = d[d.for_univ]
    pos = np.select([d.first_firm & d.last_firm, d.first_firm & ~d.last_firm, ~d.first_firm & d.last_firm],
                    ["제1+마지막", "제1(외부 마지막)", "마지막(외부 제1)"], "중간저자만")
    t = pd.Series(pos, index=d.index).groupby([pos, d.blk]).size().unstack(fill_value=0).reindex(columns=[b0, b1], fill_value=0)
    t[b0] /= NY[b0]; t[b1] /= NY[b1]; t.columns = [LAB[b0], LAB[b1]]; t["변화"] = t.iloc[:, 1] - t.iloc[:, 0]
    print(f"\n[4] {lab}: 해외 대학 포함 논문 {t.iloc[:,0].sum():.1f}→{t.iloc[:,1].sum():.1f}편/년"); print(t.round(1).to_string())
pos_table(Ud, "전체")
pos_table(Ud[Ud.corp_norm != "島津製作所"], "島津製作所 제외")
pos_table(Ud[~Ud.corp_norm.isin(["島津製作所", TOY])], "島津·도요타 제외")

# ---------- (5) 논문 매칭 없는 기업 vs 분석 기업 ----------
try:
    fm = pd.read_csv("firm_master_window.csv").drop_duplicates("corp_norm").set_index("corp_norm")
    inU = fm.index.isin(set(Ud.corp_norm)); fm["분석포함"] = inU
    print(f"\n[5] 마스터 {len(fm)}곳 중 분석 포함 {inU.sum()}곳 (Ud 기업 수 {Ud.corp_norm.nunique()})")
    print(fm.groupby("분석포함")["n_win"].describe().round(1).to_string())
    print(" 미포함 기업:", list(fm.index[~inU])[:20])
    print(" 업종 분포:"); print(pd.crosstab(fm["industry"], fm["분석포함"]).to_string())
except Exception as e: print("[5] 실패:", type(e).__name__, str(e)[:150])

In [ ]:
#

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

OUT = "oa_out"
R = pd.read_csv(f"{OUT}/gx_review_sheet_revised.csv")
print("열:", R.columns.tolist(), "| 행:", len(R))

# 평가자 열 이름 자동 탐색 (gx_class_final이 기본)
rc = "gx_class_final" if "gx_class_final" in R.columns else next(c for c in R.columns if "final" in c or "rater" in c)
R = R[["topic", rc] + [c for c in ["memo"] if c in R.columns]].rename(columns={rc: "rater"})
R["rater"] = R["rater"].astype(str).str.strip().str.lower().replace({"nan": "", "non-gx": "non_gx", "non gx": "non_gx"})
print("평가자 입력 값 분포:", R.rater.value_counts(dropna=False).to_dict())

T = pd.read_csv(f"{OUT}/topic_gx_tagging.csv")[["topic", "gx_class", "src", "n_total"]]
M = T.merge(R, on="topic", how="left")
print(f"\n분류표 주제 {len(T)}개 중 평가자 입력 {M.rater.isin(['core','enabling','legacy','other','non_gx']).sum()}개")
M = M[M.rater.isin(["core", "enabling", "legacy", "other", "non_gx"])].copy()
M["mine"] = M["gx_class"].replace({"unreviewed": "other"})
labels = ["core", "enabling", "legacy", "other", "non_gx"]

def kap(d, tag):
    k5 = cohen_kappa_score(d.mine, d.rater, labels=labels)
    k2 = cohen_kappa_score((d.mine == "core").astype(int), (d.rater == "core").astype(int))
    kw = cohen_kappa_score((d.mine == "core").astype(int), (d.rater == "core").astype(int), sample_weight=d.n_total)
    print(f"{tag:<22} 주제 {len(d):>3} | 5범주 κ {k5:.3f} | core 이진 κ {k2:.3f} | 논문수 가중 core κ {kw:.3f} | 일치율 {(d.mine == d.rater).mean():.3f}")
kap(M, "전체")
kap(M[M.src.isin(["claude_draft", "claude_patch"])], "제가 직접 본 주제")
kap(M[M.src == "rule_provisional"], "규칙 임시분류 주제")
kap(M[M.n_total >= M.n_total.quantile(0.5)], "논문 수 상위 절반")

print("\n혼동행렬 (행=제 분류, 열=평가자)")
print(pd.DataFrame(confusion_matrix(M.mine, M.rater, labels=labels), index=labels, columns=labels))

# core 여부가 갈린 주제 (결과에 직접 영향): 논문 수 순
D = M[(M.mine == "core") != (M.rater == "core")].sort_values("n_total", ascending=False)
print(f"\ncore 여부 불일치 {len(D)}개, 논문 수 합 {D.n_total.sum()} (전체 {M.n_total.sum()})")
print(D.head(25)[["topic", "mine", "rater", "n_total", "src"] + (["memo"] if "memo" in D.columns else [])].to_string(index=False))
M[M.mine != M.rater].sort_values("n_total", ascending=False).to_csv(f"{OUT}/gx_disagreements.csv", index=False, encoding="utf-8-sig")
M.to_csv(f"{OUT}/gx_rater_merged.csv", index=False, encoding="utf-8-sig")

In [ ]:
Ud = U[U.year <= YLAST].copy(); TOY, NIS = "トヨタ自動車", "日産自動車"
RATER = M.set_index("topic")["rater"].to_dict()
Ud["cls_r"] = Ud["topic"].map(RATER)
print(f"평가자 분류가 붙은 논문 비중: {Ud.cls_r.notna().mean():.3f} (빈 주제는 제 분류로 대체)")
Ud["cls_r"] = Ud["cls_r"].fillna(Ud["cls"].replace({"unreviewed": "other"}))

def block_stats(col):
    t = Ud[Ud[col] == "core"].groupby(["corp_norm", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0).div(pd.Series(NY), axis=1)
    def h(excl=()):
        o = []
        for b in BL:
            c = t[b][t[b] > 0]; c = c[~c.index.isin(excl)]; s = c / c.sum(); o.append((s ** 2).sum())
        return o
    tot = t.sum(); sh = t.loc[TOY] / tot
    return pd.DataFrame({"core편수년": tot, "도요타점유": sh, "HHI포함": h(), "HHI도요타제외": h([TOY]), "HHI도요타닛산제외": h([TOY, NIS]),
                         "도요타core": t.loc[TOY], "도요타제외core": tot - t.loc[TOY]}, index=[LAB[b] for b in BL])
print("\n[A1 | 제 분류]"); print(block_stats("cls").round(3).to_string())
print("\n[A1 | 평가자 분류]"); print(block_stats("cls_r").round(3).to_string())

In [ ]:
#

In [ ]:
# (1) 합의 반영: 명백한 오분류 3개는 평가자 판단 채택, methane reforming은 일단 평가자(other)로 두고 민감도로 병기
FINAL = M.set_index("topic")["rater"].to_dict()
pd.DataFrame({"topic": list(FINAL), "gx_class_final": list(FINAL.values())}).to_csv(f"{OUT}/gx_review_final.csv", index=False, encoding="utf-8-sig")

Ud = U[U.year <= YLAST].copy(); TOY, NIS = "トヨタ自動車", "日産自動車"
Ud["cls_r"] = Ud["topic"].map(FINAL).fillna(Ud["cls"].replace({"unreviewed": "other"}))
NYs = pd.Series(NY)

def block_stats(col, core_set=("core",)):
    t = Ud[Ud[col].isin(core_set)].groupby(["corp_norm", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0)
    t = t.div(NYs.reindex(BL), axis=1)
    def hh(excl=()):
        o = []
        for b in BL:
            c = t[b][t[b] > 0]; c = c[~c.index.isin(excl)]; s = c / c.sum(); o.append(float((s ** 2).sum()))
        return o
    tot = t.sum().values; toy = t.loc[TOY].values
    return pd.DataFrame({"core편수년": tot, "도요타core": toy, "도요타제외core": tot - toy, "도요타점유": toy / tot,
                         "HHI포함": hh(), "HHI도요타제외": hh([TOY]), "HHI도요타닛산제외": hh([TOY, NIS])}, index=[LAB[b] for b in BL])

print("[A1 | 제 분류]");   print(block_stats("cls").round(3).to_string())
print("\n[A1 | 평가자 분류]"); print(block_stats("cls_r").round(3).to_string())

# (2) 쟁점 민감도: methane reforming을 core로 되돌린 평가자 분류
Ud["cls_r2"] = Ud["cls_r"]; Ud.loc[Ud.topic == "Catalysts for Methane Reforming", "cls_r2"] = "core"
print("\n[A1 | 평가자 분류 + methane reforming core]"); print(block_stats("cls_r2").round(3).to_string())

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import os, re, glob, json, warnings, traceback
import numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf
from scipy import stats
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 200)

# ============================ 설정 ============================
NB = 1000; SEED = 0
RUN_LIT = False                      # True면 OpenAlex 문헌 후보 검색(호출 발생, 키는 환경변수)
EXTRA_RATERS = []                    # 추가 평가자 시트 경로 리스트, 열: topic, gx_class_final (예: [f"{OUT}/gx_review_rater2.csv"])
TOY, NIS = "トヨタ自動車", "日産自動車"
CLS5 = ["core", "enabling", "legacy", "non_gx", "other"]

need = ["U", "PF", "LK", "BL", "b0", "b1", "LAB", "NY", "YLAST", "OUT", "YMIN", "BLK", "cl_ols"]
miss = [v for v in need if v not in globals()]
assert not miss, f"이전 셀 변수가 없습니다: {miss}"
if "safe" not in globals():
    def safe(f):
        try: f()
        except Exception as e:
            print(f"\n[{f.__name__} 실패] {type(e).__name__}: {str(e)[:250]}"); traceback.print_exc(limit=2)
TOP4 = globals().get("TOP4", [TOY, "三菱電機", NIS, "ホンダ"])
FIN = f"{OUT}/final"; os.makedirs(FIN, exist_ok=True)
def save(df, name): df.to_csv(f"{FIN}/{name}.csv", index=False, encoding="utf-8-sig")

REG = []   # 다중검정 레지스트리
def reg(name, fam, est, p, src="이번 실행"):
    if p is not None and not (isinstance(p, float) and np.isnan(p)):
        REG.append(dict(분석=name, 가설군=fam, 추정치=est, p=max(float(p), 1e-300), 출처=src))

# ============================ 0. 최종 분류 적용 ============================
fp = f"{OUT}/gx_review_final.csv"
F_ = pd.read_csv(fp)[["topic", "gx_class_final"]].dropna()
F_["gx_class_final"] = F_["gx_class_final"].astype(str).str.strip().str.lower()
FINAL = dict(zip(F_.topic, F_.gx_class_final))
for df in (U, PF):
    if "cls_mine" not in df.columns: df["cls_mine"] = df["cls"] if "cls" in df.columns else "other"
    df["cls_f"] = df["topic"].map(FINAL).fillna(df["cls_mine"].replace({"unreviewed": "other"}))
Ud = U[U.year <= YLAST].copy()
print(f"[0] 최종 분류 적용: 주제 {len(FINAL)}개 | 논문 중 분류 매핑 비중 {U['topic'].isin(FINAL).mean():.3f} | 분류 분포 {U.cls_f.value_counts().to_dict()}")

# ============================ A. HHI 구간·대안 지표·한 기업씩 제외 ============================
def A():
    CM = Ud[Ud.cls_f == "core"].groupby(["corp_norm", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0)
    firms = list(CM.index); Fn = len(firms); Cmat = CM.values.astype(float); nb = len(BL)
    ti, ni = firms.index(TOY), firms.index(NIS)
    SC = {"전체": [], "도요타 제외": [ti], "도요타·닛산 제외": [ti, ni]}
    def hh(col, keep):
        c = col * keep; c = c[c > 0]
        if c.sum() == 0: return np.nan, np.nan
        s = c / c.sum(); return float((s ** 2).sum()), float(s.max())
    def full(col, keep):
        c = col * keep; c = c[c > 0]; s = np.sort(c / c.sum()); n = len(s); idx = np.arange(1, n + 1)
        gini = 2 * (idx * s).sum() / (n * s.sum()) - (n + 1) / n
        return dict(HHI=(s ** 2).sum(), 상위1=s[-1], 상위4=s[-4:].sum(), 유효기업수=1 / (s ** 2).sum(), Gini=gini, 기업수=n)
    rows = []
    for nm, ex in SC.items():
        keep = np.ones(Fn); keep[ex] = 0
        for k, b in enumerate(BL): rows.append(dict(시나리오=nm, 블록=LAB[b], **full(Cmat[:, k], keep)))
    P = pd.DataFrame(rows); print("[A-1] 대안 집중 지표 (core, 최종 분류, 논문 단위)"); print(P.round(3).to_string(index=False)); save(P, "A1_concentration_metrics")
    def boot(kind):
        rng = np.random.default_rng(SEED); r1 = {k: [] for k in SC}; r2 = {k: [] for k in SC}
        tot = Cmat.sum(0)
        for _ in range(NB):
            if kind == "firm": pick = rng.integers(0, Fn, Fn); cnt = Cmat[pick]
            else: pick = np.arange(Fn); cnt = np.column_stack([rng.multinomial(int(tot[k]), Cmat[:, k] / tot[k]) for k in range(nb)]).astype(float)
            for nm, ex in SC.items():
                keep = (~np.isin(pick, ex)).astype(float); h0, t0 = hh(cnt[:, 0], keep); h1, t1 = hh(cnt[:, -1], keep)
                if not np.isnan(h0 + h1): r1[nm].append(h1 - h0); r2[nm].append(t1 - t0)
        return r1, r2
    out = []
    for kind, lab in [("firm", "기업 재표집(구성 변동)"), ("paper", "논문 재표집(블록 내 논문 변동)")]:
        r1, r2 = boot(kind)
        for nm, ex in SC.items():
            keep = np.ones(Fn); keep[ex] = 0; h0, t0 = hh(Cmat[:, 0], keep); h1, t1 = hh(Cmat[:, -1], keep)
            for met, pt, rr in [("HHI 변화", h1 - h0, r1[nm]), ("상위1 점유 변화", t1 - t0, r2[nm])]:
                rr = np.array(rr); p = min(1.0, 2 * min((1 + (rr <= 0).sum()) / (1 + len(rr)), (1 + (rr >= 0).sum()) / (1 + len(rr))))
                out.append(dict(재표집=lab, 시나리오=nm, 지표=met, 점추정=pt, lo=np.percentile(rr, 2.5), hi=np.percentile(rr, 97.5), p=p, 반복=len(rr)))
                if met == "HHI 변화": reg(f"HHI 변화 {nm} ({lab})", "F1 집중", pt, p)
    O = pd.DataFrame(out); print("\n[A-2] 집중 변화(마지막-첫 블록)의 95% 구간. 기업 재표집 구간이 0을 포함해도 논문 재표집이 0을 제외하면 '기업 구성 불확실성' 때문"); print(O.round(3).to_string(index=False)); save(O, "A2_concentration_bootstrap")
    tot_c = Cmat.sum(1); loo = []
    for j in np.argsort(-tot_c)[:8]:
        keep = np.ones(Fn); keep[j] = 0; h0, t0 = hh(Cmat[:, 0], keep); h1, t1 = hh(Cmat[:, -1], keep)
        loo.append(dict(제외기업=firms[j], HHI_첫=h0, HHI_끝=h1, HHI변화=h1 - h0, 상위1점유_첫=t0, 상위1점유_끝=t1))
    L = pd.DataFrame(loo); print("\n[A-3] core 편수 상위 8개 기업을 한 곳씩 제외했을 때"); print(L.round(3).to_string(index=False)); save(L, "A3_leave_one_firm_out")
safe(A)

# ============================ B. 진입 플래그 진단 + 일관된 정의로 B3 재산출 ============================
Ua = None
def B():
    global Ua
    A_ = PF.groupby(["corp_norm", "topic_id"]).agg(fb=("blk", "min")).reset_index()
    Q = PF.merge(A_, on=["corp_norm", "topic_id"]); Q1 = Q[(Q.blk == Q.fb) & (Q.blk > b0)].copy()
    ev = Q1.groupby(["corp_norm", "topic_id", "blk"]).agg(n=("work_id", "nunique"), e_max=("entry", "max"), e_min=("entry", "min"),
                                                        domain=("domain", "first"), subfield=("subfield", "first")).reset_index()
    print(f"[B-1] 첫 등장 논문 행 {len(Q1)} (entry=0 {int((Q1.entry == 0).sum())}) | 기업x주제 첫 등장 사건 {len(ev)}")
    print(f"  사건 중 플래그가 한 건도 없음 {int((ev.e_max == 0).sum())} | 일부 논문에만 있음(혼합) {int(((ev.e_max == 1) & (ev.e_min == 0)).sum())} | 전부 있음 {int((ev.e_min == 1).sum())}")
    print("  사건 논문 수별 플래그 비율:", ev.assign(g=pd.cut(ev.n, [0, 1, 2, 5, 1000])).groupby("g").e_max.mean().round(3).to_dict())
    for col in ["domain", "subfield"]:
        fbk = U.groupby(["corp_norm", col])["blk"].min()
        ev["prior_" + col] = [(fbk.get((c, v), 99) < b) if pd.notna(v) else np.nan for c, v, b in zip(ev.corp_norm, ev[col], ev.blk)]
        print(f"  같은 기업이 이전 블록에 같은 {col}를 가졌던 비율: 플래그 있음 {ev[ev.e_max == 1]['prior_' + col].mean():.3f} vs 없음 {ev[ev.e_max == 0]['prior_' + col].mean():.3f}")
    print("  플래그 없는 사건의 블록 분포:", ev[ev.e_max == 0].groupby("blk").size().to_dict(), "| 도메인:", ev[ev.e_max == 0].groupby("domain").size().to_dict())
    save(ev, "B1_entry_events_diagnostic")
    Ua = U.merge(A_, on=["corp_norm", "topic_id"], how="left").sort_values(["corp_norm", "topic_id", "blk", "year", "w"]).reset_index(drop=True)
    Ua["entry_v1"] = Ua["entry"].astype(int); Ua["entry_v2"] = ((Ua.blk == Ua.fb) & (Ua.blk > b0)).astype(int)
    Ua["rk"] = Ua.groupby(["corp_norm", "topic_id", "blk"]).cumcount(); Ua["entry_v3"] = ((Ua.entry_v2 == 1) & (Ua.rk == 0)).astype(int)
    Ua["ln_n"] = np.log(Ua.groupby(["corp_norm", "blk"])["w"].transform("size"))
    print("\n[B-2] 정의별 진입 논문 수 (blk>첫 블록):", {v: int(Ua[Ua.blk > b0][v].sum()) for v in ["entry_v1", "entry_v2", "entry_v3"]},
          "| v1: 원래 플래그, v2: 첫 등장 블록의 모든 논문, v3: 사건당 첫 논문 1건")
    def eff2(d, y):
        d = d[d.blk > b0].copy(); d["y"] = d[y].astype(float)
        if len(d) < 80 or d.y.sum() < 15 or d.nw_any.sum() < 15: return dict(논문=len(d), 진입=int(d.y.sum()), OR=np.nan)
        X = sm.add_constant(pd.concat([d[["nw_any", "ln_n"]], pd.get_dummies(d.blk, prefix="b", drop_first=True)], axis=1).astype(float))
        m = sm.Logit(d["y"].values, X).fit(disp=0, maxiter=200, cov_type="cluster", cov_kwds={"groups": pd.factorize(d.corp_norm)[0]})
        b, s = m.params["nw_any"], m.bse["nw_any"]; mf = cl_ols("y ~ nw_any + ln_n + C(blk) + C(corp_norm)", d)
        return dict(논문=len(d), 진입=int(d.y.sum()), OR=np.exp(b), lo=np.exp(b - 1.96 * s), hi=np.exp(b + 1.96 * s), p=m.pvalues["nw_any"], FE_pp=100 * mf.params["nw_any"], FE_p=mf.pvalues["nw_any"])
    rows = []
    for v in ["entry_v1", "entry_v2", "entry_v3"]:
        base = Ua if v != "entry_v3" else Ua[(Ua.entry_v2 == 0) | (Ua.entry_v3 == 1)]
        for lab, d in [("전체", base), ("상위4 제외", base[~base.corp_norm.isin(TOP4)])]:
            r = eff2(d, v); rows.append(dict(정의=v, 표본=lab, **r))
            if lab == "전체": reg(f"신규 파트너→진입 OR ({v})", "F4 진입", r.get("OR", np.nan), r.get("p", np.nan))
    R = pd.DataFrame(rows); print("\n[B-3] 신규 파트너 -> 진입, 정의별 (논문 단위 로짓 + 기업 FE LPM)"); print(R.round(3).to_string(index=False)); save(R, "B3_entry_by_definition")
    def wald_het(d, y, grp, ref):
        d = d.copy(); lv = [g for g in d[grp].unique() if g != ref]
        X = pd.concat([d[["nw_any", "ln_n"]], pd.get_dummies(d.blk, prefix="b", drop_first=True), pd.get_dummies(d[grp], prefix="g").drop(columns=f"g_{ref}")], axis=1).astype(float)
        for g in lv: X[f"x_{g}"] = X["nw_any"] * X[f"g_{g}"]
        X = sm.add_constant(X); cols = list(X.columns)
        m = sm.Logit(d[y].astype(float).values, X.values).fit(disp=0, maxiter=300, cov_type="cluster", cov_kwds={"groups": pd.factorize(d.corp_norm)[0]})
        Rm = np.zeros((len(lv), len(cols)))
        for j, g in enumerate(lv): Rm[j, cols.index(f"x_{g}")] = 1
        return float(m.wald_test(Rm, scalar=True).pvalue)
    rows = []
    for v in ["entry_v2", "entry_v3"]:
        base = Ua[Ua.blk > b0] if v == "entry_v2" else Ua[(Ua.blk > b0) & ((Ua.entry_v2 == 0) | (Ua.entry_v3 == 1))]
        for k in ["core", "enabling", "other", "non_gx"]:
            try: rows.append(dict(정의=v, 분류=k, **eff2(base[base.cls_f == k], v)))
            except Exception: pass
        d4 = base[base.cls_f.isin(["core", "enabling", "other", "non_gx"])]
        try:
            pw = wald_het(d4, v, "cls_f", "non_gx"); print(f"[B-4] {v}: 분류 간 이질성 공동 Wald p = {pw:.3f}"); reg(f"진입 분류 간 이질성 ({v})", "F4 진입", np.nan, pw)
        except Exception as e: print("  Wald 실패", type(e).__name__)
    print("\n[B-4] 분류별 OR (최종 분류)"); print(pd.DataFrame(rows).round(3).to_string(index=False)); save(pd.DataFrame(rows), "B4_entry_by_class")
safe(B)

# ============================ C. 최종 분류로 core 의존 결과 재산출 ============================
if "classify" not in globals():
    if "PART" not in globals():
        LK["w"] = LK["work_id"].map(lambda s: str(s).split("/")[-1]); LK["iid"] = LK["inst_id"].astype(str).map(lambda s: s.split("/")[-1])
        PART = LK.groupby(["w", "corp_norm"])["iid"].agg(set).to_dict()
    def classify(w, corp):
        a = AUTH.get(w)
        if not a: return None
        pset = PART.get((w, corp), set()); out = []
        for aid, pos, corr, types, iids, ctry in a:
            ext = any(i in pset for i in iids); out.append((aid, pos, corr, ext, (not ext) and ("company" in types)))
        return out
def C_():
    # (a) 신규 기업 저자 중 단독 참여: core 대 비core
    rec = []
    for r in Ud[["w", "corp_norm", "blk", "solo", "cls_f"]].itertuples(index=False):
        cl = classify(r.w, r.corp_norm)
        if cl: rec += [(r.corp_norm, c[0], r.blk, r.solo, int(r.cls_f == "core")) for c in cl if c[4] and c[0]]
    Rr = pd.DataFrame(rec, columns=["corp_norm", "aid", "blk", "solo", "core"]).drop_duplicates()
    fb = Rr.groupby(["corp_norm", "aid"])["blk"].min().rename("fb").reset_index(); Rr = Rr.merge(fb, on=["corp_norm", "aid"])
    new = Rr[(Rr.blk == Rr.fb) & (Rr.blk > b0)].groupby(["corp_norm", "aid", "blk", "core"])["solo"].max().reset_index()
    T = new.groupby(["core", "blk"])["solo"].agg(["size", "mean"]).rename(index={0: "core 외", 1: "core"}, level=0); T.index = [(a, LAB[b]) for a, b in T.index]
    print("[C-1] 신규 기업 저자 중 단독 참여 비중 (저자 단위, 최종 분류)"); print(T.round(3).to_string())
    n2 = new.copy(); n2["late"] = (n2.blk == b1).astype(int)
    for lab, d in [("전체", n2), ("도요타 제외", n2[n2.corp_norm != TOY])]:
        m = cl_ols("solo ~ late*core + C(corp_norm)", d)
        print(f"  [{lab}] 마지막 블록 vs 2011-15 | 비core {100*m.params['late']:+.1f}pp (p={m.pvalues['late']:.3f}) | core 추가차 {100*m.params['late:core']:+.1f}pp (se {100*m.bse['late:core']:.1f}, p={m.pvalues['late:core']:.3f})")
        if lab == "전체": reg("신규 단독 저자 비중 core 추가차(late×core)", "F2 유입", m.params["late:core"], m.pvalues["late:core"]); reg("신규 단독 저자 비중 하락(비core)", "F2 유입", m.params["late"], m.pvalues["late"])
    save(T.reset_index(), "C1_inflow_core_vs_noncore")
    # (b) 단독 확률 post x core (형식 검정)
    d = Ud[Ud.blk.isin([b0, b1]) & Ud.cls_f.isin(CLS5)].copy(); d["post"] = (d.blk == b1).astype(int); d["core"] = (d.cls_f == "core").astype(int)
    for lab, dd in [("core vs 나머지 4분류", d), ("core vs non_gx만", d[d.cls_f.isin(["core", "non_gx"])])]:
        m = cl_ols("solo ~ post*core + C(corp_norm) + C(domain)", dd); nm_ = list(m.params.index); r = np.zeros(len(nm_)); r[nm_.index("post")] = 1; r[nm_.index("post:core")] = 1; tt = m.t_test(r)
        print(f"[C-2] {lab}: 비core {100*m.params['post']:+.1f}pp | core 추가차 {100*m.params['post:core']:+.1f}pp (se {100*m.bse['post:core']:.1f}, p={m.pvalues['post:core']:.3f}) | core 순변화 {100*float(tt.effect[0]):+.1f}pp (p={float(np.ravel(tt.pvalue)[0]):.3f})")
        reg(f"단독 확률 core 추가차 ({lab})", "F5 core 특이성", m.params["post:core"], m.pvalues["post:core"])
safe(C_)

# ============================ D. 분류 신뢰도: κ 구간, 추가 평가자 ============================
def D():
    Mg = pd.read_csv(f"{OUT}/gx_rater_merged.csv"); Mg = Mg[Mg.rater.isin(CLS5)].copy(); Mg["mine"] = Mg["mine"].replace({"unreviewed": "other"})
    rng = np.random.default_rng(SEED); a = Mg.mine.values; b = Mg.rater.values; w = Mg.n_total.values.astype(float); n = len(Mg); res = {"5범주": [], "core 이진": [], "논문수 가중 core": []}
    for _ in range(2000):
        i = rng.integers(0, n, n)
        try:
            res["5범주"].append(cohen_kappa_score(a[i], b[i], labels=CLS5))
            ca, cb = (a[i] == "core").astype(int), (b[i] == "core").astype(int)
            res["core 이진"].append(cohen_kappa_score(ca, cb)); res["논문수 가중 core"].append(cohen_kappa_score(ca, cb, sample_weight=w[i]))
        except Exception: pass
    pt = {"5범주": cohen_kappa_score(a, b, labels=CLS5), "core 이진": cohen_kappa_score(a == "core", b == "core"), "논문수 가중 core": cohen_kappa_score(a == "core", b == "core", sample_weight=w)}
    T = pd.DataFrame([dict(지표=k, κ=pt[k], lo=np.nanpercentile(v, 2.5), hi=np.nanpercentile(v, 97.5)) for k, v in res.items()])
    print("[D-1] κ와 주제 재표집 95% 구간 (평가자 1명 기준)"); print(T.round(3).to_string(index=False)); save(T, "D1_kappa_ci")
    if EXTRA_RATERS:
        S = Mg[["topic", "mine", "rater"]].rename(columns={"mine": "r0_claude", "rater": "r1"})
        for k, f in enumerate(EXTRA_RATERS, start=2):
            X = pd.read_csv(f)[["topic", "gx_class_final"]].rename(columns={"gx_class_final": f"r{k}"}); X[f"r{k}"] = X[f"r{k}"].astype(str).str.strip().str.lower(); S = S.merge(X, on="topic", how="left")
        rc = [c for c in S.columns if c.startswith("r")]; M2 = pd.DataFrame(index=rc, columns=rc, dtype=float)
        for x in rc:
            for y in rc:
                dd = S.dropna(subset=[x, y]); M2.loc[x, y] = cohen_kappa_score(dd[x], dd[y], labels=CLS5) if len(dd) else np.nan
        print("\n[D-2] 평가자 쌍별 κ (r0=제 초안)"); print(M2.round(3).to_string())
        hum = [c for c in rc if c != "r0_claude"]
        S["합의"] = S[hum].mode(axis=1)[0]; S["동률"] = S[hum].apply(lambda r: r.value_counts().iloc[0] == r.value_counts().iloc[1] if r.nunique() > 1 and len(r.value_counts()) > 1 else False, axis=1)
        print(f"  사람 평가자 간 불일치 {int((S[hum].nunique(axis=1) > 1).sum())}개, 동률 {int(S['동률'].sum())}개 -> 논의 대상"); save(S, "D2_multi_rater_consensus")
    else: print("[D-2] EXTRA_RATERS가 비어 있어 쌍별 κ는 건너뜀. 평가자를 추가하면 경로를 넣으세요.")
safe(D)

# ============================ E. 보안: 하드코딩된 키 스캔 (값은 마스킹) ============================
def E():
    pats = [re.compile(r"""(api[_-]?key|apikey|token|secret|password)\s*[:=]\s*\\?["']([^"'\\\s]{12,})""", re.I), re.compile(r"api_key=([A-Za-z0-9_\-]{12,})", re.I)]
    exts = (".py", ".ipynb", ".txt", ".md", ".env", ".json", ".cfg", ".yml", ".yaml", ".toml"); hits = []
    for root, dirs, files in os.walk("."):
        dirs[:] = [d for d in dirs if d not in (".git", "node_modules", "__pycache__", ".venv", "venv")]
        for fn in files:
            if not fn.lower().endswith(exts): continue
            p = os.path.join(root, fn)
            try:
                if os.path.getsize(p) > 5_000_000: continue
                for ln, line in enumerate(open(p, encoding="utf-8", errors="ignore"), 1):
                    for pt in pats:
                        m = pt.search(line)
                        if m: v = m.group(m.lastindex); hits.append((p, ln, m.group(1)[:20] if m.lastindex > 1 else "api_key", v[:3] + "***" + v[-2:]))
            except Exception: pass
    print("[E] 환경변수 OPENALEX_API_KEY 설정:", bool(os.environ.get("OPENALEX_API_KEY")))
    if hits:
        print(f"  하드코딩 의심 {len(hits)}건 (값 마스킹):"); [print("  ", h) for h in hits[:30]]
        print("  조치: 1) 해당 키 폐기·재발급 2) 코드는 os.environ만 사용 3) git을 쓴다면 이력에서도 제거(키 폐기가 우선)")
    else: print("  의심 패턴 없음(스캔 범위: 현재 폴더 하위 텍스트 파일)")
safe(E)

# ============================ F. 다중검정 레지스트리 (가설군별·전체 Holm) ============================
def F():
    se_ = lambda est, se: 2 * (1 - stats.norm.cdf(abs(est / se)))
    prior = [("신규 단독 저자 비중 하락(기업 FE, 마지막 vs 2011-15)", "F2 유입", -0.116, 0.001, "이전 출력"),
             ("저자정의: 마지막 저자 외부 주도 변화(기업 FE)", "F3 저자정의", 0.173, se_(17.3, 3.3), "이전 출력 근사(est/se)"),
             ("저자정의: 제1저자 외부 주도 변화(기업 FE)", "F3 저자정의", 0.053, se_(5.3, 3.8), "이전 출력 근사(est/se)"),
             ("저자정의: 교신저자 외부 주도 변화(기업 FE)", "F3 저자정의", 0.020, se_(2.0, 5.4), "이전 출력 근사(est/se)")]
    brk = {"ALL": 0.001, "BATTERY": 0.632, "FUELCELL": 0.091, "MATERIAL": 0.028, "POWER": 0.074, "SOLAR": 0.963, "THERMAL": 0.185, "VEHICLE": 0.006}
    prior += [(f"하락 전환점 {k} (부트스트랩 999)", "F6 전환점", np.nan, v, "이전 출력") for k, v in brk.items()]
    R = pd.DataFrame(REG + [dict(분석=a, 가설군=b, 추정치=c, p=d, 출처=e) for a, b, c, d, e in prior])
    R["p_holm_가설군내"] = np.nan
    for g, ix in R.groupby("가설군").groups.items(): R.loc[ix, "p_holm_가설군내"] = multipletests(R.loc[ix, "p"].values, method="holm")[1]
    R["p_holm_전체"] = multipletests(R.p.values, method="holm")[1]
    R["구분"] = "사후 선정한 주요 분석"; R = R.sort_values(["가설군", "p"])
    print("[F] 레지스트리 (논문 보고용: 가설군 정의는 본인이 확정. 사전 등록이 없으므로 '확증'이 아니라 '사후 선정한 주요 분석'으로 표기)"); print(R.round(4).to_string(index=False)); save(R, "F_multiple_testing_registry")
    print(f"\n  누적 모형 수는 별도 집계 필요(이전 약 205개). 이 표는 보고 대상 {len(R)}개만 담음")
safe(F)

# ============================ G. 기간·단위 정합성 검증과 정의표 ============================
def G():
    rows = []
    for b in BL:
        d = Ud[Ud.blk == b]; rows.append(dict(블록=LAB[b], 연도범위=f"{d.year.min()}-{d.year.max()}", 연수_NY=NY[b], 논문_기업별=len(d), 연평균=len(d) / NY[b]))
    T = pd.DataFrame(rows); print("[G-1] 블록별 연도 커버리지와 연 환산 분모"); print(T.round(1).to_string(index=False))
    exp = YLAST - (YMIN + b1 * BLK) + 1; print(f"  마지막 블록 기대 연수 {exp} vs NY[b1]={NY[b1]} ->", "일치" if exp == NY[b1] else "불일치(수정 필요)")
    assert exp == NY[b1], "NY와 YLAST 불일치"
    D = pd.DataFrame([
        ("특허 표본", "JPO FI 분류 GX 특허, 최초 공개 2006~2015, 기업 133곳 중 분석 66곳", "특허는 처치가 아닌 표본 틀"),
        ("논문 추적 기간", f"{YMIN}~{YLAST} (2025 색인 미완으로 제외), 마지막 블록은 4년 환산", "특허 기간과 구분 서술"),
        ("분석 단위", "기업x논문(고유 work_id x corp_norm), 5년 블록", "기업별 중복 포함, 한 논문이 여러 기업이면 각각 계산"),
        ("core 분류", "core/enabling/legacy/other/non_gx, 최종=독립 평가자 분류(gx_review_final.csv)", "κ와 구간은 D 모듈"),
        ("HHI", "블록별 core 논문의 기업 점유율 제곱합(논문 단위)", "제외 시나리오는 재정규화"),
        ("신규 기업 저자", "해당 기업 코퍼스에 처음 등장한 저자ID, 저자 단위", "누적 풀 효과·ID 오분리 한계"),
        ("진입", "v1 원래 플래그, v2 첫 등장 블록의 모든 논문, v3 사건당 첫 논문", "B 모듈에서 정의별 비교"),
        ("외부 주도", "마지막/제1/교신 저자 세 정의 병기", "'주도권'이 아닌 '저자 위치 비중'으로 서술")], columns=["항목", "정의", "비고"])
    save(T, "G1_block_coverage"); save(D, "G2_definitions"); print("\n[G-2] 정의표 저장:", f"{FIN}/G2_definitions.csv")
safe(G)

# ============================ H. (선택) 문헌 후보 검색: 읽고 판단은 본인이 ============================
def H():
    if not RUN_LIT: print("[H] RUN_LIT=False: 건너뜀"); return
    if "oa_get" not in globals(): print("[H] oa_get 함수가 없음(이전 셀 필요)"); return
    qs = ["corporate science publications decline firms", "firms scientific publications patent-based sample", "green technology patents firm scientific research",
          "concentration of corporate research publications", "Japanese firms university industry collaboration publications", "corporate authorship in scientific papers trend",
          "first-time collaboration partner new research topic firm", "knowledge production green technology firms science linkage"]
    rows = []
    for q in qs:
        js = oa_get("works", {"search": q, "filter": "publication_year:>2005,type:article", "sort": "cited_by_count:desc", "per-page": 25, "select": "id,title,publication_year,doi,cited_by_count,primary_location"})
        for r in (js or {}).get("results", []):
            src = ((r.get("primary_location") or {}).get("source") or {}).get("display_name")
            rows.append(dict(질의=q, 제목=r.get("title"), 연도=r.get("publication_year"), 인용=r.get("cited_by_count"), 저널=src, doi=r.get("doi")))
    L = pd.DataFrame(rows).drop_duplicates("doi"); save(L, "H_literature_candidates"); print(f"[H] 문헌 후보 {len(L)}편 저장. 제목만 보고 인용하지 말고 초록·본문을 읽고 `[문헌 확인]` 자리를 채우세요.")
safe(H)
print(f"\n[완료] 표는 {FIN}/ 에 저장. 붙여 주실 것: [A-1~3], [B-1~4], [C-1~2], [D-1], [E], [F] 출력")

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import os, warnings, traceback
import numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

TOY, NIS = "トヨタ自動車", "日産自動車"
NB13 = 1000; SEED = 0
need = ["U", "BL", "b0", "b1", "LAB", "NY", "YLAST", "YMIN", "BLK", "OUT"]
miss = [v for v in need if v not in globals()]; assert not miss, f"이전 셀 변수가 없습니다: {miss}"
assert "cls_f" in U.columns, "최종 분류 열 cls_f가 없습니다 (코드 ⑫의 0단계를 먼저 실행)"
if "safe" not in globals():
    def safe(f):
        try: f()
        except Exception as e:
            print(f"\n[{f.__name__} 실패] {type(e).__name__}: {str(e)[:250]}"); traceback.print_exc(limit=2)
FIN = f"{OUT}/final"; os.makedirs(FIN, exist_ok=True)
Ud = U[U.year <= YLAST].copy(); CORE = Ud[Ud.cls_f == "core"].copy()
YRS = list(range(YMIN, YLAST + 1))

def conc(counts, excl=()):
    """counts: 기업별 편수 Series -> HHI, 상위1점유, 도요타점유, 유효기업수, 기업수"""
    c = counts[counts > 0]; tot = c.sum()
    if tot == 0: return dict(HHI=np.nan, 상위1=np.nan, 도요타=np.nan, 유효기업수=np.nan, 기업수=0)
    s = c / tot; ce = c[~c.index.isin(excl)]
    he = ((ce / ce.sum()) ** 2).sum() if ce.sum() > 0 else np.nan
    return dict(HHI=(s ** 2).sum(), 상위1=s.max(), 도요타=s.get(TOY, 0.0), 유효기업수=1 / (s ** 2).sum(), 기업수=len(c), HHI_도요타제외=he)

# ================= A. 창 안 대 창 이후 =================
def A():
    rows = []
    for b in BL:
        c = CORE[CORE.blk == b].groupby("corp_norm").size() / NY[b]
        rows.append(dict(블록=LAB[b], 구간="창 안" if b <= 1 else "창 이후", 연편수=c.sum(), **conc(c, [TOY])))
    T = pd.DataFrame(rows); print("[A-1] 블록별 집중 지표 (core, 최종 분류)"); print(T.round(3).to_string(index=False))
    h = T.HHI.values; hx = T.HHI_도요타제외.values; sh = T.도요타.values; ne = T.유효기업수.values
    tot = h[-1] - h[0]
    D = pd.DataFrame([
        dict(지표="HHI", 창안_변화=h[1] - h[0], 창이후_변화=h[3] - h[2], 전체_변화=tot),
        dict(지표="HHI(도요타 제외)", 창안_변화=hx[1] - hx[0], 창이후_변화=hx[3] - hx[2], 전체_변화=hx[-1] - hx[0]),
        dict(지표="도요타 점유", 창안_변화=sh[1] - sh[0], 창이후_변화=sh[3] - sh[2], 전체_변화=sh[-1] - sh[0]),
        dict(지표="유효 기업 수", 창안_변화=ne[1] - ne[0], 창이후_변화=ne[3] - ne[2], 전체_변화=ne[-1] - ne[0])])
    D["창안_비중"] = D.창안_변화 / D.전체_변화
    print("\n[A-2] 첫→둘째 블록(창 안) 대 셋째→넷째 블록(창 이후) 변화. 주의: 블록 간 간격은 모두 5년"); print(D.round(3).to_string(index=False))
    # 부트스트랩: 논문 재표집(블록 내)과 기업 재표집 두 가지
    rng = np.random.default_rng(SEED); firms = CORE.corp_norm.unique(); piv = CORE.groupby(["corp_norm", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0)
    def stat(M, cols_ex=()):
        out = []
        for k in range(4):
            c = M[:, k].astype(float); c = c[c > 0]; s = c / c.sum(); out.append((s ** 2).sum())
        return out[1] - out[0], out[3] - out[2], (out[1] - out[0]) - (out[3] - out[2])
    M0 = piv.values; tots = M0.sum(0); res = {"논문": [], "기업": []}
    for _ in range(NB13):
        Mp = np.column_stack([rng.multinomial(int(tots[k]), M0[:, k] / tots[k]) for k in range(4)]); res["논문"].append(stat(Mp))
        Mf = M0[rng.integers(0, len(M0), len(M0))]; res["기업"].append(stat(Mf))
    pt = stat(M0); print("\n[A-3] HHI 변화의 '창 안 − 창 이후' 차이 (양수면 쏠림이 창 안에서 더 크게 형성)")
    for k, v in res.items():
        v = np.array(v); lo, hi = np.percentile(v[:, 2], [2.5, 97.5]); p = min(1, 2 * min((1 + (v[:, 2] <= 0).sum()) / (1 + len(v)), (1 + (v[:, 2] >= 0).sum()) / (1 + len(v))))
        print(f"  {k} 재표집: 창안 {pt[0]:+.3f}, 창이후 {pt[1]:+.3f}, 차이 {pt[2]:+.3f} [{lo:+.3f}, {hi:+.3f}] p≈{p:.3f} (해상도 하한 {2/(1+len(v)):.3f})")
    T.to_csv(f"{FIN}/A_window_split.csv", index=False, encoding="utf-8-sig")
safe(A)

# ================= B. 경계 이동 민감도 (연도 단위, 3년 합) =================
def B():
    cy = CORE.groupby(["year", "corp_norm"]).size().unstack(fill_value=0).reindex(YRS, fill_value=0)
    ser = []
    for y in range(YMIN + 1, YLAST):  # 3년 합이 가능한 연도
        c = cy.loc[y - 1:y + 1].sum(0); d = conc(c, [TOY]); ser.append(dict(year=y, HHI=d["HHI"], 도요타=d["도요타"], 유효=d["유효기업수"], HHI_x=d["HHI_도요타제외"]))
    S = pd.DataFrame(ser); print("[B-1] 연도별 집중(3년 이동 합; 인접 연도는 중복 사용 -> 독립 아님)"); print(S.round(3).to_string(index=False))
    out = []
    for yv in ["HHI", "도요타", "HHI_x"]:
        for K in range(2012, 2020):
            d = S.dropna(subset=[yv]).copy(); d["t"] = d.year - K; d["post"] = np.maximum(0, d.year - K)
            m = sm.OLS(d[yv].values, sm.add_constant(d[["t", "post"]])).fit(cov_type="HAC", cov_kwds={"maxlags": 2})
            out.append(dict(지표=yv, 경계=K, 이전기울기=m.params["t"], 변화=m.params["post"], p_변화=m.pvalues["post"], 적합_SSR=m.ssr))
    O = pd.DataFrame(out); best = O.loc[O.groupby("지표")["적합_SSR"].idxmin()]
    print("\n[B-2] 경계별 구간선형 적합(변화<0이면 기울기 둔화). 적합_SSR이 가장 작은 경계가 사후 최적(사후 탐색, 다중 경계 선택 보정 안 됨)")
    print(O.drop(columns="적합_SSR").round(4).to_string(index=False)); print("\n최적 경계:"); print(best[["지표", "경계", "이전기울기", "변화", "p_변화"]].round(4).to_string(index=False))
    print("\n  주의: 연도 21~22개 점에 경계를 8개 시험했으므로 p는 낙관적입니다. 경계 선택 보정이 없으니 '최적 경계 ±1' 정도의 위치 정보로만 쓰세요.")
    O.to_csv(f"{FIN}/B_breakpoints.csv", index=False, encoding="utf-8-sig")
safe(B)

# ================= C. 선발 대 후발 기업 (특허 첫 공개연도 필요) =================
def C_():
    PAT = os.environ.get("PATENT_FILE", "")
    cand = [p for p in [PAT, "patents_gx.csv", f"{OUT}/patents_gx.csv", f"{OUT}/gx_patents.csv", "firm_master_window.csv"] if p and os.path.exists(p)]
    print("[C] 후보 파일:", cand)
    if not cand: print("  특허 파일을 찾지 못했습니다. 환경변수 PATENT_FILE에 경로를 넣거나 열 이름(기업, 공개일)을 알려 주세요."); return
    P = pd.read_csv(cand[0]); print("  사용 파일:", cand[0], "| 열:", P.columns.tolist())
    ycol = next((c for c in P.columns if "first" in c.lower() and ("year" in c.lower() or "pub" in c.lower())), None) or next((c for c in P.columns if "pub" in c.lower() and ("date" in c.lower() or "year" in c.lower())), None)
    fcol = "corp_norm" if "corp_norm" in P.columns else None
    if ycol is None or fcol is None:
        print("  첫 공개연도 열이나 corp_norm 열을 자동으로 찾지 못했습니다. 위 열 목록을 알려 주세요."); return
    P["fy"] = pd.to_datetime(P[ycol], errors="coerce").dt.year if not pd.api.types.is_numeric_dtype(P[ycol]) else P[ycol]
    F = P.groupby(fcol)["fy"].min().rename("first_pat").reset_index().rename(columns={fcol: "corp_norm"})
    F["grp"] = np.where(F.first_pat <= 2010, "선발(첫 특허 ≤2010)", "후발(첫 특허 2011~2015)"); F = F[F.first_pat.between(2006, 2015)]
    print("  집단 크기:", F.grp.value_counts().to_dict(), "(주의: 특허 파일이 집계표이면 첫 연도가 아닌 창 전체 연도일 수 있음)")
    c = CORE.merge(F, on="corp_norm", how="inner"); T = c.groupby(["grp", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0)
    T = T.div(pd.Series(NY).reindex(BL), axis=1); T.columns = [LAB[b] for b in BL]; print("\n[C-1] 집단별 core 연 편수 (도요타 포함)"); print(T.round(1).to_string())
    c2 = c[c.corp_norm != TOY]; T2 = c2.groupby(["grp", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0).div(pd.Series(NY).reindex(BL), axis=1); T2.columns = [LAB[b] for b in BL]
    print("\n[C-2] 도요타 제외"); print(T2.round(1).to_string()); print("\n  해석: 후발 집단이 선발보다 늦게 늘면 '특허 선행'의 약한 시사일 뿐 인과가 아님. 두 집단의 기업 구성이 다르므로 규모 차이는 해석 금지.")
safe(C_)
print(f"\n[완료] 표는 {FIN}/ 에 저장")

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import os, re, time, warnings, traceback
import numpy as np, pandas as pd, statsmodels.api as sm, requests
from scipy import stats
warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 200)

TOY, NIS = "トヨタ自動車", "日産自動車"
need = ["U", "BL", "b0", "b1", "LAB", "NY", "YLAST", "YMIN", "BLK", "OUT"]
miss = [v for v in need if v not in globals()]; assert not miss, f"이전 셀 변수가 없습니다: {miss}"
assert "cls_f" in U.columns, "cls_f가 없습니다(코드 ⑫의 0단계 먼저)"

# ---------- 공통 함수: 모두 함수 밖에서 한 번만 정의 ----------
def safe(f):
    try: f()
    except Exception as e:
        print(f"\n[{f.__name__} 실패] {type(e).__name__}: {str(e)[:250]}"); traceback.print_exc(limit=2)

def oa_get(path, params, retries=4):
    p = dict(params)
    k = os.environ.get("OPENALEX_API_KEY"); m = os.environ.get("OPENALEX_MAILTO")
    if k: p["api_key"] = k
    if m: p["mailto"] = m
    for i in range(retries):
        try:
            r = requests.get("https://api.openalex.org/" + path, params=p, timeout=60)
            if r.status_code == 200: return r.json()
            if r.status_code in (429, 500, 502, 503): time.sleep(2 ** i); continue
            print(f"  [oa_get] HTTP {r.status_code}: {r.text[:300]}")
            return None
        except Exception as e:
            print(f"  [oa_get] 예외 {type(e).__name__}: {str(e)[:150]}"); time.sleep(2 ** i)
    return None

FIN = f"{OUT}/final"; os.makedirs(FIN, exist_ok=True)
def save(df, name): df.to_csv(f"{FIN}/{name}.csv", index=False, encoding="utf-8-sig")
NYs = pd.Series(NY).reindex(BL)

Ud = U[U.year <= YLAST].copy(); CORE = Ud[Ud.cls_f == "core"].copy()
FM = pd.read_csv("firm_master_window.csv").drop_duplicates("corp_norm").set_index("corp_norm")
print("[준비] firm_master_window 열:", FM.columns.tolist())
for c in ["first_pub", "n_2010_15", "n_win", "n_pubyears", "early_share"]:
    if c in FM.columns: print(f"  {c}: {FM[c].describe().round(2).to_dict()}")
if "tier" in FM.columns: print("  tier 분포:", FM["tier"].value_counts().to_dict())

def to_year(s):
    if pd.api.types.is_numeric_dtype(s): return s
    return pd.to_datetime(s, errors="coerce").dt.year

def conc(c):
    c = c[c > 0]; t = c.sum()
    if t == 0: return dict(편수년=0, 도요타점유=np.nan, HHI=np.nan, 유효기업수=np.nan, 기업수=0)
    s = c / t
    return dict(편수년=t, 도요타점유=float(s.get(TOY, 0.0)), HHI=float((s ** 2).sum()), 유효기업수=float(1 / (s ** 2).sum()), 기업수=len(c))

# ======================= 분석 1: 일본 기업 core 논문 대비 표본 비중 =======================
def A1():
    tid = lambda t: str(t) if re.match(r"^T\d+$", str(t)) else ("T" + str(t) if re.match(r"^\d+$", str(t)) else None)
    topics = sorted({tid(t) for t in U.loc[U.cls_f == "core", "topic_id"].dropna().unique()} - {None})
    print(f"[1] core 주제 {len(topics)}개 (최종 분류). 분모: 주 주제가 core 주제이고, 논문 안에 일본 소재 기관과 기업형 기관이 (같은 저자가 아니어도) 존재하는 논문")
    if not topics: print("  주제 ID를 만들지 못했습니다(topic_id 형식 확인)"); return
    # 먼저 한 번만 시험 호출해 필터 문법을 확인
    test = oa_get("works", {"filter": f"primary_topic.id:{topics[0]},publication_year:{YMIN}-{YMIN + 4}", "per-page": 1})
    if test is None: print("  시험 호출 실패: 위 HTTP 메시지를 확인하세요(필터 문법/키/한도)"); return
    print(f"  시험 호출 OK (주제 {topics[0]}, 첫 블록 논문 {test['meta']['count']}편)")
    chunks = [topics[i:i + 10] for i in range(0, len(topics), 10)]; rows = []; inst_tot = {}
    for b in BL:
        y0 = YMIN + b * BLK; y1 = min(y0 + BLK - 1, YLAST); tot = 0; trunc = False
        for ch in chunks:
            flt = f"primary_topic.id:{'|'.join(ch)},authorships.institutions.country_code:JP,authorships.institutions.type:company,publication_year:{y0}-{y1}"
            js = oa_get("works", {"filter": flt, "per-page": 1})
            if js is None: print(f"  호출 실패 블록 {LAB[b]}"); return
            tot += js["meta"]["count"]
            g = oa_get("works", {"filter": flt, "group_by": "authorships.institutions.id"})
            if g:
                if len(g.get("group_by", [])) >= 200: trunc = True
                for r in g.get("group_by", []): inst_tot[(b, r["key"])] = inst_tot.get((b, r["key"]), 0) + r["count"]
        ours = CORE[CORE.blk == b]["w"].nunique()
        rows.append(dict(블록=LAB[b], 연도=f"{y0}-{y1}", OpenAlex_일본기업_core논문=tot, 우리표본_core_고유논문=ours, 비율=ours / tot if tot else np.nan, 그룹_잘림=trunc))
    R = pd.DataFrame(rows); print(R.round(3).to_string(index=False)); save(R, "1_japan_company_core_coverage")
    print("  주의: 필터의 일본 조건과 기업 조건은 같은 기관에 대한 조건이 아니라 논문 어딘가의 조건이라 분모가 넓게 잡힙니다(일본 대학 + 외국 기업 공저 포함 가능). 따라서 비율은 하한에 가까운 보수적 값입니다.")
    print("  또 분모는 주 주제 기준이고 우리 표본은 논문의 주 주제 매칭 기준이 같은지 확인이 필요합니다(topic_id가 primary_topic이면 동일).")
    ids = pd.Series({k: v for k, v in inst_tot.items() if k[0] == b1}).sort_values(ascending=False).head(60)
    if len(ids) == 0: print("  기관 그룹 결과 없음"); return
    keys = [k[1].split("/")[-1] for k in ids.index]; meta = []
    for i in range(0, len(keys), 50):
        js = oa_get("institutions", {"filter": "openalex:" + "|".join(keys[i:i + 50]), "select": "id,display_name,country_code,type", "per-page": 50})
        meta += (js or {}).get("results", [])
    M = pd.DataFrame(meta)
    if len(M): M = M[(M.country_code == "JP") & (M.type == "company")]
    if len(M):
        M["inst"] = M["id"].str.split("/").str[-1]; cnt = {k[1].split("/")[-1]: v for k, v in ids.items()}; M["마지막블록_core논문"] = M.inst.map(cnt)
        M["corp_norm(수기입력)"] = ""; M = M.sort_values("마지막블록_core논문", ascending=False)
        print("\n[1-b] 마지막 블록 일본 기업 기관 상위(표본 66곳과 대조해 corp_norm을 채우면 '빠진 기업' 목록이 됩니다)")
        print(M[["display_name", "inst", "마지막블록_core논문"]].head(40).to_string(index=False)); save(M, "1_top_japan_company_institutions")
safe(A1)

# ======================= 분석 2: 표본 창 민감도 (창을 좁히는 방향) =======================
def A2():
    firms = set(CORE.corp_norm.unique()); F = FM.loc[FM.index.isin(firms)].copy()
    if "first_pub" in F.columns: F["fp"] = to_year(F["first_pub"])
    V = {"기본(분석 전체)": firms}
    if "fp" in F.columns: V["선발: 첫 특허 ≤2010"] = set(F.index[F.fp <= 2010])
    if "n_2010_15" in F.columns: V["후반 활동 큰 기업: n_2010_15 ≥ 중앙값"] = set(F.index[F.n_2010_15 >= F.n_2010_15.median()])
    if "n_win" in F.columns:
        V["특허 상위 30곳"] = set(F.n_win.sort_values(ascending=False).index[:30]); V["n_win ≥ 30"] = set(F.index[F.n_win >= 30])
    rows = []
    for nm, S in V.items():
        for b in BL:
            c = CORE[(CORE.blk == b) & CORE.corp_norm.isin(S)].groupby("corp_norm").size() / NY[b]
            rows.append(dict(변형=nm, 기업수_표본=len(S), 블록=LAB[b], **conc(c)))
    T = pd.DataFrame(rows); print("[2-1] 변형별 core 집중 지표"); print(T.round(3).to_string(index=False))
    L = T[T.블록 == LAB[BL[0]]].set_index("변형"); E = T[T.블록 == LAB[b1]].set_index("변형")
    D = pd.DataFrame({"도요타점유_첫": L.도요타점유, "도요타점유_끝": E.도요타점유, "HHI_첫": L.HHI, "HHI_끝": E.HHI, "유효기업수_첫": L.유효기업수, "유효기업수_끝": E.유효기업수,
                      "core편수_정점대비_끝": E.편수년 / T.groupby("변형").편수년.max()})
    D["점유상승"] = D.도요타점유_끝 > D.도요타점유_첫; D["유효기업수감소"] = D.유효기업수_끝 < D.유효기업수_첫
    print("\n[2-2] 요약 판정 (두 열이 모두 True이면 패턴 유지)"); print(D.round(3).to_string()); save(T, "2_window_sensitivity")
    print("  주의: 도요타가 모든 변형에 들어 있어 점유 상승은 기계적으로 유지되기 쉽습니다. 도요타 제외 HHI(아래)도 함께 보세요.")
    rows = []
    for nm, S in V.items():
        for b in [BL[0], b1]:
            c = CORE[(CORE.blk == b) & CORE.corp_norm.isin(S - {TOY})].groupby("corp_norm").size(); s = c / c.sum() if c.sum() else c
            rows.append(dict(변형=nm, 블록=LAB[b], HHI_도요타제외=float((s ** 2).sum()) if len(s) else np.nan))
    X = pd.DataFrame(rows).pivot(index="변형", columns="블록", values="HHI_도요타제외"); X["변화"] = X.iloc[:, -1] - X.iloc[:, 0]
    print("\n[2-3] 도요타 제외 HHI"); print(X.round(3).to_string())
safe(A2)

# ======================= 분석 3: 특허 활동 지속성과 창 이후 과학 =======================
def A3():
    cols = [c for c in ["early_share", "n_pubyears", "n_win", "first_pub"] if c in FM.columns]
    if not cols: print("[3] 필요한 열이 없습니다"); return
    pa = Ud.groupby(["corp_norm", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0)
    early = pa[[BL[0], BL[1]]].sum(1) / (NYs[BL[0]] + NYs[BL[1]]); late = pa[[BL[2], b1]].sum(1) / (NYs[BL[2]] + NYs[b1])
    D = pd.DataFrame({"early": early, "late": late}).join(FM[cols], how="left"); D = D[(D.early >= 2) & D.index.notna()]
    D["lr"] = np.log((D.late + 0.5) / (D.early + 0.5)); D["ln_win"] = np.log(D.n_win) if "n_win" in D.columns else np.nan
    D2 = D.drop(index=[TOY], errors="ignore").dropna(subset=["lr"])
    print(f"[3] 기업 {len(D2)}곳(도요타 제외, 창 안 연 2편 이상). 결과변수 = ln(창 이후 연 논문 / 창 안 연 논문)")
    print(D2[["early", "late", "lr"] + cols].describe().round(2).T.to_string())
    for x in [c for c in ["early_share", "n_pubyears"] if c in D2.columns]:
        d = D2.dropna(subset=[x]); z = (d[x] - d[x].mean()) / d[x].std()
        X = sm.add_constant(pd.DataFrame({"z": z, "ln_win": d.ln_win}).dropna()); y = d.loc[X.index, "lr"]
        m = sm.OLS(y, X).fit(cov_type="HC1"); rho, p = stats.spearmanr(d[x], d["lr"])
        print(f"  {x}: 표준화 계수 {m.params['z']:+.3f} (se {m.bse['z']:.3f}, p={m.pvalues['z']:.3f}) | 특허규모 통제 | Spearman {rho:+.3f} (p={p:.3f}) | N={int(m.nobs)}")
        d = d.assign(tert=pd.qcut(d[x].rank(method="first"), 3, labels=["하", "중", "상"])); print(d.groupby("tert").agg(기업수=("lr", "size"), 중앙_lr=("lr", "median"), 평균_lr=("lr", "mean")).round(3).to_string())
    print("  해석 가이드: early_share의 정확한 정의를 확인한 뒤에만 해석하세요. 탐색적, 인과 아님.")
    save(D2.reset_index(), "3_patent_persistence_vs_science")
safe(A3)

# ======================= 분석 4: 특허 쏠림 대 과학 쏠림 =======================
def A4():
    firms = sorted(Ud.corp_norm.unique()); pat = FM["n_win"].reindex(firms).fillna(0); ps = pat / pat.sum()
    def met(s):
        s = s[s > 0]; s = s / s.sum(); return dict(도요타=float(s.get(TOY, 0)), 상위4=float(s.nlargest(4).sum()), HHI=float((s ** 2).sum()), 유효기업수=float(1 / (s ** 2).sum()))
    rows = [dict(구분="특허(마스터 133곳)", **met(FM["n_win"])), dict(구분="특허(분석 66곳)", **met(pat))]
    for b in BL: rows.append(dict(구분=f"전체 논문 {LAB[b]}", **met(Ud[Ud.blk == b].groupby("corp_norm").size().reindex(firms).fillna(0))))
    for b in BL: rows.append(dict(구분=f"core 논문 {LAB[b]}", **met(CORE[CORE.blk == b].groupby("corp_norm").size().reindex(firms).fillna(0))))
    T = pd.DataFrame(rows); print("[4-1] 특허 점유 대 논문 점유"); print(T.round(3).to_string(index=False)); save(T, "4_patent_vs_science_concentration")
    rows = []
    for b in BL:
        c = CORE[CORE.blk == b].groupby("corp_norm").size().reindex(firms).fillna(0); a = Ud[Ud.blk == b].groupby("corp_norm").size().reindex(firms).fillna(0)
        r1, p1 = stats.spearmanr(pat, c); r2, p2 = stats.spearmanr(pat, a)
        rows.append(dict(블록=LAB[b], 도요타_특허점유=float(ps[TOY]), 도요타_전체논문점유=float(a[TOY] / a.sum()), 도요타_core점유=float(c[TOY] / c.sum()),
                         core과대표현=float(c[TOY] / c.sum() / ps[TOY]), 순위상관_특허_core=r1, p_core=p1, 순위상관_특허_전체=r2, p_전체=p2))
    R = pd.DataFrame(rows); print("\n[4-2] 도요타의 특허 대비 과학 점유와 기업 수준 순위상관"); print(R.round(3).to_string(index=False)); save(R, "4_overrepresentation")
    print("  서술 주의: 특허 점유는 창 안 합계, 논문 점유는 블록별. '나란히 나타났다'로만 쓰고 방향은 주장하지 않습니다.")
safe(A4)

# ======================= 분석 5: 분야 단위 점검 =======================
def A5():
    D0 = pd.read_csv("panel_new_design.csv"); pc = next((c for c in ["n_pat_new", "n_pat", "n_patent"] if c in D0.columns), None)
    print("[5] panel_new_design 열:", D0.columns.tolist(), "| 특허 열:", pc)
    if pc is None: print("  특허 열을 못 찾음"); return
    P = D0.groupby(["domain", "corp_norm"])[pc].sum().reset_index()
    def met(s):
        s = s[s > 0]
        if s.sum() == 0: return dict(도요타=np.nan, 상위4=np.nan, HHI=np.nan, 기업수=0)
        s = s / s.sum(); return dict(도요타=float(s.get(TOY, 0)), 상위4=float(s.nlargest(4).sum()), HHI=float((s ** 2).sum()), 기업수=len(s))
    rows = []
    for dm in sorted(Ud.domain.dropna().unique()):
        pt = P[P.domain == dm].set_index("corp_norm")[pc]; mp = met(pt)
        r = dict(분야=dm, 특허_도요타=mp["도요타"], 특허_HHI=mp["HHI"], 특허_기업수=mp["기업수"])
        for b in [BL[0], b1]:
            s = Ud[(Ud.domain == dm) & (Ud.blk == b)].groupby("corp_norm").size(); ms = met(s)
            r[f"논문_도요타_{LAB[b]}"] = ms["도요타"]; r[f"논문_HHI_{LAB[b]}"] = ms["HHI"]; r[f"논문편수_{LAB[b]}"] = s.sum()
        rows.append(r)
    T = pd.DataFrame(rows); T["도요타점유_변화"] = T[f"논문_도요타_{LAB[b1]}"] - T[f"논문_도요타_{LAB[BL[0]]}"]; T["HHI_변화"] = T[f"논문_HHI_{LAB[b1]}"] - T[f"논문_HHI_{LAB[BL[0]]}"]
    print("\n[5-1] 분야별 특허 점유 대 논문 점유"); print(T.round(3).to_string(index=False)); save(T, "5_domain_patent_vs_science")
    d = T.dropna(subset=["특허_도요타", "도요타점유_변화"])
    if len(d) >= 5:
        r, p = stats.spearmanr(d["특허_도요타"], d["도요타점유_변화"]); r2, p2 = stats.spearmanr(d["특허_HHI"], d["HHI_변화"])
        print(f"\n[5-2] 분야 {len(d)}개 순위상관: 특허 도요타 점유 vs 논문 도요타 점유 변화 {r:+.3f} (p={p:.3f}) | 특허 HHI vs 논문 HHI 변화 {r2:+.3f} (p={p2:.3f})")
        print("  분야 8개뿐이라 검정력이 매우 낮습니다. 부호와 표만 서술하세요.")
    c = CORE.groupby(["domain", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0).div(NYs, axis=1); c.columns = [LAB[b] for b in BL]
    print("\n[5-3] 분야별 core 연 편수"); print(c.round(1).to_string())
safe(A5)
print(f"\n[완료] 표는 {FIN}/ 에 저장")

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import os, re, time, warnings, traceback
from collections import Counter
import numpy as np, pandas as pd, requests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 200)

need = ["U", "FM", "CORE", "BL", "LAB", "YMIN", "YLAST", "BLK", "OUT", "AUTH", "PART", "NY", "b1"]
miss = [v for v in need if v not in globals()]; assert not miss, f"이전 셀 변수가 없습니다: {miss}"
if "safe" not in globals():
    def safe(f):
        try: f()
        except Exception as e:
            print(f"\n[{f.__name__} 실패] {type(e).__name__}: {str(e)[:250]}"); traceback.print_exc(limit=2)
if "oa_get" not in globals():
    def oa_get(path, params, retries=4):
        p = dict(params); k = os.environ.get("OPENALEX_API_KEY"); m = os.environ.get("OPENALEX_MAILTO")
        if k: p["api_key"] = k
        if m: p["mailto"] = m
        for i in range(retries):
            try:
                r = requests.get("https://api.openalex.org/" + path, params=p, timeout=60)
                if r.status_code == 200: return r.json()
                if r.status_code in (429, 500, 502, 503): time.sleep(2 ** i); continue
                print(f"  [oa_get] HTTP {r.status_code}: {r.text[:200]}"); return None
            except Exception as e: time.sleep(2 ** i)
        return None
FIN = f"{OUT}/final"; os.makedirs(FIN, exist_ok=True)
def save(df, name): df.to_csv(f"{FIN}/{name}.csv", index=False, encoding="utf-8-sig")
TOY = globals().get("TOY", "トヨタ自動車")
Ud = U[U.year <= YLAST].copy()
short = lambda v: str(v).split("/")[-1]

# ====================== A. 기업 수 정리 ======================
def A():
    LKc = set(LK.corp_norm.unique()) if "LK" in globals() else set()
    rows = [("특허 마스터(firm_master_window)", len(FM)), ("파트너 링크 파일에 등장", len(LKc & set(FM.index)) if LKc else np.nan),
            ("논문 보유(U)", Ud.corp_norm.nunique()), ("core 논문 보유", CORE.corp_norm.nunique())]
    for b in BL: rows.append((f"core 보유, 블록 {LAB[b]}", CORE[CORE.blk == b].corp_norm.nunique()))
    T = pd.DataFrame(rows, columns=["기준", "기업 수"]); print("[A] 서술에서 쓸 기업 수 기준"); print(T.to_string(index=False))
    inU = set(Ud.corp_norm); print("  U에는 있으나 마스터에 없는 기업:", sorted(inU - set(FM.index))[:10], "| 마스터에는 있으나 U에 없는 기업 수:", len(set(FM.index) - inU))
    save(T, "A_firm_counts")
safe(A)

# ====================== B. 일본 기업 상위 기관 대조 ======================
OWN = {}
def build_own():
    cnt = Counter()
    for r in Ud[["w", "corp_norm"]].drop_duplicates().itertuples(index=False):
        a = AUTH.get(r.w)
        if not a: continue
        pset = PART.get((r.w, r.corp_norm), set()); seen = set()
        for aid, pos, corr, types, iids, ctry in a:
            if len(types) != len(iids): continue
            for t, i in zip(types, iids):
                if t == "company" and i not in pset and (r.corp_norm, i) not in seen: cnt[(r.corp_norm, i)] += 1; seen.add((r.corp_norm, i))
    best = {}
    for (c, i), n in cnt.items():
        if n >= 3 and (i not in best or n > best[i][1]): best[i] = (c, n)
    OWN.update({i: c for i, (c, n) in best.items()})
    print(f"[B-0] 표본 기업의 자기 기관 추정: 기관 {len(OWN)}개, 기업 {len(set(OWN.values()))}곳 (논문 3편 이상 등장한 기업 소속 기관)")
    print("  도요타 계열:", [i for i, c in OWN.items() if c == TOY][:10])

def B():
    build_own()
    tid = lambda t: str(t) if re.match(r"^T\d+$", str(t)) else ("T" + str(t) if re.match(r"^\d+$", str(t)) else None)
    topics = sorted({tid(t) for t in U.loc[U.cls_f == "core", "topic_id"].dropna().unique()} - {None})
    chunks = [topics[i:i + 10] for i in range(0, len(topics), 10)]; inst = {}
    for b in BL:
        y0 = YMIN + b * BLK; y1 = min(y0 + BLK - 1, YLAST)
        for ch in chunks:
            flt = f"primary_topic.id:{'|'.join(ch)},authorships.institutions.country_code:JP,authorships.institutions.type:company,publication_year:{y0}-{y1}"
            g = oa_get("works", {"filter": flt, "group_by": "authorships.institutions.id"})
            for r in (g or {}).get("group_by", []): inst[(b, short(r["key"]))] = inst.get((b, short(r["key"]), ), 0) + r["count"]
    keys = sorted({k[1] for k in inst}); meta = {}
    for i in range(0, len(keys), 50):
        js = oa_get("institutions", {"filter": "openalex:" + "|".join(keys[i:i + 50]), "select": "id,display_name,country_code,type", "per-page": 50})
        for r in (js or {}).get("results", []): meta[short(r["id"])] = (r["display_name"], r["country_code"], r["type"])
    rows = [dict(블록=b, inst=i, 기관=meta.get(i, ("?", "?", "?"))[0], 국가=meta.get(i, ("?", "?", "?"))[1], 유형=meta.get(i, ("?", "?", "?"))[2], 논문=n) for (b, i), n in inst.items()]
    D = pd.DataFrame(rows); D = D[(D.국가 == "JP") & (D.유형 == "company")].copy(); D["표본기업"] = D.inst.map(OWN)
    D["표본내"] = D.표본기업.notna()
    S = D.groupby(["블록", "표본내"]).논문.sum().unstack(fill_value=0).rename(columns={True: "표본 기관 논문(중복 합)", False: "표본 밖 기관 논문(중복 합)"})
    S["표본 기관 비중"] = S.iloc[:, -1] / S.sum(1) if S.shape[1] == 2 else np.nan
    S.index = [LAB[b] for b in S.index]; print("\n[B-1] 일본 기업 기관 기준 논문 수(기관별 합산이라 한 논문이 여러 기관에 중복). 표본 기관의 비중"); print(S.round(3).to_string())
    L = D[D.블록 == b1].sort_values("논문", ascending=False).head(40)[["기관", "inst", "논문", "표본기업"]]
    print("\n[B-2] 마지막 블록 일본 기업 기관 상위 40 (표본기업이 NaN이면 표본 밖 또는 자기 기관 추정 실패)"); print(L.to_string(index=False))
    save(D, "B_japan_company_inst_vs_sample")
safe(B)

# ====================== C. 표본 밖 대기업의 원인 ======================
ALIAS = {"Panasonic": ["パナソニック", "松下"], "Hitachi": ["日立"], "Denso": ["デンソー"], "Osaka Gas": ["大阪ガス"], "Tosoh": ["東ソー"], "Murata": ["村田"],
         "Noritake": ["ノリタケ"], "Asahi Kasei": ["旭化成"], "Toshiba": ["東芝"], "Qualtec": ["クォルテック"], "De Nora": ["デノラ", "ペルメレック"], "Eneos": ["ＥＮＥＯＳ", "ENEOS", "JXTG", "新日本石油", "JX"],
         "JEOL": ["日本電子"], "Rigaku": ["リガク"], "Sumitomo Chemical": ["住友化学"], "Mitsubishi Chemical": ["三菱ケミカル", "三菱化学"], "Kyocera": ["京セラ"], "Sony": ["ソニー"]}
def C_():
    rows = []
    corp = FM.reset_index(); corp["corp_ja"] = corp["corp_ja"].astype(str)
    for en, kws in ALIAS.items():
        hit = corp[corp.corp_ja.apply(lambda s: any(k in s for k in kws)) | corp.corp_norm.astype(str).apply(lambda s: any(k in s for k in kws))]
        if len(hit) == 0: rows.append(dict(기업=en, 마스터=False, 해당=None, 논문보유=False, core논문=0, n_win=np.nan))
        for r in hit.itertuples(index=False):
            c = r.corp_norm; rows.append(dict(기업=en, 마스터=True, 해당=c, 논문보유=c in set(Ud.corp_norm), core논문=int((CORE.corp_norm == c).sum()), n_win=r.n_win))
    T = pd.DataFrame(rows); print("[C] 표본 밖 후보 대기업의 상태 (마스터=False면 특허 창 추출에서 안 잡힘, 마스터=True & 논문보유=False면 논문 매칭 누락)")
    print(T.to_string(index=False)); save(T, "C_missing_firm_status")
    print("  마스터=True, 논문보유=False인 기업이 있으면 기관 매칭 단계에서 탈락한 것이라 고칠 수 있습니다(해당 기관 ID를 매칭표에 추가).")
    print("  마스터=False인 대기업이 있으면 FI 분류 8개 분야 안에서 2006~2015 특허 10건 미만이거나 추출 조건에서 빠진 것입니다. 특허 추출 쿼리를 확인하세요.")
safe(C_)

# ====================== D. 열 정의 추정 ======================
def D_():
    F = FM.copy(); F["from_n"] = 1 - F.n_2010_15 / F.n_win
    print("[D-1] early_share 추정: 1 - n_2010_15/n_win 와 비교")
    d = F[["early_share", "from_n"]].dropna(); print("  상관", round(d.corr().iloc[0, 1], 3), "| 완전 일치 비율", round((d.early_share - d.from_n).abs().lt(0.005).mean(), 3), "| 평균 절대차", round((d.early_share - d.from_n).abs().mean(), 3))
    F["frac_wo"] = F.n_wo / F.n_win; print("[D-2] n_wo: 설명 없음. n_win 대비 분포"); print(F[["n_wo", "n_win", "frac_wo"]].describe().round(3).T.to_string())
    print("  n_wo == n_win - n_2010_15 비율:", round(((F.n_wo - (F.n_win - F.n_2010_15)).abs() < 0.5).mean(), 3), "| n_wo == n_win 비율:", round((F.n_wo == F.n_win).mean(), 3))
    D0 = pd.read_csv("panel_new_design.csv"); s = D0.groupby("corp_norm")[["n_pat_new", "n_pat_old"]].sum().join(FM[["n_win", "n_2010_15"]], how="inner")
    print("\n[D-3] panel의 n_pat_new, n_pat_old (기업 합) 와 마스터 n_win 비교")
    print(s.assign(합=s.n_pat_new + s.n_pat_old).describe().round(1).T.to_string())
    print("  n_pat_new 합 == n_2010_15: ", round(((s.n_pat_new - s.n_2010_15).abs() < 0.5).mean(), 3), "| n_pat_new+n_pat_old == n_win:", round(((s.n_pat_new + s.n_pat_old - s.n_win).abs() < 0.5).mean(), 3),
          "| n_pat_new 합 <= n_win:", round((s.n_pat_new <= s.n_win + 0.5).mean(), 3))
    print("  해석: 일치율이 0.9 이상인 가정이 있으면 그 정의로 확정하고, 모두 낮으면 열 생성 코드를 직접 확인하세요. 분야에 중복 배정된 특허는 합이 n_win을 넘을 수 있습니다.")
safe(D_)

# ====================== E. 마스터에서 논문이 없는 기업 ======================
def E():
    ex = FM[~FM.index.isin(set(Ud.corp_norm))].sort_values("n_win", ascending=False)
    print(f"[E] 논문이 없는 마스터 기업 {len(ex)}곳, 특허 합 {int(ex.n_win.sum())}건 (마스터 전체 {int(FM.n_win.sum())}건의 {ex.n_win.sum()/FM.n_win.sum():.1%})")
    print("  상위 15곳:"); print(ex[["corp_ja", "industry", "n_win", "tier"]].head(15).to_string())
    print("\n  tier별 포함 비율:"); print(pd.crosstab(FM.tier, FM.index.isin(set(Ud.corp_norm)), normalize="index").round(3).rename(columns={True: "논문 있음", False: "논문 없음"}).to_string())
    save(ex.reset_index(), "E_excluded_firms")
safe(E)
print(f"\n[완료] 표는 {FIN}/ 에 저장")

In [ ]:
#

In [ ]:
import pandas as pd
df = pd.read_pickle("patents_clean.pkl")
df["pub_year"] = df["pub_date"].dt.year
df["fil_year"] = df["filing"].dt.year
t = df.groupby("fil_year").agg(n=("app_key","size"), firms=("corporate","nunique"))
print(t.loc[2008:2022].to_string())
print("공개일 최댓값:", df["pub_date"].max())
print("출원 2014 이후 건수가 공개지연으로 잘리는지: 출원연도별 공개 연도 중앙값")
print(df.groupby("fil_year")["pub_year"].median().loc[2010:2020].to_string())

In [ ]:
import pandas as pd
for f in ["Japanese GX patents.xlsx", "Japanese GX patents_with Englishi title.xlsx"]:
    try:
        x = pd.read_excel(f); print(f, len(x), "행 | 열:", list(x.columns)[:12])
        c = next(c for c in x.columns if "Filing" in str(c) or "出願日" in str(c))
        y = pd.to_datetime(x[c], errors="coerce").dt.year
        print(y.value_counts().sort_index().to_string())
    except Exception as e: print(f, "실패", type(e).__name__, str(e)[:100])

In [ ]:
#

In [ ]:
import pandas as pd
a = pd.read_excel("Japanese GX patents_with Englishi title.xlsx")
b = pd.read_csv("patents_Eng.csv", low_memory=False)
print("원본 기업 수:", a["corporate"].nunique(), "| 우리 파일:", b["corporate"].nunique())
print("원본 Applicant에 파나소닉 포함:", int(a["Applicant/Right holder"].astype(str).str.contains("パナソニック|松下").sum()))
print("원본 corporate에 파나소닉/덴소/히타치:", [int(a["corporate"].astype(str).str.contains(k).sum()) for k in ["パナソニック", "デンソー", "日立製作所"]])
a["fy"] = pd.to_datetime(a["Filing date"], errors="coerce").dt.year
b["fy"] = pd.to_datetime(b["Filing date"], unit="D", origin="1899-12-30", errors="coerce").dt.year
print("원본 출원 ≤2015:", int((a.fy <= 2015).sum()), "| 우리 파일:", len(b))
key = "Application number"
miss = a[~a[key].isin(set(b[key]))]
print("원본에만 있는 건수:", len(miss), "| 그중 출원≤2015:", int((miss.fy <= 2015).sum()))
print("원본에만 있는 기업 상위:"); print(miss["corporate"].value_counts().head(10).to_string())

In [ ]:
#

In [ ]:
import pandas as pd
a = pd.read_excel("Japanese GX patents_with Englishi title.xlsx")
a["fy"] = pd.to_datetime(a["Filing date"], errors="coerce").dt.year
a["py"] = pd.to_datetime(a["Publication date"], errors="coerce").dt.year
w = a[(a.py >= 2006) & (a.py <= 2015)]
print("원본에서 공개 2006~2015:", len(w), "| 기업", w.corporate.nunique(), "(우리 창: 19,802건, 133곳)")
late = a[a.fy >= 2016]
early = set(w.corporate)
print("출원 2016+ 특허 중 창 안 기업 소속 비중:", round(late.corporate.isin(early).mean(), 3))
new = late[~late.corporate.isin(early)]
print("창 이후에야 등장한 기업:", new.corporate.nunique(), "곳, 특허", len(new), f"건 ({len(new)/len(late):.1%})")
print(new.corporate.value_counts().head(10).to_string())
c1 = w.groupby("corporate").size(); c2 = late.groupby("corporate").size()
x = pd.concat([c1.rename("win"), c2.rename("late")], axis=1).fillna(0)
print("창 안 vs 창 이후 특허 수 순위상관:", round(x.corr(method="spearman").iloc[0, 1], 3))

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import os, re, json, time, subprocess, warnings, traceback
import numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf, requests
from scipy import stats
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 200)

# ============================ 설정 ============================
ORIG_XLSX = "Japanese GX patents_with Englishi title.xlsx"
CLEAN_PKL = "patents_clean.pkl"
RATER2_FILE = f"{OUT}/gx_review_rater2.csv"   # 평가자 2가 채운 파일(열: topic, gx_class_final). 없으면 D는 시트만 만듦
RUN_LIT = False                                # True면 OpenAlex 문헌 후보 검색
NB = 1000; SEED = 0
TOY, NIS = "トヨタ自動車", "日産自動車"
CLS5 = ["core", "enabling", "legacy", "non_gx", "other"]
FIN = f"{OUT}/final"; os.makedirs(FIN, exist_ok=True)
def save(df, name): df.to_csv(f"{FIN}/{name}.csv", index=False, encoding="utf-8-sig")
def safe(f):
    try: f()
    except Exception as e:
        print(f"\n[{f.__name__} 실패] {type(e).__name__}: {str(e)[:250]}"); traceback.print_exc(limit=2)
for v in ["U", "BL", "b0", "b1", "LAB", "NY", "YLAST", "YMIN", "BLK", "OUT"]:
    assert v in globals(), f"이전 셀 변수 없음: {v}"
Ud = U[U.year <= YLAST].copy()
if "cls_f" not in Ud.columns: print("경고: cls_f가 없어 core 관련 모듈(D, E, F 일부)이 실패할 수 있음(코드 ⑫ 0단계)")
CORE = Ud[Ud.cls_f == "core"].copy() if "cls_f" in Ud.columns else Ud.iloc[0:0]

def load_orig():
    a = pd.read_excel(ORIG_XLSX)
    a["fy"] = pd.to_datetime(a["Filing date"], errors="coerce").dt.year
    a["py"] = pd.to_datetime(a["Publication date"], errors="coerce").dt.year
    return a
_ORIG = {}
def orig():
    if "a" not in _ORIG: _ORIG["a"] = load_orig()
    return _ORIG["a"]

# ============================ A. 대형 출원인 비포함 정량화 + 저자 문의 ============================
def A():
    a = orig(); col = "Applicant/Right holder"; s = a[col].astype(str)
    names = {"Panasonic": "パナソニック|松下電器|松下電工", "Denso": "デンソー", "Hitachi(본체)": "日立製作所", "Sony": "ソニー", "Sharp": "シャープ",
             "Toshiba(본체)": "東芝(?!テック)", "Honda R&D": "本田技術研究所", "Mitsubishi Chemical": "三菱ケミカル|三菱化学", "Asahi Kasei": "旭化成", "Kyocera": "京セラ", "Murata": "村田製作所"}
    rows = []
    for k, pat in names.items():
        m = s.str.contains(pat, regex=True, na=False); sub = a[m]
        single = sub[col].astype(str).str.strip().str.fullmatch(pat).fillna(False) if len(sub) else pd.Series([], dtype=bool)
        rows.append(dict(기업=k, 출원인열_등장=int(m.sum()), 공개창2006_15_등장=int((m & a.py.between(2006, 2015)).sum()),
                         corporate열에도_있음=int(a["corporate"].astype(str).str.contains(pat, regex=True, na=False).sum()),
                         공동출원_상대_상위=", ".join(sub["corporate"].value_counts().head(3).index.astype(str))))
    T = pd.DataFrame(rows); print("[A-1] 대형 출원인이 원본에서 어떻게 나타나는가 (corporate=0이면 단독 기업 단위로는 목록에 없음)"); print(T.to_string(index=False)); save(T, "A1_large_applicants")
    w = a[a.py.between(2006, 2015)]; co = w[col].astype(str).str.contains(r"[,;／/、，]|\n", regex=True)
    print(f"\n[A-2] 공개 창 {len(w)}건 중 공동출원(구분자 포함) 추정 {co.mean():.1%}. 구분자 형식은 원본 확인 필요")
    ex = a.loc[a[col].astype(str).str.contains("パナソニック"), ["corporate", col, "Publication date"]].head(5); print(ex.to_string(index=False))
    mail = f"""Subject: Question about company coverage in "Japanese corporates GX patents" (Mendeley Data / F1000Research)

Dear Dr. Jiang,

I am using your dataset (37,476 patents; Toyo Keizai company list; JPO GXTI search formula) to study the scientific output of the listed firms.
I noticed that several large Japanese firms do not appear as a 'corporate' (company-name) entry, although they appear as co-applicants of other firms' patents
(e.g., Panasonic, Denso, Hitachi (parent), Sony, Sharp). Could you clarify:
1. Were these firms absent from the Toyo Keizai list used, or were they searched but returned no hits (e.g., due to name changes or holding-company structures)?
2. How was the final firm list of about 294-298 companies determined (industry, listing status, year)?
3. Is the search formula's keyword part applied to titles/abstracts, and is the same formula applied to all firms?
I will cite your answer in the limitations section. Thank you very much.

Best regards,
[Name]
"""
    open(f"{FIN}/A_email_to_author.txt", "w", encoding="utf-8").write(mail)
    lim = ("Limitation (draft): The firm cohort is defined by an external list (Toyo Keizai) rather than by patent activity. Some large applicants (e.g., Panasonic, Denso, Hitachi) "
           "appear only as co-applicants of cohort firms and not as cohort members. Concentration and share estimates therefore describe this cohort, not Japan's GX science as a whole.")
    open(f"{FIN}/A_limitation_paragraph.txt", "w", encoding="utf-8").write(lim); print("\n  저장: A_email_to_author.txt, A_limitation_paragraph.txt (사유 답변이 오면 인용 문장을 수정하세요)")
safe(A)

# ============================ B. 2016년 이후 특허의 연속성 ============================
def B():
    a = orig(); w = a[a.py.between(2006, 2015)]; win = set(w.corporate)
    rows = []
    for y in range(2012, 2023):
        d = a[a.fy == y]; rows.append(dict(출원연도=y, 건수=len(d), 기업수=d.corporate.nunique(), 창안기업_비중=d.corporate.isin(win).mean() if len(d) else np.nan))
    T = pd.DataFrame(rows); print("[B-1] 출원연도별 건수와 창 안 기업 소속 비중"); print(T.round(3).to_string(index=False)); save(T, "B1_continuity_by_year")
    late = a[(a.fy >= 2016) & (a.fy <= 2018)]; full = a[a.fy >= 2016]
    print(f"\n[B-2] 출원 2016+ {len(full)}건 중 창 안 기업 {full.corporate.isin(win).mean():.1%} | 2016~2018만 {late.corporate.isin(win).mean():.1%}")
    new = full[~full.corporate.isin(win)]; print(f"  창 이후에야 등장한 기업 {new.corporate.nunique()}곳, {len(new)}건({len(new)/len(full):.1%})")
    c1 = w.groupby("corporate").size().rename("win"); c2 = late.groupby("corporate").size().rename("late"); X = pd.concat([c1, c2], axis=1).fillna(0)
    X = X[X.win > 0]; r = stats.spearmanr(X.win, X.late)[0]; rng = np.random.default_rng(SEED); bs = []
    for _ in range(NB):
        s = X.iloc[rng.integers(0, len(X), len(X))]
        if s.win.nunique() > 1 and s.late.nunique() > 1: bs.append(stats.spearmanr(s.win, s.late)[0])
    print(f"[B-3] 기업별 특허 수 순위상관(창 안 vs 출원 2016~2018): {r:.3f} [{np.percentile(bs,2.5):.3f}, {np.percentile(bs,97.5):.3f}] (기업 재표집, 기업 {len(X)}곳)")
    def sh(c): s = c / c.sum(); return float(s.get(TOY, 0)), float(s.nlargest(4).sum()), float((s ** 2).sum())
    for lab, c in [("창 안(공개 2006~15)", c1), ("출원 2016~2018", c2[c2 > 0])]:
        t, f4, h = sh(c); print(f"  {lab:<20} 도요타 {t:.1%} | 상위4 {f4:.1%} | HHI {h:.3f}")
    print("  해석: 창 이후 특허의 구성이 창 안과 비슷하면 '창이 코호트 구조를 임의로 자른 것은 아니다'로 서술. 인과 주장 아님.")
    save(X.reset_index(), "B3_firm_patent_counts_window_vs_late")
safe(B)

# ============================ C. 공개 창 건수 불일치 진단 ============================
def C_():
    a = orig(); w = a[a.py.between(2006, 2015)].copy()
    ak = "Application number"; dk = "Document number"
    print(f"[C-1] 원본 공개 2006~2015: 행 {len(w)} | 고유 출원번호 {w[ak].nunique()} | 고유 문헌번호 {w[dk].nunique()} | 고유 (기업,출원번호) {w[['corporate', ak]].drop_duplicates().shape[0]}")
    print(f"  FI 결측 {int(w['FI'].isna().sum())} | 출원일 결측 {int(w.fy.isna().sum())} | 공개일 결측 {int(w.py.isna().sum())}")
    if os.path.exists(CLEAN_PKL):
        b = pd.read_pickle(CLEAN_PKL); print("  우리 파일 열:", list(b.columns)[:15], "| 행", len(b))
        pcol = next((c for c in ["pub_date", "Publication date"] if c in b.columns), None)
        if pcol is not None:
            py = pd.to_datetime(b[pcol], errors="coerce").dt.year; bw = b[py.between(2006, 2015)]; print(f"  우리 파일 중 공개 2006~2015: {len(bw)}행")
            kcol = next((c for c in ["app_key", "Application number"] if c in b.columns), None)
            if kcol is not None:
                norm = lambda s: s.astype(str).str.replace(r"\D", "", regex=True)
                A_ = set(norm(w[ak])); B_ = set(norm(bw[kcol]))
                print(f"  출원번호(숫자만) 기준: 원본 창 {len(A_)} | 우리 창 {len(B_)} | 원본에만 {len(A_ - B_)} | 우리에만 {len(B_ - A_)}")
                miss = w[~norm(w[ak]).isin(B_)]
                print("  원본에만 있는 건의 특성:"); print("   기업 상위:", miss.corporate.value_counts().head(8).to_dict()); print("   출원연도:", miss.fy.value_counts().sort_index().to_dict())
                print("   FI 결측 비중:", round(miss["FI"].isna().mean(), 3), "| 공개연도 분포:", miss.py.value_counts().sort_index().to_dict())
                save(miss.drop(columns=[c for c in miss.columns if miss[c].dtype == object and miss[c].astype(str).str.len().max() > 500], errors="ignore"), "C_missing_in_ours")
    dup = w.duplicated(subset=[ak, "corporate"], keep=False); print(f"\n[C-2] 원본 창 안 (출원번호,기업) 중복 행 {int(dup.sum())}건 (한 출원에 문헌번호 여러 개이면 정제에서 하나로 합쳐졌을 수 있음)")
    print(f"  원본 창 행수 − 고유(출원번호,기업) = {len(w) - w[[ak, 'corporate']].drop_duplicates().shape[0]} | 목표 차이 {len(w) - 19802}")
    print("  판정: 위 '중복 행 수'와 '원본에만 있는 건수'가 차이(721)를 설명하는지 확인. 설명 안 되면 정제 코드(patents_clean 생성부)를 직접 보세요.")
safe(C_)

# ============================ D. 평가자 2: 블라인드 시트 + κ + 합의 분류 재산출 ============================
def conc_table(cmap, label):
    d = Ud.copy(); d["c"] = d["topic"].map(cmap).fillna(d["cls_f"])
    t = d[d.c == "core"].groupby(["corp_norm", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0).div(pd.Series(NY).reindex(BL), axis=1)
    rows = []
    for k, b in enumerate(BL):
        c = t[b][t[b] > 0]; s = c / c.sum(); x = c[c.index != TOY]; sx = x / x.sum(); y = c[~c.index.isin([TOY, NIS])]; sy = y / y.sum()
        rows.append(dict(분류=label, 블록=LAB[b], core편수년=c.sum(), 도요타점유=float(s.get(TOY, 0)), HHI=float((s ** 2).sum()), HHI도요타제외=float((sx ** 2).sum()), HHI도요타닛산제외=float((sy ** 2).sum()), 유효기업수=float(1 / (s ** 2).sum())))
    return pd.DataFrame(rows)
def D():
    S = pd.read_csv(f"{OUT}/gx_review_sheet.csv"); keep = [c for c in ["topic", "domain", "n_total"] if c in S.columns]
    blank = S[keep].copy(); blank["gx_class_final"] = ""; blank["memo"] = ""
    blank = blank.sample(frac=1, random_state=SEED).reset_index(drop=True)   # 순서 섞기(논문 수 순위가 판단을 유도하지 않게)
    blank.to_csv(f"{OUT}/gx_review_rater2_blank.csv", index=False, encoding="utf-8-sig")
    guide = "# 평가자 2 안내\n- 각 주제를 core / enabling / legacy / other / non_gx 중 하나로 분류 (gx_class_final 열)\n- 기존 분류(제 초안, 평가자 1)를 보지 말 것\n- core: 탈탄소 에너지의 생산·저장·변환·수송 핵심 기술(태양광, 전지, 연료전지, 수소, 전기차 등)\n- enabling: 전력망, 인버터, 모터, 열 시스템 등 전환을 가능하게 하는 기술\n- legacy: 내연기관, 석탄 등 기존 화석 기반\n- other: 판단 불가·경계(메모에 이유)\n- non_gx: GX와 무관한 일반 재료·화학·생물\n- 원칙: 주제 이름만으로 '이 주제의 대표 논문이 GX 기능을 하는가'로 판단\n"
    open(f"{OUT}/gx_review_rater2_guide.md", "w", encoding="utf-8").write(guide)
    print(f"[D-0] 블라인드 시트 저장: {OUT}/gx_review_rater2_blank.csv ({len(blank)}주제, 순서 섞음), 안내문: gx_review_rater2_guide.md")
    if not os.path.exists(RATER2_FILE): print(f"[D] {RATER2_FILE} 가 아직 없어 κ는 건너뜀. 평가자 2가 채운 파일을 이 경로로 저장하고 다시 실행하세요."); return
    R2 = pd.read_csv(RATER2_FILE)[["topic", "gx_class_final"]].rename(columns={"gx_class_final": "r2"}); R2["r2"] = R2.r2.astype(str).str.strip().str.lower().replace({"non-gx": "non_gx", "nan": ""})
    M = pd.read_csv(f"{OUT}/gx_rater_merged.csv")[["topic", "mine", "rater", "n_total"]].rename(columns={"mine": "r0", "rater": "r1"}).merge(R2, on="topic", how="inner")
    M = M[M.r2.isin(CLS5) & M.r1.isin(CLS5)]; M["r0"] = M.r0.replace({"unreviewed": "other"}); print(f"  쌍별 비교 가능 주제 {len(M)}개")
    def kp(x, y, w=None):
        return dict(k5=cohen_kappa_score(M[x], M[y], labels=CLS5), kcore=cohen_kappa_score(M[x] == "core", M[y] == "core"), kw=cohen_kappa_score(M[x] == "core", M[y] == "core", sample_weight=M.n_total))
    rows = [dict(쌍=f"{x} vs {y}", **kp(x, y)) for x, y in [("r1", "r2"), ("r0", "r2"), ("r0", "r1")]]
    print("[D-1] 쌍별 κ (r0=제 초안, r1=평가자 1, r2=평가자 2)"); print(pd.DataFrame(rows).round(3).to_string(index=False)); save(pd.DataFrame(rows), "D1_pairwise_kappa")
    dis = M[M.r1 != M.r2].sort_values("n_total", ascending=False); cd = M[(M.r1 == "core") != (M.r2 == "core")].sort_values("n_total", ascending=False)
    print(f"\n[D-2] 평가자 1·2 불일치 {len(dis)}개, 그중 core 여부 불일치 {len(cd)}개(논문 {int(cd.n_total.sum())}편) -> 논의 대상"); print(cd.head(15).to_string(index=False)); save(dis, "D2_rater_disagreements")
    base = dict(zip(M.topic, M.r1)); m1 = dict(zip(M.topic, M.r2)); agree = {t: c for t, c in zip(M.topic, M.r1) if c == dict(zip(M.topic, M.r2))[t]}
    out = pd.concat([conc_table(base, "평가자1"), conc_table(m1, "평가자2"), conc_table({**base, **{t: "other" for t in cd.topic}}, "불일치 core→other(엄격)"),
                     conc_table({**base, **{t: "core" for t in cd.topic}}, "불일치 모두 core(관대)")])
    f, l = LAB[BL[0]], LAB[b1]; P = out[out.블록.isin([f, l])].pivot(index="분류", columns="블록", values=["도요타점유", "HHI", "HHI도요타제외", "HHI도요타닛산제외"]); print("\n[D-3] 분류별 core 집중(첫 블록, 끝 블록)"); print(P.round(3).to_string()); save(out, "D3_concentration_by_rater")
    print("  읽기: 모든 분류에서 도요타 점유 상승, 도요타 제외 HHI 상승, 도요타·닛산 제외 평탄이면 '2인 독립 분류에서 재현'")
safe(D)

# ============================ E. HHI 두 구간 한 표 ============================
def E():
    CM = CORE.groupby(["corp_norm", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0); firms = list(CM.index); M0 = CM.values.astype(float); Fn = len(firms); ti, ni = firms.index(TOY), firms.index(NIS)
    SC = {"전체": [], "도요타 제외": [ti], "도요타·닛산 제외": [ti, ni]}
    def hh(col, ex):
        c = col.copy(); c[ex] = 0; c = c[c > 0]; s = c / c.sum(); return float((s ** 2).sum())
    rng = np.random.default_rng(SEED); tot = M0.sum(0); R = {k: {"firm": [], "paper": []} for k in SC}
    for _ in range(NB):
        Mp = np.column_stack([rng.multinomial(int(tot[k]), M0[:, k] / tot[k]) for k in range(len(BL))]).astype(float); pick = rng.integers(0, Fn, Fn); Mf = M0[pick]
        for nm, ex in SC.items():
            R[nm]["paper"].append(hh(Mp[:, -1], ex) - hh(Mp[:, 0], ex)); R[nm]["firm"].append(hh(Mf[:, -1], [i for i, p in enumerate(pick) if p in ex]) - hh(Mf[:, 0], [i for i, p in enumerate(pick) if p in ex]))
    rows = []
    for nm, ex in SC.items():
        pt = hh(M0[:, -1], ex) - hh(M0[:, 0], ex)
        r = dict(시나리오=nm, HHI_첫=hh(M0[:, 0], ex), HHI_끝=hh(M0[:, -1], ex), 변화=pt)
        for kind, lab in [("paper", "논문재표집"), ("firm", "기업재표집")]:
            v = np.array(R[nm][kind]); r[f"{lab}_lo"], r[f"{lab}_hi"] = np.percentile(v, [2.5, 97.5]); r[f"{lab}_p"] = max(min(1, 2 * min((v <= 0).mean(), (v >= 0).mean())), 1 / (NB + 1))
        rows.append(r)
    T = pd.DataFrame(rows); print("[E] HHI 변화(마지막−첫 블록)의 두 95% 구간. p는 해상도 하한 1/(NB+1)로 클립"); print(T.round(3).to_string(index=False)); save(T, "E_hhi_two_intervals")
    print(f"  논문 문장: 같은 기업 구성에서 논문 표본 변동 구간이 0을 제외하면 '코호트 내 집중 상승이 관측됨'. 기업 구간이 0을 포함하면 '기업 구성 변동까지 고려하면 불확실, 도요타 의존 반영'을 병기.")
safe(E)

# ============================ F. 다중검정: 정확한 p 재계산 + 레지스트리 ============================
REG = []
def reg(name, fam, est, p, src): REG.append(dict(분석=name, 가설군=fam, 추정치=est, p=float(max(p, 1e-300)), 출처=src))
def F():
    # F3 저자 정의: 정확한 p
    if all(v in globals() for v in ["LFw", "classify", "cl_ols"]):
        rows = []
        for r in LFw.itertuples(index=False):
            cl = classify(r.w, r.corp_norm)
            if not cl or not any(c[2] for c in cl): continue
            ce = any(c[2] and c[3] for c in cl); cf = any(c[2] and c[4] for c in cl)
            rows.append(dict(corp_norm=r.corp_norm, blk=r.blk, corr=int(ce and not cf), last=int(r.ext), first=int(not bool(r.first_firm))))
        Cc = pd.DataFrame(rows); Cc = Cc[Cc.blk.isin([b0, b1])].copy(); Cc["post"] = (Cc.blk == b1).astype(int)
        for y, nm in [("last", "마지막 저자"), ("first", "제1저자"), ("corr", "교신저자")]:
            m = cl_ols(f"{y} ~ post + C(corp_norm)", Cc); reg(f"외부 주도 비중 변화(기업 FE): {nm}", "F3 저자정의", 100 * m.params["post"], m.pvalues["post"], "정확값")
    else: print("[F] LFw/classify/cl_ols 없음: F3 건너뜀")
    # F2 유입
    if all(v in globals() for v in ["classify", "cl_ols"]):
        rec = []
        for r in Ud[["w", "corp_norm", "blk", "solo", "cls_f"]].itertuples(index=False):
            cl = classify(r.w, r.corp_norm)
            if cl: rec += [(r.corp_norm, c[0], r.blk, r.solo, int(r.cls_f == "core")) for c in cl if c[4] and c[0]]
        Rr = pd.DataFrame(rec, columns=["corp_norm", "aid", "blk", "solo", "core"]).drop_duplicates()
        fb = Rr.groupby(["corp_norm", "aid"])["blk"].min().rename("fb").reset_index(); Rr = Rr.merge(fb, on=["corp_norm", "aid"])
        new = Rr[(Rr.blk == Rr.fb) & (Rr.blk > b0)].groupby(["corp_norm", "aid", "blk", "core"])["solo"].max().reset_index(); new["late"] = (new.blk == b1).astype(int)
        m = cl_ols("solo ~ late*core + C(corp_norm)", new); reg("신규 단독 저자 비중 하락(비core, 기업 FE)", "F2 유입", 100 * m.params["late"], m.pvalues["late"], "정확값")
        reg("신규 단독 저자 비중 core 추가차", "F2 유입", 100 * m.params["late:core"], m.pvalues["late:core"], "정확값")
        d = Ud[Ud.blk.isin([b0, b1]) & Ud.cls_f.isin(CLS5)].copy(); d["post"] = (d.blk == b1).astype(int); d["core"] = (d.cls_f == "core").astype(int)
        m2 = cl_ols("solo ~ post*core + C(corp_norm) + C(domain)", d); reg("단독 확률 core 추가차", "F5 core 특이성", 100 * m2.params["post:core"], m2.pvalues["post:core"], "정확값")
    # F4 진입
    if "Ua" in globals() and Ua is not None:
        for v in ["entry_v1", "entry_v2", "entry_v3"]:
            d = Ua[Ua.blk > b0]; d = d if v != "entry_v3" else d[(d.entry_v2 == 0) | (d.entry_v3 == 1)]
            X = sm.add_constant(pd.concat([d[["nw_any", "ln_n"]], pd.get_dummies(d.blk, prefix="b", drop_first=True)], axis=1).astype(float))
            m = sm.Logit(d[v].astype(float).values, X.values).fit(disp=0, maxiter=200, cov_type="cluster", cov_kwds={"groups": pd.factorize(d.corp_norm)[0]})
            reg(f"신규 파트너→진입 로그OR ({v})", "F4 진입", m.params[1], m.pvalues[1], "정확값")
    else: print("[F] Ua 없음(코드 ⑫ B 모듈 필요): F4 건너뜀")
    # F1 집중: 코드 E 결과 재사용
    p = f"{FIN}/E_hhi_two_intervals.csv"
    if os.path.exists(p):
        for r in pd.read_csv(p).itertuples():
            reg(f"HHI 변화 {r.시나리오} (논문 재표집)", "F1 집중", r.변화, r.논문재표집_p, "하한 클립"); reg(f"HHI 변화 {r.시나리오} (기업 재표집)", "F1 집중", r.변화, r.기업재표집_p, "하한 클립")
    # F6 전환점: 이전 출력 값(수기 보존). 재계산하려면 Q6a를 NB_BRK=999로 다시 실행
    for k, v in {"ALL": 0.001, "VEHICLE": 0.006, "MATERIAL": 0.028, "POWER": 0.074, "FUELCELL": 0.091, "THERMAL": 0.185, "BATTERY": 0.632, "SOLAR": 0.963}.items():
        reg(f"하락 전환점 {k}", "F6 전환점", np.nan, v, "이전 출력(부트스트랩 999)")
    Rg = pd.DataFrame(REG); Rg["p_holm_가설군내"] = np.nan; Rg["p_bh_가설군내"] = np.nan
    for g, ix in Rg.groupby("가설군").groups.items():
        Rg.loc[ix, "p_holm_가설군내"] = multipletests(Rg.loc[ix, "p"], method="holm")[1]; Rg.loc[ix, "p_bh_가설군내"] = multipletests(Rg.loc[ix, "p"], method="fdr_bh")[1]
    Rg["p_holm_전체"] = multipletests(Rg.p, method="holm")[1]; Rg["표기"] = "사후 선정한 주요 분석"
    print("[F] 레지스트리(사전 등록 없음 -> '확증'이 아니라 '사후 선정한 주요 분석'). 가설군 정의는 본인이 확정하세요"); print(Rg.sort_values(["가설군", "p"]).round(4).to_string(index=False)); save(Rg, "F_registry_exact")
    print("  p가 1e-300이면 논문에는 'p<0.001'로 표기. 누적 모형 수(약 280개)는 별도 장부로 관리하고, 이 표에는 보고 대상만 넣으세요.")
safe(F)

# ============================ G. 문헌: 후보 수집 + 주장×문헌 매트릭스 ============================
def oa_get(path, params, retries=4):
    p = dict(params); k = os.environ.get("OPENALEX_API_KEY"); m = os.environ.get("OPENALEX_MAILTO")
    if k: p["api_key"] = k
    if m: p["mailto"] = m
    for i in range(retries):
        try:
            r = requests.get("https://api.openalex.org/" + path, params=p, timeout=60)
            if r.status_code == 200: return r.json()
            if r.status_code in (429, 500, 502, 503): time.sleep(2 ** i); continue
            print(f"  [oa_get] HTTP {r.status_code}"); return None
        except Exception: time.sleep(2 ** i)
    return None
def G():
    claims = pd.DataFrame({"주장": ["기업 과학 논문의 감소·기업 R&D의 과학 이탈", "단독 연구에서 협력 연구로의 이동·팀 규모 증가", "해외 대학 협력 증가와 저자 위치(제1·마지막·교신)의 해석", "기업 논문 식별: 기관 매칭과 OpenAlex 기관 데이터의 한계",
                                 "특허와 논문의 연결(science linkage)과 특허 기반 표본", "새 파트너와 새 주제 진입(탐색·활용)", "연구 집중(소수 기업 쏠림)과 기업 과학 불평등", "일본 산학연 협력과 기업 중앙연구소 쇠퇴", "녹색기술(GX) 정의와 특허 분류(GXTI/Y02)",
                                 "저자 순서 규범과 주도권 해석"]})
    claims["핵심 키워드 제안"] = ["corporate science decline publications firms", "team size solo authorship decline collaboration", "corresponding author last author leadership international collaboration", "OpenAlex affiliation disambiguation company institutions",
                          "patent paper linkage firm science green technology", "new collaboration partners exploration topic entry firm", "concentration corporate research publications", "Japan industry university collaboration central research laboratory", "green transformation technology patents classification Japan", "author order norms credit contribution"]
    claims["읽은 문헌(저자·연도·저널)"] = "[문헌 확인]"; claims["이 연구와의 차이·연결"] = "[작성]"; claims["인용 확정"] = False; save(claims, "G_claim_literature_matrix")
    print("[G] 주장×문헌 매트릭스 템플릿 저장: G_claim_literature_matrix.csv (읽은 뒤 채우세요)")
    if not RUN_LIT: print("  RUN_LIT=False: 자동 후보 검색은 건너뜀"); return
    rows = []
    for k, q in enumerate(claims["핵심 키워드 제안"]):
        js = oa_get("works", {"search": q, "filter": "publication_year:>2004,type:article", "sort": "relevance_score:desc", "per-page": 25, "select": "id,title,publication_year,doi,cited_by_count,primary_location,relevance_score"})
        for r in (js or {}).get("results", []):
            src = ((r.get("primary_location") or {}).get("source") or {}).get("display_name")
            rows.append(dict(주장번호=k, 제목=r.get("title"), 연도=r.get("publication_year"), 인용=r.get("cited_by_count"), 저널=src, doi=r.get("doi")))
    L = pd.DataFrame(rows).drop_duplicates("doi"); save(L, "G_candidates"); print(f"  후보 {len(L)}편 저장. 제목만 보고 인용하지 말고 초록·본문을 읽은 뒤 매트릭스를 채우세요.")
safe(G)

# ============================ H. 보안: 폴더·git 이력 스캔 + 키 작동 확인 ============================
def H():
    pats = [re.compile(r"""(api[_-]?key|apikey|token|secret|password)\s*[:=]\s*\\?["']([^"'\\\s]{12,})""", re.I), re.compile(r"api_key=([A-Za-z0-9_\-]{12,})", re.I)]
    mask = lambda v: v[:3] + "***" + v[-2:]
    hits = []
    for root, dirs, files in os.walk("."):
        dirs[:] = [d for d in dirs if d not in (".git", "node_modules", "__pycache__", ".venv", "venv")]
        for fn in files:
            if not fn.lower().endswith((".py", ".ipynb", ".txt", ".md", ".env", ".json", ".cfg", ".yml", ".yaml", ".toml", ".csv")): continue
            pth = os.path.join(root, fn)
            try:
                if os.path.getsize(pth) > 20_000_000: continue
                for ln, line in enumerate(open(pth, encoding="utf-8", errors="ignore"), 1):
                    for pt in pats:
                        m = pt.search(line)
                        if m: hits.append((pth, ln, mask(m.group(m.lastindex))))
            except Exception: pass
    print(f"[H-1] 작업 폴더 하드코딩 의심 {len(hits)}건 (값 마스킹)"); [print("  ", h) for h in hits[:20]]
    if os.path.isdir(".git"):
        try:
            out = subprocess.run(["git", "log", "--all", "-p", "-G", r"api_key|apikey|OPENALEX|secret|token", "--", "."], capture_output=True, text=True, timeout=300, errors="ignore").stdout[:60_000_000]
            gh = set()
            for line in out.splitlines():
                if line.startswith("+"):
                    for pt in pats:
                        m = pt.search(line)
                        if m: gh.add(mask(m.group(m.lastindex)))
            print(f"[H-2] git 이력에서 키 형태 문자열 {len(gh)}종 발견(마스킹):", sorted(gh)[:10])
            if gh: print("  조치: (1) 해당 키를 먼저 폐기·재발급 (2) 이력 정리(git filter-repo 등)는 그 다음. 이미 push했다면 이력 정리와 무관하게 유출로 간주")
        except Exception as e: print("[H-2] git 스캔 실패:", type(e).__name__, str(e)[:120])
    else: print("[H-2] .git 폴더 없음: 이력 스캔 생략(다른 위치에 저장소가 있으면 그 폴더에서 실행)")
    gi = open(".gitignore", encoding="utf-8").read() if os.path.exists(".gitignore") else ""
    print("[H-3] .gitignore에 .env 포함:", ".env" in gi, "| oa_out 포함:", "oa_out" in gi, "(대용량 중간 파일과 캐시는 제외 권장)")
    k = os.environ.get("OPENALEX_API_KEY")
    if k:
        try:
            r = requests.get("https://api.openalex.org/works", params={"per-page": 1, "api_key": k}, timeout=30); print("[H-4] 현재 환경변수 키 작동 확인 HTTP", r.status_code, "(키 값은 출력하지 않음)")
        except Exception as e: print("[H-4] 호출 실패:", type(e).__name__)
    else: print("[H-4] OPENALEX_API_KEY 환경변수 없음")
    print("  옛 키가 폐기되었는지는 코드로 확인할 수 없습니다. OpenAlex 계정 페이지에서 키 목록과 사용량을 직접 확인하세요.")
safe(H)
print(f"\n[완료] 표와 초안은 {FIN}/ 에 저장. 붙여 주실 것: [A-1], [B-1~3], [C-1~2], [D-0~3], [E], [F], [H-1~4]")

In [ ]:
#

In [ ]:
import pandas as pd, re
D = pd.read_csv(f"{FIN}/B_japan_company_inst_vs_sample.csv")
O0 = (D[~D.표본내].groupby(["inst","기관"]).논문.sum().sort_values(ascending=False).reset_index())

EXCL = r"Daihatsu|Soken|SOKEN|Toyota|Japan Broadcasting|Samsung|Mizuho|Spring-8|Chemical Analysis|Qualtec|KRI, Inc|Asahi Glass|AGC|TEPCO|Tokyo Electric|Nippon Steel|Tokyo Gas|Nissan|Honda|Mitsubishi Electric|Idemitsu"
GROUP = {"Panasonic":"Panasonic群","Sanyo":"Panasonic群","Hitachi Maxell":"Panasonic群",
         "Showa Denko":"Showa Denko群","Mitsubishi Chemical":"MChem群","AGC Seimi":"AGC群"}
def grp(n):
    for k,v in GROUP.items():
        if k in n: return v
    return re.sub(r" \(Japan\)","",n)

O0["제외"] = O0.기관.str.contains(EXCL, regex=True)
# 코호트 기업명과 대조 (FM.corp_ja에 영문/약칭이 안 맞을 수 있으니 수동 확인용 열)
O0["코호트일본어명_후보"] = ""
O0["기업군"] = O0.기관.map(grp)
OUTS = O0[~O0.제외].copy()
G = OUTS.groupby("기업군").논문.sum().sort_values(ascending=False)
print("제외된 기관:", O0[O0.제외].기관.tolist()[:25])
print("정리 후 기업군 상위 20:"); print(G.head(20).to_string())
print("기업군 수:", len(G), "| 상위 1군 비중:", round(G.iloc[0]/G.sum(),3))
OUTS.to_csv(f"{FIN}/outside_firms_final.csv", index=False, encoding="utf-8-sig")

In [ ]:
#

In [ ]:
import os, re, json, time, warnings
import numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf, requests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
MAILTO = "<REDACTED_EMAIL>"; OA_KEY = os.environ.get("OPENALEX_API_KEY", "")
OA_KEY = "<REDACTED_API_KEY>"

TOPG = 40; MAXPAGES = 60
FIN = f"{OUT}/final"; os.makedirs(FIN, exist_ok=True); CACHE = f"{OUT}/outside_works_cache_v2.jsonl"
short = lambda v: str(v).split("/")[-1] if v else None
tid = lambda t: str(t) if re.match(r"^T\d+$", str(t)) else ("T" + str(t) if re.match(r"^\d+$", str(t)) else None)

def oa_get(path, params, retries=4):
    p = dict(params); k = os.environ.get("OPENALEX_API_KEY"); m = os.environ.get("OPENALEX_MAILTO")
    if k: p["api_key"] = k
    if m: p["mailto"] = m
    for i in range(retries):
        try:
            r = requests.get("https://api.openalex.org/" + path, params=p, timeout=60)
            if r.status_code == 200: return r.json()
            if r.status_code in (429, 500, 502, 503): time.sleep(2 ** i); continue
            print("  HTTP", r.status_code, r.text[:150]); return None
        except Exception: time.sleep(2 ** i)
    return None

# ---------- 0. 외부 기업군 ----------
D = pd.read_csv(f"{FIN}/B_japan_company_inst_vs_sample.csv")
O0 = D[~D.표본내].groupby(["inst", "기관"]).논문.sum().sort_values(ascending=False).reset_index()
EXCL = (r"Daihatsu|Soken|SOKEN|Toyota|Japan Broadcasting|Samsung|Mizuho|Spring-8|Chemical Analysis|Qualtec|KRI, Inc|"
        r"Asahi Glass|AGC|TEPCO|Tokyo Electric|Nippon Steel|Tokyo Gas|Nissan Motor|Nissan \(Japan\)|Honda|Mitsubishi Electric|Idemitsu")
GROUP = {"Panasonic": "Panasonic群", "Sanyo": "Panasonic群", "Hitachi Maxell": "Panasonic群", "Showa Denko": "Showa Denko群", "Mitsubishi Chemical": "MChem群"}
def grp(n):
    for k, v in GROUP.items():
        if k in n: return v
    return re.sub(r" \(Japan\)", "", n)
O0["기업군"] = O0.기관.map(grp); O0 = O0[~O0.기관.str.contains(EXCL, regex=True)]
G = O0.groupby("기업군").논문.sum().sort_values(ascending=False); TOPGROUPS = list(G.index[:TOPG])
IDS = {g: set(O0[O0.기업군 == g].inst) for g in TOPGROUPS}; INST2G = {i: g for g, s in IDS.items() for i in s}
print("[0] 외부 기업군", len(TOPGROUPS), "| 기관 ID", len(INST2G), "| 상위:", TOPGROUPS[:8])

# ---------- 1. 수집 (같은 주제·기간) ----------
topics = sorted({tid(t) for t in U.topic_id.dropna().unique()} - {None})
TCLS = {tid(t): c for t, c in U.dropna(subset=["topic_id"]).drop_duplicates("topic_id")[["topic_id", "cls_f"]].itertuples(index=False)}
seen = {}
if os.path.exists(CACHE):
    for line in open(CACHE, encoding="utf-8"):
        try: r = json.loads(line); seen[r["id"]] = r
        except Exception: pass
print("[1] 캐시 논문", len(seen))
def fetch():
    ids = sorted(INST2G); ig = [ids[i:i + 25] for i in range(0, len(ids), 25)]; tg = [topics[i:i + 25] for i in range(0, len(topics), 25)]
    with open(CACHE, "a", encoding="utf-8") as fh:
        for a in ig:
            for b in tg:
                flt = f"authorships.institutions.id:{'|'.join(a)},primary_topic.id:{'|'.join(b)},publication_year:{YMIN}-{YLAST}"
                cur, pages = "*", 0
                while cur and pages < MAXPAGES:
                    js = oa_get("works", {"filter": flt, "per-page": 200, "cursor": cur, "select": "id,publication_year,authorships,primary_topic"})
                    if not js: break
                    for r in js.get("results", []):
                        wid = short(r["id"])
                        if wid in seen: continue
                        au = [[short((x.get("author") or {}).get("id")),
                               [short(i["id"]) for i in (x.get("institutions") or []) if i.get("id")],
                               [i.get("type") for i in (x.get("institutions") or []) if i.get("id")],
                               [i.get("country_code") for i in (x.get("institutions") or []) if i.get("id")]] for x in (r.get("authorships") or [])]
                        rec = dict(id=wid, year=r["publication_year"], topic=tid(short((r.get("primary_topic") or {}).get("id"))), au=au)
                        seen[wid] = rec; fh.write(json.dumps(rec) + "\n")
                    cur = js["meta"].get("next_cursor"); pages += 1
                print(f"  청크 완료 (누적 논문 {len(seen)})")
if len(seen) < 100: fetch()

# ---------- 2. 기업군-논문 표 ----------
rows = []
for w, r in seen.items():
    if r["topic"] not in TCLS or r["year"] > YLAST or not r["au"]: continue
    groups = {INST2G[i] for a in r["au"] for i in a[1] if i in INST2G}
    for g in groups:
        own = IDS[g]
        others = {i for a in r["au"] for i in a[1] if i not in own}
        foruniv = any(t == "education" and c != "JP" for a in r["au"] for i, t, c in zip(a[1], a[2], a[3]) if i not in own)
        last, first = r["au"][-1], r["au"][0]
        rows.append(dict(w=w, firm=g, year=r["year"], blk=(r["year"] - YMIN) // BLK, topic=r["topic"], cls=TCLS[r["topic"]],
                         solo=int(len(others) == 0), foruniv=int(foruniv),
                         last_ext=int(not (set(last[1]) & own)), first_ext=int(not (set(first[1]) & own)),
                         partners=others, authors=[a[0] for a in r["au"] if a[0] and (set(a[1]) & own)]))
O = pd.DataFrame(rows).reset_index(drop=True)
print("[2] 외부 기업군-논문", len(O), "| 기업군", O.firm.nunique(), "| core", int((O.cls == "core").sum()))
O.drop(columns=["partners", "authors"]).to_csv(f"{FIN}/outside_firm_works_v2.csv", index=False, encoding="utf-8-sig")
Ud = U[U.year <= YLAST]

def analyze(O, tag):
    print(f"\n==================== {tag} (기업군 {O.firm.nunique()}, 논문 {len(O)}) ====================")
    # 3 단독 비중
    c = Ud.groupby("blk").solo.mean().reindex(BL); o = O.groupby("blk").solo.mean().reindex(BL)
    T = pd.DataFrame({"코호트": c.values, "외부": o.values}, index=[LAB[b] for b in BL]); T.loc["변화"] = T.iloc[-1] - T.iloc[0]
    print("[3] 단독 비중"); print(T.round(3).to_string())
    O2 = O[O.blk.isin([b0, b1])].assign(post=lambda d: (d.blk == b1).astype(int))
    m = smf.ols("solo ~ post + C(firm)", data=O2).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(O2.firm)[0]})
    print(f"  기업군 FE: 끝 블록 {100*m.params['post']:+.1f}pp (se {100*m.bse['post']:.1f}, p={m.pvalues['post']:.3f}) | 코호트 비core 약 -13pp")
    # 4 신규 기업 저자 중 단독 참여
    rec = [(r.firm, a, r.blk, r.solo) for r in O.itertuples(index=False) for a in set(r.authors)]
    R = pd.DataFrame(rec, columns=["firm", "aid", "blk", "solo"]).drop_duplicates()
    fb = R.groupby(["firm", "aid"]).blk.min().rename("fb").reset_index(); R = R.merge(fb, on=["firm", "aid"])
    N = R[(R.blk == R.fb) & (R.blk > b0)].groupby(["firm", "aid", "blk"]).solo.max().reset_index()
    g = N.groupby("blk").solo.agg(["size", "mean"]); g.index = [LAB[b] for b in g.index]
    print("[4] 신규 기업 저자 중 단독 참여 (코호트 56.0→55.4→43.5%)"); print(g.round(3).to_string())
    N["late"] = (N.blk == b1).astype(int); Nn = N[N.blk >= BL[1]]
    try:
        m = smf.ols("solo ~ late + C(firm)", data=Nn).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(Nn.firm)[0]})
        print(f"  기업군 FE: 마지막 블록 {100*m.params['late']:+.1f}pp (p={m.pvalues['late']:.3f}) | 코호트 -11.5pp")
    except Exception as e: print("  FE 실패", type(e).__name__)
    # 5 저자 정의 의존성
    F = O[O.foruniv == 1]; rows = []
    for nm, col in [("마지막 저자 외부", "last_ext"), ("제1저자 외부", "first_ext")]:
        a = F.groupby("blk")[col].mean().reindex([b0, b1]); rows.append(dict(기준=nm, 첫=a[b0], 끝=a[b1], 변화pp=100 * (a[b1] - a[b0]), N=len(F)))
    print("[5] 해외 대학 포함 논문의 외부 주도 비중 (코호트: 마지막 +17~21pp, 제1 +5~8pp)"); print(pd.DataFrame(rows).round(3).to_string(index=False))
    print("  해외 대학 포함 논문 비중:", O.groupby("blk").foruniv.mean().reindex(BL).round(3).to_dict())
    # 6 core 집중
    Oc = O[O.cls == "core"]; rows = []
    for b in BL:
        cnt = Oc[Oc.blk == b].groupby("firm").size() / NY[b]; s = cnt / cnt.sum() if cnt.sum() else cnt
        rows.append(dict(블록=LAB[b], core편수년=cnt.sum(), 군수=len(cnt), 상위1점유=s.max() if len(s) else np.nan, HHI=(s ** 2).sum() if len(s) else np.nan, 유효수=1 / (s ** 2).sum() if len(s) else np.nan))
    print("[6] 외부 core 집중 (코호트 HHI 0.124→0.322; 방향만 비교)"); print(pd.DataFrame(rows).round(3).to_string(index=False))
    # 7 신규 파트너 -> 진입
    O = O.sort_values(["firm", "year"]).reset_index(drop=True); O["nw_any"] = 0
    for firm, gg in O.groupby("firm"):
        hist = set()
        for b in sorted(gg.blk.unique()):
            ix = gg.index[gg.blk == b]
            if b > b0:
                for i in ix: O.at[i, "nw_any"] = int(len(O.at[i, "partners"] - hist) > 0)
            for i in ix: hist |= O.at[i, "partners"]
    ft = O.groupby(["firm", "topic"]).blk.min().to_dict(); O["entry"] = [int(ft[(f, t)] == b and b > b0) for f, t, b in zip(O.firm, O.topic, O.blk)]
    Q = O[O.blk > b0].copy(); Q["ln_n"] = np.log(Q.groupby(["firm", "blk"]).w.transform("size"))
    X = sm.add_constant(pd.concat([Q[["nw_any", "ln_n"]], pd.get_dummies(Q.blk, prefix="b", drop_first=True)], axis=1).astype(float))
    mm = sm.Logit(Q.entry.astype(float).values, X.values).fit(disp=0, maxiter=200, cov_type="cluster", cov_kwds={"groups": pd.factorize(Q.firm)[0]})
    b_, s_ = mm.params[1], mm.bse[1]
    print(f"[7] 신규 파트너→진입 OR {np.exp(b_):.2f} [{np.exp(b_-1.96*s_):.2f}, {np.exp(b_+1.96*s_):.2f}] p={mm.pvalues[1]:.3f} | 논문 {len(Q)}, 진입 {int(Q.entry.sum())} (코호트 v1 2.18, v2 1.80, v3 2.33; 정의가 완전히 같지는 않음)")

analyze(O, "외부 기업군 전체")
analyze(O[O.firm != "Panasonic群"], "파나소닉 군 제외 민감도")

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import os, re, json, time, hashlib, datetime, warnings, traceback
import numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf, requests
from scipy import stats
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 200)

# ============================ 설정 ============================
NB = 500; SEED = 0; MAXPAGES = 60
RUN_FREEZE = True            # 가설 고정 파일 생성 (이미 있으면 덮어쓰지 않고 해시만 검증)
RUN_HOLDOUT_FETCH = True     # 보류 표본(외부 41~80위) 수집: OpenAlex 호출 발생
HOLD = (40, 80)              # 외부 기업군 순위 슬라이스 (0부터)
RUN_SPEC = True; RUN_FIG = True
VERIFY_SEEDS = True          # 시드 문헌 4편이 OpenAlex에 있는지 확인 (호출 4회)
RUN_LIT = False              # True면 주장별 문헌 후보 검색 (호출 약 10회)
RATER2_FILE = f"{OUT}/gx_review_rater2.csv"
THR4 = 0.198                 # 코호트 core HHI 변화 점추정 (사전 고정 값)
TOY, NIS = "トヨタ自動車", "日産自動車"
CLS5 = ["core", "enabling", "legacy", "non_gx", "other"]
FIN = f"{OUT}/final"; os.makedirs(FIN, exist_ok=True)
CACHE1 = f"{OUT}/outside_works_cache_v2.jsonl"          # 외부 1~40군 (코드 ⑰b 캐시)
CACHE2 = f"{OUT}/outside_works_cache_holdout.jsonl"     # 보류 표본 (41~80군)
PREREG = f"{FIN}/PREREG_holdout_v1.json"

need = ["U", "BL", "b0", "b1", "LAB", "NY", "YLAST", "YMIN", "BLK", "OUT", "classify", "PART"]
miss = [v for v in need if v not in globals()]; assert not miss, f"이전 셀 변수가 없습니다: {miss}"
assert "cls_f" in U.columns, "cls_f가 없습니다 (코드 ⑫ 0단계 먼저)"
Ud = U[U.year <= YLAST].copy()

def safe(f):
    try: f()
    except Exception as e:
        print(f"\n[{f.__name__} 실패] {type(e).__name__}: {str(e)[:250]}"); traceback.print_exc(limit=2)
def save(df, name): df.to_csv(f"{FIN}/{name}.csv", index=False, encoding="utf-8-sig")
short = lambda v: str(v).split("/")[-1] if v else None
tid = lambda t: str(t) if re.match(r"^T\d+$", str(t)) else ("T" + str(t) if re.match(r"^\d+$", str(t)) else None)
def oa_get(path, params, retries=4):
    p = dict(params); k = os.environ.get("OPENALEX_API_KEY"); m = os.environ.get("OPENALEX_MAILTO")
    if k: p["api_key"] = k
    if m: p["mailto"] = m
    for i in range(retries):
        try:
            r = requests.get("https://api.openalex.org/" + path, params=p, timeout=60)
            if r.status_code == 200: return r.json()
            if r.status_code in (429, 500, 502, 503): time.sleep(2 ** i); continue
            print("  HTTP", r.status_code, r.text[:150]); return None
        except Exception: time.sleep(2 ** i)
    return None
def ols_cl(formula, d, key="firm"):
    d = d.reset_index(drop=True); mod = smf.ols(formula, data=d, missing="drop")
    return mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.loc[mod.data.row_labels, key])[0]})

# ---------- 외부 기업군 목록 (코드 ⑰b와 같은 규칙) ----------
EXCL = (r"Daihatsu|Soken|SOKEN|Toyota|Japan Broadcasting|Samsung|Mizuho|Spring-8|Chemical Analysis|Qualtec|KRI, Inc|"
        r"Asahi Glass|AGC|TEPCO|Tokyo Electric|Nippon Steel|Tokyo Gas|Nissan Motor|Nissan \(Japan\)|Honda|Mitsubishi Electric|Idemitsu")
GROUP = {"Panasonic": "Panasonic群", "Sanyo": "Panasonic群", "Hitachi Maxell": "Panasonic群", "Showa Denko": "Showa Denko群", "Mitsubishi Chemical": "MChem群"}
def grp(n):
    for k, v in GROUP.items():
        if k in n: return v
    return re.sub(r" \(Japan\)", "", n)
def ranked_groups():
    D = pd.read_csv(f"{FIN}/B_japan_company_inst_vs_sample.csv")
    O0 = D[~D.표본내].groupby(["inst", "기관"]).논문.sum().sort_values(ascending=False).reset_index()
    O0["기업군"] = O0.기관.map(grp); O0 = O0[~O0.기관.str.contains(EXCL, regex=True)]
    G = O0.groupby("기업군").논문.sum().sort_values(ascending=False)
    return O0, G
O0, G = ranked_groups()
GROUPS_EXT1 = list(G.index[:HOLD[0]]); GROUPS_HOLD = list(G.index[HOLD[0]:HOLD[1]])
print(f"[준비] 외부 기업군 순위: 1~{HOLD[0]}군 {len(GROUPS_EXT1)}개, 보류 {HOLD[0]+1}~{HOLD[1]}군 {len(GROUPS_HOLD)}개 | 보류 논문 수 범위 {int(G.loc[GROUPS_HOLD].min())}~{int(G.loc[GROUPS_HOLD].max())}")

topics = sorted({tid(t) for t in U.topic_id.dropna().unique()} - {None})
TCLS = {tid(t): c for t, c in U.dropna(subset=["topic_id"]).drop_duplicates("topic_id")[["topic_id", "cls_f"]].itertuples(index=False)}

# ============================ B-1. 가설 고정 (FREEZE) ============================
HYP = {
 "H1": dict(text="보류 표본에서 신규 기업 저자 중 단독 참여 비중은 마지막 블록이 이전 두 블록보다 낮다", stat="기업 FE OLS(기업 군집)의 late 계수(pp)", rule="95% 구간 상한 < 0"),
 "H2": dict(text="해외 대학 포함 논문에서 외부(비기업) 마지막 저자 비중의 첫→마지막 블록 증가가 제1저자 기준 증가보다 크다", stat="기업 재표집 부트스트랩 95% 구간", rule="마지막 저자 변화 구간 하한 > 0 이고 (마지막-제1) 차이 구간 하한 > 0"),
 "H3": dict(text="신규 파트너 포함 논문의 새 주제 진입 오즈가 더 높다", stat="로짓 OR(블록 더미, ln 논문수 통제, 기업 군집), 진입=기업군x주제 첫 등장 블록의 논문", rule="95% 구간 하한 > 1"),
 "H4": dict(text="보류 표본의 core 논문 집중은 코호트만큼 상승하지 않는다", stat="HHI(마지막-첫 블록) 변화, 기업 재표집 95% 구간", rule=f"95% 구간 상한 < {THR4} (코호트 점추정)"),
 "H5": dict(text="해외 대학 포함 논문 비중은 첫→마지막 블록에서 증가한다", stat="기업 FE LPM(기업 군집) post 계수", rule="95% 구간 하한 > 0"),
}
def FREEZE():
    if os.path.exists(PREREG):
        d = json.load(open(PREREG, encoding="utf-8")); body = json.dumps(d["hypotheses"], ensure_ascii=False, sort_keys=True)
        ok = hashlib.sha256(body.encode()).hexdigest() == d["sha256"]
        print(f"[FREEZE] 이미 고정됨 ({d['created']}). 가설 해시 검증: {'일치' if ok else '불일치(가설이 수정됨!)'} | 고정 시점 보류 캐시 존재: {d['holdout_cache_existed_at_freeze']}")
        return
    existed = os.path.exists(CACHE2) and os.path.getsize(CACHE2) > 0
    body = json.dumps(HYP, ensure_ascii=False, sort_keys=True)
    rec = dict(created=datetime.datetime.now().isoformat(timespec="seconds"), hypotheses=HYP, thresholds=dict(THR4=THR4), holdout_groups=GROUPS_HOLD,
               params=dict(NB=NB, SEED=SEED, BLK=BLK, YMIN=YMIN, YLAST=YLAST), holdout_cache_existed_at_freeze=existed, sha256=hashlib.sha256(body.encode()).hexdigest(),
               note="코호트와 외부 1~40군 결과를 본 뒤 세운 가설이므로 맹검 확증이 아니라 보지 않은 표본에서의 재현 검정이다.")
    json.dump(rec, open(PREREG, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print(f"[FREEZE] 저장: {PREREG}", "| 경고: 보류 캐시가 이미 있어 '수집 전 고정'이 아님" if existed else "| 수집 전 고정 확인")
    print("  이 파일을 OSF에 올리거나 git 커밋(날짜가 남는 곳)으로 보관하세요. 결과를 본 뒤 가설을 바꾸면 보류 표본의 의미가 사라집니다.")
if RUN_FREEZE: safe(FREEZE)

# ============================ 공통: 표본 구축 ============================
def build_cohort():
    rows = []
    for r in Ud.itertuples(index=False):
        cl = classify(r.w, r.corp_norm); auth = [c[0] for c in cl if c[4] and c[0]] if cl else []
        rows.append(dict(w=r.w, firm=r.corp_norm, year=int(r.year), blk=int(r.blk), topic=str(r.topic_id), cls=r.cls_f, solo=int(r.solo),
                         foruniv=int(bool(r.for_univ)), last_ext=int(not bool(r.last_firm)), first_ext=int(not bool(r.first_firm)),
                         partners=set(PART.get((r.w, r.corp_norm), set())), authors=auth))
    return pd.DataFrame(rows)
def load_cache(path):
    seen = {}
    if os.path.exists(path):
        for line in open(path, encoding="utf-8"):
            try: r = json.loads(line); seen[r["id"]] = r
            except Exception: pass
    return seen
def group_maps(groups):
    IDS = {g: set(O0[O0.기업군 == g].inst) for g in groups}; return IDS, {i: g for g, s in IDS.items() for i in s}
def fetch(INST2G, cache, seen):
    ids = sorted(INST2G); ig = [ids[i:i + 25] for i in range(0, len(ids), 25)]; tg = [topics[i:i + 25] for i in range(0, len(topics), 25)]
    with open(cache, "a", encoding="utf-8") as fh:
        for a in ig:
            for b in tg:
                flt = f"authorships.institutions.id:{'|'.join(a)},primary_topic.id:{'|'.join(b)},publication_year:{YMIN}-{YLAST}"; cur, pages = "*", 0
                while cur and pages < MAXPAGES:
                    js = oa_get("works", {"filter": flt, "per-page": 200, "cursor": cur, "select": "id,publication_year,authorships,primary_topic"})
                    if not js: break
                    for r in js.get("results", []):
                        wid = short(r["id"])
                        if wid in seen: continue
                        au = [[short((x.get("author") or {}).get("id")), [short(i["id"]) for i in (x.get("institutions") or []) if i.get("id")],
                               [i.get("type") for i in (x.get("institutions") or []) if i.get("id")], [i.get("country_code") for i in (x.get("institutions") or []) if i.get("id")]]
                              for x in (r.get("authorships") or [])]
                        rec = dict(id=wid, year=r["publication_year"], topic=tid(short((r.get("primary_topic") or {}).get("id"))), au=au); seen[wid] = rec; fh.write(json.dumps(rec) + "\n")
                    cur = js["meta"].get("next_cursor"); pages += 1
                print(f"  청크 완료 (누적 논문 {len(seen)})")
def build_O(seen, IDS, INST2G):
    rows = []
    for w, r in seen.items():
        if r["topic"] not in TCLS or r["year"] > YLAST or not r["au"]: continue
        for g in {INST2G[i] for a in r["au"] for i in a[1] if i in INST2G}:
            own = IDS[g]; others = {i for a in r["au"] for i in a[1] if i not in own}
            foruniv = any(t == "education" and c != "JP" for a in r["au"] for i, t, c in zip(a[1], a[2], a[3]) if i not in own)
            last, first = r["au"][-1], r["au"][0]
            rows.append(dict(w=w, firm=g, year=r["year"], blk=(r["year"] - YMIN) // BLK, topic=r["topic"], cls=TCLS[r["topic"]], solo=int(len(others) == 0), foruniv=int(foruniv),
                             last_ext=int(not (set(last[1]) & own)), first_ext=int(not (set(first[1]) & own)), partners=others, authors=[a[0] for a in r["au"] if a[0] and (set(a[1]) & own)]))
    return pd.DataFrame(rows).reset_index(drop=True)
def add_entry_nw(df):
    df = df.sort_values(["firm", "year", "w"]).reset_index(drop=True).copy(); nw = np.zeros(len(df), dtype=int)
    for firm, g in df.groupby("firm"):
        hist = set()
        for b in sorted(g.blk.unique()):
            idx = g.index[g.blk == b]
            if b > b0:
                for i in idx: nw[i] = int(len(df.partners.iat[i] - hist) > 0)
            for i in idx: hist |= df.partners.iat[i]
    df["nw_any"] = nw; ft = df.groupby(["firm", "topic"]).blk.min().to_dict()
    df["entry"] = [int(ft[(f, t)] == b and b > b0) for f, t, b in zip(df.firm, df.topic, df.blk)]
    df["ln_n"] = np.log(df.groupby(["firm", "blk"]).w.transform("size")); df["rk"] = df.groupby(["firm", "topic", "blk"]).cumcount()
    df["entry_v3"] = ((df.entry == 1) & (df.rk == 0)).astype(int); return df

SAMPLES = {}
def make_samples():
    SAMPLES["코호트"] = add_entry_nw(build_cohort())
    I1, M1 = group_maps(GROUPS_EXT1); s1 = load_cache(CACHE1)
    if len(s1) < 100: print("[경고] 외부 1~40군 캐시가 비었습니다. 수집을 시도합니다"); fetch(M1, CACHE1, s1)
    e1 = add_entry_nw(build_O(s1, I1, M1)); SAMPLES["외부1~40"] = e1; SAMPLES["외부1~40(덴소 제외)"] = add_entry_nw(e1[e1.firm != "Denso"].drop(columns=["nw_any", "entry", "ln_n", "rk", "entry_v3"]))
    I2, M2 = group_maps(GROUPS_HOLD); s2 = load_cache(CACHE2)
    if RUN_HOLDOUT_FETCH and len(s2) < 100:
        if not os.path.exists(PREREG): print("[중단] PREREG 파일이 없습니다. FREEZE를 먼저 실행하세요 (보류 표본 수집 전 가설 고정)"); return
        fetch(M2, CACHE2, s2)
    if len(s2) >= 1: SAMPLES["보류41~80"] = add_entry_nw(build_O(s2, I2, M2))
    print("[표본]", {k: (len(v), v.firm.nunique()) for k, v in SAMPLES.items()}, "(논문x기업군 수, 기업군 수)")
safe(make_samples)

# ============================ 추정량 H1~H5 ============================
def df_authors(df):
    rec = [(r.firm, a, r.blk, r.solo) for r in df.itertuples(index=False) for a in set(r.authors)]
    return pd.DataFrame(rec, columns=["firm", "aid", "blk", "solo"]).groupby(["firm", "aid", "blk"], as_index=False).solo.max()
def new_author_frame(df):
    A = df_authors(df); fb = A.groupby(["firm", "aid"]).blk.min().rename("fb").reset_index(); A = A.merge(fb, on=["firm", "aid"])
    return A[(A.blk == A.fb) & (A.blk > b0)].copy()
def est_H1(df, ref=None, fe=True, drop_top=False):
    N = new_author_frame(df)
    if drop_top and len(N): N = N[N.firm != N.groupby("firm").size().idxmax()]
    ref = list(ref) if ref is not None else [x for x in BL[1:] if x != b1]
    Nn = N[N.blk.isin(ref + [b1])].copy(); Nn["late"] = (Nn.blk == b1).astype(int)
    m = ols_cl("solo ~ late" + (" + C(firm)" if fe else ""), Nn); b, s = m.params["late"], m.bse["late"]
    return dict(est=100 * b, lo=100 * (b - 1.96 * s), hi=100 * (b + 1.96 * s), p=m.pvalues["late"], n=int(m.nobs))
def est_H2(df):
    F = df[df.foruniv == 1]
    def stat(X):
        g = X.groupby("blk")[["last_ext", "first_ext"]].mean()
        if b0 not in g.index or b1 not in g.index: return (np.nan, np.nan, np.nan)
        dl = g.loc[b1, "last_ext"] - g.loc[b0, "last_ext"]; dfi = g.loc[b1, "first_ext"] - g.loc[b0, "first_ext"]; return dl, dfi, dl - dfi
    pt = stat(F); rng = np.random.default_rng(SEED); grp_ = {f: x for f, x in F.groupby("firm")}; firms = list(grp_); bs = []
    for _ in range(NB):
        pick = rng.choice(len(firms), len(firms)); t = stat(pd.concat([grp_[firms[i]] for i in pick]))
        if not np.isnan(t[0]): bs.append(t)
    bs = np.array(bs); lo_l, hi_l = np.percentile(bs[:, 0], [2.5, 97.5]); lo_d, hi_d = np.percentile(bs[:, 2], [2.5, 97.5])
    return dict(est=100 * pt[0], lo=100 * lo_l, hi=100 * hi_l, lo_last=100 * lo_l, lo_diff=100 * lo_d, hi_diff=100 * hi_d, est_first=100 * pt[1], est_diff=100 * pt[2],
                p=(1 + (bs[:, 2] <= 0).sum()) / (1 + len(bs)), n=len(F))
def logit_or(d, y, covs):
    X = sm.add_constant(pd.concat([d[["nw_any"] + covs], pd.get_dummies(d.blk, prefix="b", drop_first=True)], axis=1).astype(float))
    m = sm.Logit(d[y].astype(float).values, X.values).fit(disp=0, maxiter=200, cov_type="cluster", cov_kwds={"groups": pd.factorize(d.firm)[0]})
    j = list(X.columns).index("nw_any"); b, s = m.params[j], m.bse[j]
    return dict(est=np.exp(b), lo=np.exp(b - 1.96 * s), hi=np.exp(b + 1.96 * s), p=m.pvalues[j], logb=b, se=s, n=len(d), events=int(d[y].sum()))
def est_H3(df): return logit_or(df[df.blk > b0], "entry", ["ln_n"])
def hhi_vec(x): c = x[x > 0]; s = c / c.sum(); return float((s ** 2).sum())
def core_matrix(df): return df[df.cls == "core"].groupby(["firm", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0)
def est_H4(df):
    M = core_matrix(df).values.astype(float); ch = lambda X: hhi_vec(X[:, -1]) - hhi_vec(X[:, 0]) if X[:, 0].sum() > 0 and X[:, -1].sum() > 0 else np.nan
    pt = ch(M); rng = np.random.default_rng(SEED); bs = np.array([v for v in (ch(M[rng.integers(0, len(M), len(M))]) for _ in range(NB)) if not np.isnan(v)])
    return dict(est=pt, lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5), p=(1 + (bs >= THR4).sum()) / (1 + len(bs)), n=len(M), hhi_first=hhi_vec(M[:, 0]), hhi_last=hhi_vec(M[:, -1]))
def est_H5(df):
    d = df[df.blk.isin([b0, b1])].copy(); d["post"] = (d.blk == b1).astype(int); m = ols_cl("foruniv ~ post + C(firm)", d); b, s = m.params["post"], m.bse["post"]
    return dict(est=100 * b, lo=100 * (b - 1.96 * s), hi=100 * (b + 1.96 * s), p=m.pvalues["post"], n=int(m.nobs))
EST = {"H1": est_H1, "H2": est_H2, "H3": est_H3, "H4": est_H4, "H5": est_H5}
RULE = {"H1": lambda r: r["hi"] < 0, "H2": lambda r: r["lo_last"] > 0 and r["lo_diff"] > 0, "H3": lambda r: r["lo"] > 1, "H4": lambda r: r["hi"] < THR4, "H5": lambda r: r["lo"] > 0}
RES = {}

# ============================ B-2. 검정 실행 (HOLDOUT) ============================
def HOLDOUT():
    rows = []
    for name, df in SAMPLES.items():
        for H, fn in EST.items():
            try: r = fn(df); r["지지"] = bool(RULE[H](r))
            except Exception as e: r = dict(err=f"{type(e).__name__}: {str(e)[:80]}", 지지=None)
            RES[(H, name)] = r
            rows.append(dict(가설=H, 표본=name, 추정=r.get("est", np.nan), lo=r.get("lo", np.nan), hi=r.get("hi", np.nan), p=r.get("p", np.nan), 지지=r["지지"], 비고=r.get("err", "")))
    T = pd.DataFrame(rows); print("[HOLDOUT-1] H1~H5 표본별 결과 (H1·H2·H5는 pp, H3는 OR, H4는 HHI 변화)"); print(T.round(3).to_string(index=False)); save(T, "B_holdout_results")
    h = T[(T.표본 == "보류41~80") & T.p.notna()].copy()
    if len(h):
        h["p_holm_보류5개"] = multipletests(h.p.values, method="holm")[1]
        print("\n[HOLDOUT-2] 확증 가설군 = 보류 표본의 H1~H5 (Holm). H2·H4의 p는 부트스트랩 단측 p (H4는 'HHI 변화 ≥ 0.198'에 대한 p)"); print(h[["가설", "추정", "lo", "hi", "p", "p_holm_보류5개", "지지"]].round(3).to_string(index=False)); save(h, "B_holdout_confirmatory_holm")
    print("\n  읽기: 보류 표본에서 지지가 True인 가설만 본문 '재현 확인'으로, False 또는 계산 실패는 '가려지지 않음/비재현'으로 그대로 보고하세요. 결과를 보고 기준을 바꾸지 마세요.")
safe(HOLDOUT)

# ============================ B-3. 사양 곡선 (SPEC) ============================
def SPEC():
    rows = []
    for sname, df in [(k, v) for k, v in SAMPLES.items() if k != "외부1~40(덴소 제외)"]:
        for refname, ref in [("이전 두 블록", [x for x in BL[1:] if x != b1]), ("직전 블록", [BL[-2]]), ("2011-15만", [BL[1]])]:
            for fe in [True, False]:
                for dt in [False, True]:
                    try: r = est_H1(df, ref=ref, fe=fe, drop_top=dt); rows.append(dict(결과="H1 신규 단독 참여(pp)", 표본=sname, 기준=refname, FE=fe, 최대기업제외=dt, est=r["est"], lo=r["lo"], hi=r["hi"]))
                    except Exception: pass
        for dn in ["entry", "entry_v3"]:
            for dt in [False, True]:
                for covs in [[], ["ln_n"]]:
                    try:
                        d = df[df.blk > b0]
                        if dn == "entry_v3": d = d[(d.entry == 0) | (d.entry_v3 == 1)]
                        if dt: d = d[d.firm != d.groupby("firm").size().idxmax()]
                        r = logit_or(d, dn, covs); rows.append(dict(결과="H3 진입 로그OR", 표본=sname, 기준=dn, FE=False, 최대기업제외=dt, 통제="ln_n" if covs else "없음", est=np.log(r["est"]), lo=np.log(r["lo"]), hi=np.log(r["hi"])))
                    except Exception: pass
    S = pd.DataFrame(rows); save(S, "B_spec_curve")
    for k, g in S.groupby("결과"):
        print(f"[SPEC] {k}: 사양 {len(g)}개 | 부호 일관(전부 {'음' if k.startswith('H1') else '양'}) 비율 {((g.est < 0) if k.startswith('H1') else (g.est > 0)).mean():.2f} | 95% 구간이 0 제외 비율 {(((g.hi < 0) if k.startswith('H1') else (g.lo > 0))).mean():.2f} | 추정 범위 {g.est.min():.2f} ~ {g.est.max():.2f}")
    if RUN_FIG:
        try:
            import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
            fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
            for a, (k, g) in zip(ax, S.groupby("결과")):
                g = g.sort_values("est").reset_index(drop=True); a.vlines(g.index, g.lo, g.hi, color="lightgray", lw=1); a.scatter(g.index, g.est, s=12, c="k"); a.axhline(0, color="r", lw=0.8)
                a.set_title(f"{k}: specification curve (n={len(g)})", fontsize=10); a.set_xlabel("specifications (sorted)"); a.set_ylabel("estimate")
            plt.tight_layout(); plt.savefig(f"{FIN}/B_spec_curve.png", dpi=200); plt.close(); print("  그림 저장: B_spec_curve.png")
        except Exception as e: print("  그림 실패:", type(e).__name__)
if RUN_SPEC: safe(SPEC)

# ============================ A. 단독 저자 흐름 분해 (FLOW) ============================
FLOWRES = {}
def flow_table(df):
    A = df_authors(df); A["key"] = list(zip(A.firm, A.aid))
    act = {b: set(A.loc[A.blk == b, "key"]) for b in BL}; sol = {b: set(A.loc[(A.blk == b) & (A.solo == 1), "key"]) for b in BL}
    TERMS = ["유지(단독→단독)", "전환유입(협력→단독)", "복귀(전전 블록 활동)", "신규 유입", "전환이탈(단독→협력만)", "미관측 이탈"]; rows = []; per = []
    for t in BL[1:]:
        p = t - 1; before = set().union(*[act[x] for x in BL if x < t]); Sp, St = sol[p], sol[t]
        parts = {TERMS[0]: Sp & St, TERMS[1]: St & (act[p] - Sp), TERMS[2]: St & (before - act[p]), TERMS[3]: St - before, TERMS[4]: (Sp & act[t]) - St, TERMS[5]: Sp - act[t]}
        assert len(St) == sum(len(parts[k]) for k in TERMS[:4]) and len(Sp) == len(parts[TERMS[0]]) + len(parts[TERMS[4]]) + len(parts[TERMS[5]]), "항등식 위반"
        for k, v in parts.items():
            for firm, n in pd.Series([x[0] for x in v]).value_counts().items(): per.append((t, k, firm, n))
        r = dict(전환=f"{LAB[p]}→{LAB[t]}", 이전_단독저자=len(Sp), 이후_단독저자=len(St), 순변화=len(St) - len(Sp)); r.update({k: len(v) for k, v in parts.items()}); rows.append(r)
    T = pd.DataFrame(rows); P = pd.DataFrame(per, columns=["t", "term", "firm", "n"]); rng = np.random.default_rng(SEED); ci = []
    for t in BL[1:]:
        pv = P[P.t == t].pivot_table(index="firm", columns="term", values="n", aggfunc="sum", fill_value=0).reindex(columns=TERMS, fill_value=0)
        if len(pv) == 0: continue
        V = pv.values.astype(float); base = lambda X: X[:, [0, 4, 5]].sum()
        stat = lambda X: (X[:, 3].sum() / base(X), X[:, 4].sum() / base(X), X[:, 5].sum() / base(X)); bs = np.array([stat(V[rng.integers(0, len(V), len(V))]) for _ in range(NB)]); pt = stat(V)
        for j, nm in enumerate(["신규 유입/이전 단독저자", "전환이탈/이전 단독저자", "미관측 이탈/이전 단독저자"]):
            ci.append(dict(전환=f"{LAB[t-1]}→{LAB[t]}", 지표=nm, 점추정=pt[j], lo=np.percentile(bs[:, j], 2.5), hi=np.percentile(bs[:, j], 97.5)))
    return T, pd.DataFrame(ci)
def FLOW():
    for name in ["코호트", "외부1~40", "보류41~80"]:
        if name not in SAMPLES: continue
        T, CI = flow_table(SAMPLES[name]); FLOWRES[name] = (T, CI)
        print(f"\n[FLOW] {name}: 단독 참여 저자 수의 변화 = 신규 유입 + 전환유입 + 복귀 − 전환이탈 − 미관측 이탈 (저자 단위, 블록 길이가 달라 마지막 전환은 4년 기준)"); print(T.to_string(index=False))
        print(CI.round(3).to_string(index=False)); save(T, f"A_flow_{name}"); save(CI, f"A_flow_ci_{name}")
    print("\n  읽기: '신규 유입'이 음의 방향으로 줄어드는 만큼이 아니라, 이탈·전환이탈 비율이 코호트와 외부에서 비슷하고 신규 유입 항만 변하면 '충원 감소' 서술을 유지.")
    print("  주의: 미관측 이탈은 이직·주제 이탈·ID 분리를 모두 포함하고, 한 기업군 안의 다른 기관 이동도 포함됨.")
safe(FLOW)

# ============================ A. 그림 3장 (FIG) ============================
def FIG():
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    names = [n for n in ["코호트", "외부1~40", "보류41~80"] if n in SAMPLES]; lab = {"코호트": "Cohort", "외부1~40": "Non-cohort (1-40)", "보류41~80": "Holdout (41-80)"}
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
    for n in names:
        N = new_author_frame(SAMPLES[n]); g = N.groupby("blk").solo.mean().reindex(BL[1:]); ax[0].plot([LAB[b] for b in BL[1:]], g.values * 100, marker="o", label=lab[n])
    ax[0].set_title("Share of new corporate authors who co-author solo papers (%)", fontsize=10); ax[0].legend(fontsize=8); ax[0].tick_params(axis="x", labelsize=8)
    w = 0.25
    for k, n in enumerate(names):
        r = RES.get(("H2", n), {})
        if "est" in r: ax[1].bar(np.arange(2) + k * w, [r["est"], r["est_first"]], w, label=lab[n])
    ax[1].set_xticks(np.arange(2) + w); ax[1].set_xticklabels(["Last author\nexternal share", "First author\nexternal share"]); ax[1].axhline(0, color="k", lw=0.6)
    ax[1].set_title("Change in external-led share, foreign-university papers (pp)", fontsize=10); ax[1].legend(fontsize=8)
    for n in names:
        M = core_matrix(SAMPLES[n]).values.astype(float); ax[2].plot([LAB[b] for b in BL], [hhi_vec(M[:, k]) for k in range(len(BL))], marker="o", label=lab[n])
    ax[2].set_title("Concentration (HHI) of core GX papers across firms", fontsize=10); ax[2].legend(fontsize=8); ax[2].tick_params(axis="x", labelsize=8)
    plt.tight_layout(); plt.savefig(f"{FIN}/A_figures.png", dpi=200); plt.close(); print("[FIG] 저장: A_figures.png (개별 패널로 쪼개 투고용으로 다듬으세요)")
if RUN_FIG: safe(FIG)

# ============================ C. 문헌·기여 문장 (LIT) ============================
SEEDS = [("The decline of science in corporate R&D", "Arora, Belenzon, Patacconi", 2018), ("The increasing dominance of teams in production of knowledge", "Wuchty, Jones, Uzzi", 2007),
         ("The dual frontier: patented inventions and prior scientific advance", "Ahmadpoor, Jones", 2017), ("Large teams develop and small teams disrupt science and technology", "Wu, Wang, Evans", 2019)]
CLAIMS = pd.DataFrame({"주장": ["기업 과학 논문의 감소", "단독 연구 감소와 팀 규모 증가", "해외 대학 협력과 저자 위치 해석", "기업 논문 식별과 기관 매칭의 한계", "특허-논문 연결과 특허 기반 표본",
                              "새 파트너와 새 주제 진입", "연구 집중과 기업 과학 불평등", "일본 산학연 협력과 중앙연구소", "녹색기술 정의와 특허 분류", "저자 순서 규범"],
                       "검색 키워드": ["corporate science decline publications firms", "team size solo authorship decline collaboration", "corresponding author last author leadership international collaboration",
                                  "OpenAlex affiliation disambiguation company institutions", "patent paper linkage firm science", "new collaboration partners exploration topic entry firm",
                                  "concentration corporate research publications", "Japan industry university collaboration central research laboratory", "green technology patents classification",
                                  "author order norms credit contribution"]})
def fmt(H, n, k="est", nd=1):
    r = RES.get((H, n), {}); return "NA" if k not in r or pd.isna(r.get(k)) else f"{r[k]:.{nd}f}"
def LIT():
    if VERIFY_SEEDS:
        rows = []
        for title, au, yr in SEEDS:
            js = oa_get("works", {"search": title, "per-page": 3, "select": "id,title,publication_year,doi,primary_location,cited_by_count"})
            for r in (js or {}).get("results", [])[:2]:
                rows.append(dict(찾는_제목=title, 저자_제기억=au, 연도_제기억=yr, 검색_제목=r.get("title"), 연도=r.get("publication_year"), 저널=((r.get("primary_location") or {}).get("source") or {}).get("display_name"), doi=r.get("doi"), 인용=r.get("cited_by_count")))
        V = pd.DataFrame(rows); print("[LIT-1] 시드 문헌 존재 확인 (제 기억의 서지이므로 제목·연도·저널을 직접 대조하세요)"); print(V.to_string(index=False)); save(V, "C_seed_verification")
    M = CLAIMS.copy(); M["읽은 문헌(저자·연도·저널)"] = "[문헌 확인]"; M["이 연구와의 차이·연결"] = "[작성]"; M["인용 확정"] = False
    if RUN_LIT:
        cands = []
        for k, q in enumerate(CLAIMS["검색 키워드"]):
            js = oa_get("works", {"search": q, "filter": "publication_year:>2004,type:article", "per-page": 15, "select": "id,title,publication_year,doi,cited_by_count,primary_location"})
            for r in (js or {}).get("results", []): cands.append(dict(주장번호=k, 제목=r.get("title"), 연도=r.get("publication_year"), 인용=r.get("cited_by_count"), 저널=((r.get("primary_location") or {}).get("source") or {}).get("display_name"), doi=r.get("doi")))
        save(pd.DataFrame(cands).drop_duplicates("doi"), "C_candidates"); print("[LIT-2] 후보 저장: C_candidates.csv (제목만 보고 인용하지 말고 초록·본문을 읽으세요)")
    save(M, "C_claim_literature_matrix")
    keep = pd.DataFrame([("신규 단독 저자 유입 감소 (H1, FLOW)", "기여 1: 단독 연구 감소의 저자 단위 분해", "본문 핵심"), ("저자 정의 의존성 (H2)", "기여 2: 측정", "본문 핵심"), ("신규 파트너→진입 (H3)", "기여 3: 일반 탐색 메커니즘", "본문 지지"),
                         ("core 집중 코호트 대 외부 (H4)", "대비 결과: 코호트 특이", "본문 한 절"), ("해외 대학 포함 비중 증가 (H5)", "배경", "본문 한 문단"), ("편수 관성(IV 0.72), 거리 구성효과", "배경", "부록"),
                         ("특허-core 정렬, core 특이 철수, 도요타 점유 단독 서사", "기여 문장을 지지하지 않음", "삭제 또는 부록 '확인되지 않음'")], columns=["분석", "지지하는 기여 문장", "위치"])
    save(keep, "C_keep_drop_table"); print("\n[LIT-3] 분석별 위치 표:"); print(keep.to_string(index=False))
    h1 = lambda n: f"{fmt('H1', n)} pp [{fmt('H1', n, 'lo')}, {fmt('H1', n, 'hi')}]"
    md = f"""# 기여 문장 초안 (수치 자동 삽입, 문헌은 직접 채우세요)

## English
This paper decomposes the decline of corporate science in a fixed cohort of Japanese listed firms (patent window: published 2006-2015; papers 2006-2024) and replicates the findings outside the cohort.
(1) The decline in solo-authored output reflects fewer new solo-authoring corporate researchers, not conversion to co-authorship or faster exit: the share of new corporate authors who co-author solo papers fell by {h1('코호트')} in the cohort, {h1('외부1~40')} in 40 non-cohort firm groups, and {h1('보류41~80')} in a holdout of firm groups ranked 41-80.
(2) Conclusions about firm "leadership" in collaboration with foreign universities depend on the authorship definition: the change in external last-author share is {fmt('H2','코호트')} pp (cohort), {fmt('H2','외부1~40')} pp (non-cohort), versus {fmt('H2','코호트','est_first')} and {fmt('H2','외부1~40','est_first')} pp for first-author.
(3) New topic entry coincides with new partners (OR {fmt('H3','코호트',nd=2)} cohort; {fmt('H3','외부1~40',nd=2)} non-cohort; {fmt('H3','보류41~80',nd=2)} holdout).
(4) Concentration of core GX output on one firm (HHI change {fmt('H4','코호트',nd=3)}) is cohort-specific (non-cohort {fmt('H4','외부1~40',nd=3)}; holdout {fmt('H4','보류41~80',nd=3)}).
Relation to prior work: [문헌 확인: Arora et al.; Wuchty et al.; Ahmadpoor & Jones; Wu et al.] This study differs by [작성: 저자 단위 흐름 분해, 코호트 밖 재현].

## 한국어 요약
고정 코호트에서 단독 연구의 감소는 신규 단독 저자 충원 감소에서 나왔고 코호트 밖에서도 재현된다. 협력의 '기업 주도권'은 저자 정의에 따라 달라진다. 집중은 코호트 특이이다.
단, 위 수치는 이번 실행의 추정이며 보류 표본 지지 여부는 B_holdout_results.csv의 '지지' 열로 판단한다.
"""
    open(f"{FIN}/C_contribution_draft.md", "w", encoding="utf-8").write(md); print("[LIT-4] 기여 문장 초안 저장: C_contribution_draft.md")
safe(LIT)

# ============================ 보조: 출원번호 중복 분할계수 (FRAC) ============================
def FRAC():
    a = pd.read_excel("Japanese GX patents_with Englishi title.xlsx"); a["py"] = pd.to_datetime(a["Publication date"], errors="coerce").dt.year
    w = a[a.py.between(2006, 2015)]; ak = "Application number"; u = w[[ak, "corporate"]].drop_duplicates(); nf = u.groupby(ak)["corporate"].transform("size")
    whole = u.groupby("corporate").size(); frac = (1 / nf).groupby(u["corporate"]).sum()
    print(f"[FRAC] 고유 출원 {u[ak].nunique()} | 둘 이상 기업에 걸린 출원 {int((u.groupby(ak).size() > 1).sum())}")
    for nm, s in [("전부 계산", whole), ("분할계수", frac)]:
        p = s / s.sum(); print(f"  {nm:<8} 도요타 {p.get('トヨタ自動車', 0):.3f} | 상위4 {p.nlargest(4).sum():.3f} | HHI {(p**2).sum():.3f}")
    print("  도요타 점유 차이가 1~2pp 안이면 '중복은 결과에 영향이 없다'로 한 문장 처리")
safe(FRAC)

# ============================ 보조: 평가자 2 (RATER2) ============================
def core_conc(cmap):
    d = Ud.copy(); d["c"] = d["topic"].map(cmap).fillna(d["cls_f"]); t = d[d.c == "core"].groupby(["corp_norm", "blk"]).size().unstack(fill_value=0).reindex(columns=BL, fill_value=0)
    out = {}
    for lab, k in [("첫", BL[0]), ("끝", b1)]:
        c = t[k][t[k] > 0]; s = c / c.sum(); x = c[c.index != TOY]; y = c[~c.index.isin([TOY, NIS])]
        out[lab] = dict(도요타점유=float(s.get(TOY, 0)), HHI=float((s ** 2).sum()), HHI_도요타제외=float(((x / x.sum()) ** 2).sum()), HHI_도요타닛산제외=float(((y / y.sum()) ** 2).sum()))
    return pd.DataFrame(out).T
def RATER2():
    if not os.path.exists(RATER2_FILE): print(f"[RATER2] {RATER2_FILE} 없음: 평가자 2가 gx_review_rater2_blank.csv를 채워 이 경로로 저장하면 κ와 재산출을 수행"); return
    from sklearn.metrics import cohen_kappa_score
    M = pd.read_csv(f"{OUT}/gx_rater_merged.csv")[["topic", "mine", "rater", "n_total"]].rename(columns={"mine": "r0", "rater": "r1"})
    R2 = pd.read_csv(RATER2_FILE)[["topic", "gx_class_final"]].rename(columns={"gx_class_final": "r2"}); R2["r2"] = R2.r2.astype(str).str.strip().str.lower().replace({"non-gx": "non_gx"})
    M = M.merge(R2, on="topic"); M = M[M.r2.isin(CLS5) & M.r1.isin(CLS5)]; M["r0"] = M.r0.replace({"unreviewed": "other"})
    for x, y in [("r1", "r2"), ("r0", "r2")]:
        print(f"[RATER2] {x} vs {y}: 5범주 κ {cohen_kappa_score(M[x], M[y], labels=CLS5):.3f} | core 이진 κ {cohen_kappa_score(M[x]=='core', M[y]=='core'):.3f} | 논문수 가중 {cohen_kappa_score(M[x]=='core', M[y]=='core', sample_weight=M.n_total):.3f}")
    cd = M[(M.r1 == "core") != (M.r2 == "core")].sort_values("n_total", ascending=False); print(f"  core 여부 불일치 {len(cd)}개 (논문 {int(cd.n_total.sum())}편)"); print(cd.head(10).to_string(index=False))
    for lab, cmap in [("평가자1", dict(zip(M.topic, M.r1))), ("평가자2", dict(zip(M.topic, M.r2)))]: print(f"\n  core 집중 | {lab}"); print(core_conc(cmap).round(3).to_string())
safe(RATER2)
print(f"\n[완료] 표·그림·초안은 {FIN}/ 에 저장. 붙여 주실 것: [HOLDOUT-1], [HOLDOUT-2], [SPEC], [FLOW], [LIT-1], [FRAC], [RATER2]")

In [ ]:
#

In [ ]:
def reblock(df):
    d = df[df.year.between(2009, 2024)].copy()
    d["blk"] = (d.year - 2009) // 4          # 2009-12, 2013-16, 2017-20, 2021-24 (모두 4년)
    return d

_BL, _LAB = BL, LAB
BL = [0, 1, 2, 3]
LAB = {0: "2009-12", 1: "2013-16", 2: "2017-20", 3: "2021-24"}
try:
    for name in ["코호트", "외부1~40", "보류41~80"]:
        T, CI = flow_table(reblock(SAMPLES[name]))
        print(f"\n[FLOW 4년 창] {name}"); print(T.to_string(index=False)); print(CI.round(3).to_string(index=False))
finally:
    BL, LAB = _BL, _LAB                      # 전역 복원

In [ ]:
#피드백을 기반으로 코드를 제시해주세요. H1 신규 단독 참여가 비지지(방향 일치)인데 이떤 측면에서 이런 현상이 강한지, H2 마지막−제1저자가 비지지인데 세부적으로 분해해도 모두 비지지인지 H5 해외 대학 포함 비중은 모든 해외대학이 그런건지 국가별로 다른건지등을 알고 싶어요. 이를 모두 반영한 풀코드를 주세요.

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import os, re, warnings, traceback
from collections import Counter
import numpy as np, pandas as pd, statsmodels.api as sm, statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 300)

# ============================ 설정 ============================
NB = 500; SEED = 0
MIN_PAPERS = 30; MIN_FIRMS = 5; MIN_AUTH = 100; TOPC = 10
need = ["U", "LK", "AUTH", "classify", "PART", "BL", "b0", "b1", "LAB", "NY", "YLAST", "YMIN", "BLK", "OUT",
        "O0", "GROUPS_EXT1", "GROUPS_HOLD", "group_maps", "load_cache", "CACHE1", "CACHE2", "tid", "short", "TCLS"]
miss = [v for v in need if v not in globals()]; assert not miss, f"이전 셀 변수가 없습니다: {miss}"
TOY = globals().get("TOY", "トヨタ自動車")
FIN = f"{OUT}/final"; os.makedirs(FIN, exist_ok=True)
def save(df, name): df.to_csv(f"{FIN}/{name}.csv", index=False, encoding="utf-8-sig")
def safe(f):
    try: f()
    except Exception as e:
        print(f"\n[{f.__name__} 실패] {type(e).__name__}: {str(e)[:250]}"); traceback.print_exc(limit=2)
Ud = U[U.year <= YLAST].copy()
SN = {"coh": "코호트", "ext": "외부1~40", "hold": "보류41~80"}

# ---------- 국가 그룹 ----------
EU = {"DE", "FR", "GB", "IT", "ES", "NL", "CH", "SE", "DK", "FI", "NO", "BE", "AT", "IE", "PL", "CZ", "PT", "GR", "HU"}
ASIA = {"KR", "TW", "SG", "IN", "TH", "MY", "ID", "VN", "HK", "PH", "PK", "BD"}
GR = ["US", "CN", "EU", "AsiaOther", "Other"]
def cgrp(c): return "US" if c == "US" else "CN" if c == "CN" else "EU" if c in EU else "AsiaOther" if c in ASIA else "Other"

# ============================ 데이터 프레임 구축 (호출 없음) ============================
_t = U.dropna(subset=["topic_id"]).copy(); _t["t"] = _t.topic_id.map(tid)
TDOM = _t.groupby("t").domain.agg(lambda s: s.mode().iat[0]).to_dict()

def coh_frame():
    L = LK.copy()
    if "w" not in L.columns: L["w"] = L["work_id"].map(short)
    if "iid" not in L.columns: L["iid"] = L["inst_id"].astype(str).map(short)
    E = L[(L.itype == "education") & (L.country != "JP") & L.country.notna()]
    fu = {k: (set(g.country), set(g.iid)) for k, g in E.groupby(["w", "corp_norm"])}
    rows = []
    for r in Ud.itertuples(index=False):
        cl = classify(r.w, r.corp_norm); auth = [x[0] for x in cl if x[4] and x[0]] if cl else []
        ctry, inst = fu.get((r.w, r.corp_norm), (set(), set()))
        rows.append(dict(w=r.w, firm=r.corp_norm, year=int(r.year), blk=int(r.blk), topic=str(r.topic_id), cls=r.cls_f, dom=TDOM.get(tid(r.topic_id), "NA"),
                         solo=int(r.solo), foruniv=int(bool(r.for_univ)), last_ext=int(not bool(r.last_firm)), first_ext=int(not bool(r.first_firm)),
                         n_auth=len(AUTH.get(r.w) or []), ctry=ctry, inst=inst, authors=auth))
    return pd.DataFrame(rows)

def ext_frame(seen, IDS, INST2G):
    rows = []
    for w, r in seen.items():
        if r["topic"] not in TCLS or r["year"] > YLAST or not r["au"]: continue
        for g in {INST2G[i] for a in r["au"] for i in a[1] if i in INST2G}:
            own = IDS[g]; others = {i for a in r["au"] for i in a[1] if i not in own}
            fu = [(i, c) for a in r["au"] for i, t, c in zip(a[1], a[2], a[3]) if i not in own and t == "education" and c and c != "JP"]
            last, first = r["au"][-1], r["au"][0]
            rows.append(dict(w=w, firm=g, year=r["year"], blk=(r["year"] - YMIN) // BLK, topic=r["topic"], cls=TCLS[r["topic"]], dom=TDOM.get(r["topic"], "NA"),
                             solo=int(len(others) == 0), foruniv=int(len(fu) > 0), last_ext=int(not (set(last[1]) & own)), first_ext=int(not (set(first[1]) & own)),
                             n_auth=len(r["au"]), ctry={c for _, c in fu}, inst={i for i, _ in fu}, authors=[a[0] for a in r["au"] if a[0] and (set(a[1]) & own)]))
    return pd.DataFrame(rows)

I1, M1 = group_maps(GROUPS_EXT1); I2, M2 = group_maps(GROUPS_HOLD)
FR = {"coh": coh_frame(), "ext": ext_frame(load_cache(CACHE1), I1, M1), "hold": ext_frame(load_cache(CACHE2), I2, M2)}
cnt = Counter()
for d in FR.values():
    for s in d[d.foruniv == 1].ctry: cnt.update(s)
TOPCTRY = [c for c, _ in cnt.most_common(TOPC)]
def add_cols(d):
    d = d.copy(); c = d.groupby("firm").size(); q = pd.qcut(c.rank(method="first"), 3, labels=["small", "mid", "large"]); d["size"] = d.firm.map(q.to_dict())
    for g in GR: d["c_" + g] = d.ctry.map(lambda s, g=g: int(any(cgrp(x) == g for x in s)))
    for c_ in TOPCTRY: d["k_" + c_] = d.ctry.map(lambda s, c_=c_: int(c_ in s))
    d["abin"] = pd.cut(d.n_auth, [0, 2, 3, 5, 10, 10 ** 6], labels=["2", "3", "4-5", "6-10", "11+"])
    return d
FR = {k: add_cols(v) for k, v in FR.items()}
print("[0] 프레임:", {SN[k]: (len(v), v.firm.nunique(), int(v.foruniv.sum())) for k, v in FR.items()}, "(논문x기업, 기업, 해외대학 포함 논문)")
for k, d in FR.items():
    f = d[d.foruniv == 1]; print(f"   {SN[k]}: 해외대학 포함 논문 중 국가 정보 있음 {np.mean([len(s) > 0 for s in f.ctry]):.3f} | 저자 수 중앙값 {f.n_auth.median():.0f} | 상위 국가 {Counter([c for s in f.ctry for c in s]).most_common(5)}")
print("   개별 국가 분석 대상:", TOPCTRY)

HRES = []
def rec(mod, sn, stratum, r, **kw):
    if r is None: return
    HRES.append(dict(모듈=mod, 표본=SN.get(sn, sn), 층=stratum, **r, **kw))
def cl_ols(formula, d, key="firm"):
    d = d.reset_index(drop=True); mod = smf.ols(formula, data=d, missing="drop")
    return mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.loc[mod.data.row_labels, key])[0]})
def lpm_post(d, y, a=b0, z=b1, fe=True):
    d = d[d.blk.isin([a, z])].copy(); d["post"] = (d.blk == z).astype(int)
    if d.firm.nunique() < MIN_FIRMS or (d.post == 0).sum() < MIN_PAPERS or (d.post == 1).sum() < MIN_PAPERS: return None
    m = cl_ols(f"{y} ~ post" + (" + C(firm)" if fe else ""), d); b, s = m.params["post"], m.bse["post"]
    return dict(est=100 * b, lo=100 * (b - 1.96 * s), hi=100 * (b + 1.96 * s), p=m.pvalues["post"], n=int(m.nobs), firms=d.firm.nunique())
def strata_list(d, extra=True):
    out = [("core", d[d.cls == "core"]), ("non-core", d[d.cls != "core"])]
    for k in ["small", "mid", "large"]: out.append((f"규모 {k}", d[d["size"] == k]))
    for k in sorted(d.dom.unique()): out.append((f"분야 {k}", d[d.dom == k]))
    return out

# ---------- Kitagawa 기업 내·기업 간 분해 (배열 기반) ----------
def firm_arr(d, A, Z, xcol):
    t = d[d.blk.isin(A + Z)].copy(); t["g"] = t.blk.isin(Z).astype(int)
    g = t.groupby(["firm", "g"]).agg(n=(xcol, "size"), x=(xcol, "sum")).unstack(fill_value=0)
    n = g["n"].reindex(columns=[0, 1], fill_value=0).values.astype(float); x = g["x"].reindex(columns=[0, 1], fill_value=0).values.astype(float)
    return list(g.index), n[:, 0], x[:, 0], n[:, 1], x[:, 1]
def kit_arr(n0, x0, n1, x1):
    N0, N1 = n0.sum(), n1.sum()
    if N0 == 0 or N1 == 0: return (np.nan,) * 3, None
    w0, w1 = n0 / N0, n1 / N1
    r0 = np.divide(x0, n0, out=np.full(len(n0), np.nan), where=n0 > 0); r1 = np.divide(x1, n1, out=np.full(len(n1), np.nan), where=n1 > 0)
    r0f = np.nan_to_num(np.where(n0 > 0, r0, r1)); r1f = np.nan_to_num(np.where(n1 > 0, r1, r0))
    btw = (w1 - w0) * (r0f + r1f) / 2; wth = (w0 + w1) / 2 * (r1f - r0f)
    return (x1.sum() / N1 - x0.sum() / N0, btw.sum(), wth.sum()), btw + wth
def kit_boot(firms, n0, x0, n1, x1):
    pt, contrib = kit_arr(n0, x0, n1, x1); rng = np.random.default_rng(SEED); bs = []
    for _ in range(NB):
        ix = rng.integers(0, len(n0), len(n0)); t, _ = kit_arr(n0[ix], x0[ix], n1[ix], x1[ix])
        if not np.isnan(t[0]): bs.append(t)
    bs = np.array(bs); lo, hi = np.percentile(bs, 2.5, axis=0), np.percentile(bs, 97.5, axis=0)
    return pt, lo, hi, (pd.Series(contrib, index=firms) if contrib is not None else None)

# ============================ H5: 해외 대학 포함 비중 ============================
def H5():
    # (a) 재현 + 층별
    for sn, d in FR.items():
        rec("H5a 전체·층별", sn, "전체", lpm_post(d, "foruniv"))
        for lab, sub in strata_list(d): rec("H5a 전체·층별", sn, lab, lpm_post(sub, "foruniv"))
    # (b) 기업 내·기업 간, 진입 vs 강도
    rows = []
    for sn, d in FR.items():
        firms, n0, x0, n1, x1 = firm_arr(d, [b0], [b1], "foruniv"); pt, lo, hi, contrib = kit_boot(firms, n0, x0, n1, x1)
        both = (n0 >= 5) & (n1 >= 5); any0 = (x0[both] > 0).mean(); any1 = (x1[both] > 0).mean()
        i0 = (x0[both] / n0[both])[x0[both] > 0].mean() if (x0[both] > 0).any() else np.nan; i1 = (x1[both] / n1[both])[x1[both] > 0].mean() if (x1[both] > 0).any() else np.nan
        rows.append(dict(표본=SN[sn], 총변화pp=100 * pt[0], 기업간_구성pp=100 * pt[1], 구성_lo=100 * lo[1], 구성_hi=100 * hi[1], 기업내pp=100 * pt[2], 내_lo=100 * lo[2], 내_hi=100 * hi[2],
                         양쪽활동기업=int(both.sum()), 해외대학경험기업비중_첫=any0, 비중_끝=any1, 경험기업내_강도_첫=i0, 강도_끝=i1))
        top = contrib.sort_values(ascending=False).head(4) if contrib is not None else None
        print(f"   [H5b] {SN[sn]} 증가 기여 상위 기업(pp): {(100 * top).round(2).to_dict()}")
    T = pd.DataFrame(rows); print("\n[H5b] 증가의 구성: 기업 내(강도) vs 기업 간(활동 기업 구성 이동). 경험기업=해당 블록에 해외대학 논문이 있는 기업"); print(T.round(3).to_string(index=False)); save(T, "H5b_within_between")
    # (c) 국가 그룹·개별 국가
    rows = []
    for sn, d in FR.items():
        tot = lpm_post(d, "foruniv")
        for g in GR + TOPCTRY:
            col = ("c_" + g) if g in GR else ("k_" + g); r = lpm_post(d, col)
            if r is None: continue
            f0 = d[(d.blk == b0) & (d.foruniv == 1)]; f1 = d[(d.blk == b1) & (d.foruniv == 1)]
            r.update(국가=g, 해외대학논문중_첫=f0[col].mean() if len(f0) else np.nan, 해외대학논문중_끝=f1[col].mean() if len(f1) else np.nan,
                     전체증가대비=r["est"] / tot["est"] if tot and tot["est"] != 0 else np.nan, 표본=SN[sn]); rows.append(r); rec("H5c 국가", sn, g, {k: r[k] for k in ["est", "lo", "hi", "p", "n", "firms"]})
    T = pd.DataFrame(rows)[["표본", "국가", "est", "lo", "hi", "p", "해외대학논문중_첫", "해외대학논문중_끝", "전체증가대비"]]
    print("\n[H5c] 국가별: 전체 논문 중 해당 국가 대학이 포함된 비중의 변화(pp, 기업 FE). 한 논문이 여러 국가를 가지면 중복 계산되어 '전체증가대비'의 합은 100%를 넘을 수 있음")
    print(T.round(3).to_string(index=False)); save(T, "H5c_by_country")
    # (d) 국가 간 이질성 Wald
    for sn, d in FR.items():
        for lab, groups in [("국가 그룹", GR), ("상위 개별국가", TOPCTRY)]:
            parts = []
            for g in groups:
                col = ("c_" + g) if g in GR else ("k_" + g); t = d[d.blk.isin([b0, b1])][["firm", "blk"]].copy(); t["y"] = d.loc[t.index, col].values; t["grp"] = g; parts.append(t)
            S = pd.concat(parts).reset_index(drop=True); S["post"] = (S.blk == b1).astype(int)
            try:
                m = cl_ols("y ~ post*C(grp) + C(firm)", S); names = [n for n in m.params.index if n.startswith("post:C(grp)")]
                R = np.zeros((len(names), len(m.params))); idx = list(m.params.index)
                for i, n in enumerate(names): R[i, idx.index(n)] = 1
                print(f"[H5d] {SN[sn]} {lab} 간 증가폭 이질성 Wald p = {float(m.wald_test(R, scalar=True).pvalue):.3f}")
            except Exception as e: print(f"[H5d] {SN[sn]} {lab} 실패 {type(e).__name__}")
    # (e) 파트너 기관 집중도
    rows = []
    for sn, d in FR.items():
        seen = set()
        for b in BL:
            f = d[(d.blk == b) & (d.foruniv == 1)]; c = Counter(i for s in f.inst for i in s); links = sum(c.values())
            if links == 0: continue
            sh = np.array(sorted(c.values(), reverse=True)) / links; cont = sum(v for i, v in c.items() if i in seen) / links
            rows.append(dict(표본=SN[sn], 블록=LAB[b], 링크=links, 기관수=len(c), 상위5점유=sh[:5].sum(), HHI=(sh ** 2).sum(), 이전블록에도_있던_기관비중=cont if seen else np.nan)); seen |= set(c)
    T = pd.DataFrame(rows); print("\n[H5e] 해외 대학 파트너 기관 집중도(논문x기관 링크 기준)"); print(T.round(3).to_string(index=False)); save(T, "H5e_partner_concentration")
safe(H5)

# ============================ H2: 저자 정의 차이의 분해 ============================
CELLS = ["A 기업제1·외부마지막", "B 외부제1·기업마지막", "C 둘다외부(기업 중간)", "D 둘다기업"]
def h2_arrays(d, a=b0, z=b1):
    d = d[(d.blk.isin([a, z])) & (d.foruniv == 1)].copy()
    if d.empty: return None
    d["A"] = ((d.first_ext == 0) & (d.last_ext == 1)).astype(int); d["B"] = ((d.first_ext == 1) & (d.last_ext == 0)).astype(int)
    d["Cc"] = ((d.first_ext == 1) & (d.last_ext == 1)).astype(int); d["Dd"] = ((d.first_ext == 0) & (d.last_ext == 0)).astype(int); d["one"] = 1
    g = d.groupby(["firm", "blk"])[["one", "last_ext", "first_ext", "A", "B", "Cc", "Dd"]].sum().unstack(fill_value=0)
    return {k: g[k].reindex(columns=[a, z], fill_value=0).values.astype(float) for k in ["one", "last_ext", "first_ext", "A", "B", "Cc", "Dd"]}
def h2_stat(A):
    n0, n1 = A["one"][:, 0].sum(), A["one"][:, 1].sum()
    if n0 == 0 or n1 == 0: return np.full(7, np.nan)
    ch = lambda k: A[k][:, 1].sum() / n1 - A[k][:, 0].sum() / n0
    dl, df = ch("last_ext"), ch("first_ext"); return np.array([dl, df, dl - df, ch("A"), ch("B"), ch("Cc"), ch("Dd")])
def h2_boot(A):
    pt = h2_stat(A); rng = np.random.default_rng(SEED); F = len(A["one"]); bs = []
    for _ in range(NB):
        ix = rng.integers(0, F, F); t = h2_stat({k: v[ix] for k, v in A.items()})
        if not np.isnan(t[0]): bs.append(t)
    bs = np.array(bs); lo, hi = np.percentile(bs, 2.5, axis=0), np.percentile(bs, 97.5, axis=0)
    p = max(min(1, 2 * min((bs[:, 2] <= 0).mean(), (bs[:, 2] >= 0).mean())), 1 / (len(bs) + 1))
    return pt, lo, hi, p
def h2_stratum(d):
    A = h2_arrays(d)
    if A is None or A["one"][:, 0].sum() < MIN_PAPERS or A["one"][:, 1].sum() < MIN_PAPERS or ((A["one"].sum(1)) > 0).sum() < MIN_FIRMS: return None
    pt, lo, hi, p = h2_boot(A); ok = bool(lo[0] > 0 and lo[2] > 0)
    return dict(마지막Δ=100 * pt[0], 마지막_lo=100 * lo[0], 제1Δ=100 * pt[1], 차이Δ=100 * pt[2], 차이_lo=100 * lo[2], 차이_hi=100 * hi[2], p=p, 사전규칙통과=ok, n0=int(A["one"][:, 0].sum()), n1=int(A["one"][:, 1].sum()))
def H2():
    # (a) 4칸 분해
    rows = []
    for sn, d in FR.items():
        A = h2_arrays(d); pt, lo, hi, p = h2_boot(A)
        chk = abs((pt[0] - pt[1]) - (pt[3] - pt[4])) < 1e-9
        for j, nm in enumerate(["마지막 외부 비중 Δ", "제1 외부 비중 Δ", "차이(마지막-제1) Δ"] + CELLS):
            rows.append(dict(표본=SN[sn], 항목=nm, 점추정pp=100 * pt[j], lo=100 * lo[j], hi=100 * hi[j]))
        print(f"   [H2a] {SN[sn]} 항등식 (마지막Δ−제1Δ = A칸Δ − B칸Δ) 성립: {chk} | 해외대학 논문 {int(A['one'][:,0].sum())}→{int(A['one'][:,1].sum())}")
    T = pd.DataFrame(rows); print("\n[H2a] 해외대학 포함 논문의 저자 위치 4칸 구성 변화 (기업 재표집 95% 구간). 차이 = A칸 변화 − B칸 변화"); print(T.round(2).to_string(index=False)); save(T, "H2a_four_cells")
    # (b) 층별 판정
    rows = []
    for sn, d in FR.items():
        F = d[d.foruniv == 1]; strata = [("전체", F)]
        for k in ["2", "3", "4-5", "6-10", "11+"]: strata.append((f"저자수 {k}", F[F.abin == k]))
        strata += [("core", F[F.cls == "core"]), ("non-core", F[F.cls != "core"])]
        for k in ["small", "mid", "large"]: strata.append((f"규모 {k}", F[F["size"] == k]))
        for k in sorted(F.dom.unique()): strata.append((f"분야 {k}", F[F.dom == k]))
        for g in GR: strata.append((f"상대국 {g}", F[F["c_" + g] == 1]))
        for lab, sub in strata:
            r = h2_stratum(sub)
            if r is not None: rows.append(dict(표본=SN[sn], 층=lab, **r)); rec("H2b 층별", sn, lab, dict(est=r["차이Δ"], lo=r["차이_lo"], hi=r["차이_hi"], p=r["p"], n=r["n0"] + r["n1"], firms=np.nan))
    T = pd.DataFrame(rows); print("\n[H2b] 층별 (마지막Δ·제1Δ·차이Δ는 pp). 사전규칙통과 = 마지막 변화 하한>0 이고 차이 하한>0 (사전등록 H2와 같은 규칙)"); print(T.round(2).to_string(index=False)); save(T, "H2b_strata")
    for sn in SN.values():
        t = T[(T.표본 == sn) & (T.층 != "전체")]
        if len(t): print(f"   [H2b 요약] {sn}: 층 {len(t)}개 | 규칙 통과 {int(t.사전규칙통과.sum())}개 | 차이 점추정>0 {int((t.차이Δ > 0).sum())}개 | 차이 구간이 0 제외(양) {int((t.차이_lo > 0).sum())}개 (귀무에서 우연히 나올 기대치 약 {0.025*len(t):.1f}개)")
    # (c) 블록별 추이
    rows = []
    for sn, d in FR.items():
        F = d[d.foruniv == 1].groupby("blk").agg(n=("w", "size"), 마지막외부=("last_ext", "mean"), 제1외부=("first_ext", "mean")).reindex(BL)
        for b, r in F.iterrows(): rows.append(dict(표본=SN[sn], 블록=LAB[b], 해외대학논문=r.n, 마지막외부=r.마지막외부, 제1외부=r.제1외부, 차이=r.마지막외부 - r.제1외부))
    T = pd.DataFrame(rows); print("\n[H2c] 블록별 외부 주도 비중"); print(T.round(3).to_string(index=False)); save(T, "H2c_by_block")
    # (d) 한 기업씩 제외
    rows = []
    for sn, d in FR.items():
        F = d[d.foruniv == 1]; top = F[F.blk.isin([b0, b1])].groupby("firm").size().sort_values(ascending=False).head(5).index
        for f in ["(없음)"] + list(top):
            sub = F if f == "(없음)" else F[F.firm != f]; r = h2_stratum(sub)
            if r: rows.append(dict(표본=SN[sn], 제외기업=f, **{k: r[k] for k in ["마지막Δ", "제1Δ", "차이Δ", "차이_lo", "차이_hi", "p", "사전규칙통과"]}))
    T = pd.DataFrame(rows); print("\n[H2d] 해외대학 논문이 많은 상위 5개 기업을 한 곳씩 제외"); print(T.round(2).to_string(index=False)); save(T, "H2d_leave_one_firm_out")
safe(H2)

# ============================ H1: 신규 기업 저자 중 단독 참여 ============================
NEWA = {}
def author_table(d):
    rows = [(r.firm, a, r.blk, r.year, r.w, r.solo, r.foruniv, r.cls, r.dom, r.size) for r in d.itertuples(index=False) for a in set(r.authors)]
    A = pd.DataFrame(rows, columns=["firm", "aid", "blk", "year", "w", "solo", "fu", "cls", "dom", "size"]).sort_values(["firm", "aid", "blk", "year", "w"])
    ab = A.groupby(["firm", "aid", "blk"]).agg(solo=("solo", "max"), fu=("fu", "max"), cls=("cls", "first"), dom=("dom", "first"), size=("size", "first")).reset_index()
    fb = ab.groupby(["firm", "aid"]).blk.min().rename("fb").reset_index(); ab = ab.merge(fb, on=["firm", "aid"])
    N = ab[(ab.blk == ab.fb) & (ab.blk > b0)].copy(); N["cat"] = np.where(N.solo == 1, "단독", np.where(N.fu == 1, "해외대학 공저", "기타 공저")); return N
def h1_est(N, ref, fe=True):
    d = N[N.blk.isin(ref + [b1])].copy(); d["late"] = (d.blk == b1).astype(int)
    if d.firm.nunique() < MIN_FIRMS or (d.late == 0).sum() < MIN_AUTH // 2 or (d.late == 1).sum() < MIN_AUTH // 2: return None
    m = cl_ols("solo ~ late" + (" + C(firm)" if fe else ""), d); b, s = m.params["late"], m.bse["late"]
    return dict(est=100 * b, lo=100 * (b - 1.96 * s), hi=100 * (b + 1.96 * s), p=m.pvalues["late"], n=int(m.nobs), firms=d.firm.nunique())
REF = [x for x in BL[1:] if x != b1]
def H1():
    for sn, d in FR.items(): NEWA[sn] = author_table(d)
    print("[H1-0] 신규 기업 저자 수(블록별):", {SN[k]: v.groupby("blk").size().reindex(BL[1:]).tolist() for k, v in NEWA.items()})
    # (a) 층별 + 기준 변형
    for sn, N in NEWA.items():
        rec("H1a 층별", sn, "전체", h1_est(N, REF))
        for lab, sub in [("core", N[N.cls == "core"]), ("non-core", N[N.cls != "core"])] + [(f"규모 {k}", N[N["size"] == k]) for k in ["small", "mid", "large"]] + [(f"분야 {k}", N[N.dom == k]) for k in sorted(N.dom.unique())]:
            rec("H1a 층별", sn, lab, h1_est(sub, REF))
        top = N.groupby("firm").size().idxmax(); rec("H1a 층별", sn, f"최대기업 제외({top})", h1_est(N[N.firm != top], REF))
        rec("H1a 층별", sn, "기준 직전 블록만", h1_est(N, [BL[-2]])); rec("H1a 층별", sn, "기준 2011-15만", h1_est(N, [BL[1]])); rec("H1a 층별", sn, "기업 FE 없음", h1_est(N, REF, fe=False))
    # (b) 신규 저자의 유형 이동
    rows = []
    for sn, N in NEWA.items():
        sh = N.groupby(["blk", "cat"]).size().unstack(fill_value=0); sh = sh.div(sh.sum(1), axis=0)
        for cat in ["단독", "해외대학 공저", "기타 공저"]:
            d = N[N.blk.isin(REF + [b1])].copy(); d["late"] = (d.blk == b1).astype(int); d["y"] = (d.cat == cat).astype(int)
            m = cl_ols("y ~ late + C(firm)", d); b, s = m.params["late"], m.bse["late"]
            rows.append(dict(표본=SN[sn], 유형=cat, **{LAB[b_]: sh.loc[b_, cat] if (b_ in sh.index and cat in sh.columns) else np.nan for b_ in BL[1:]}, 변화pp=100 * b, lo=100 * (b - 1.96 * s), hi=100 * (b + 1.96 * s)))
    T = pd.DataFrame(rows); print("\n[H1b] 신규 기업 저자의 유형 구성(단독 > 해외대학 공저 > 기타 공저 순으로 상호배타)과 마지막 블록 변화(pp, 기업 FE)"); print(T.round(3).to_string(index=False)); save(T, "H1b_type_shift")
    # (c) 로그 분해: 신규 단독 저자 수 = 신규 저자 수 x 단독 비중 (연 환산)
    rows = []
    for sn, N in NEWA.items():
        firms, n0, x0, n1, x1 = firm_arr(N.assign(one=1), REF, [b1], "solo"); y0 = sum(NY[b] for b in REF); y1 = NY[b1]
        def lg(n0, x0, n1, x1):
            if min(n0.sum(), x0.sum(), n1.sum(), x1.sum()) <= 0: return np.full(3, np.nan)
            return np.array([np.log((x1.sum() / y1) / (x0.sum() / y0)), np.log((n1.sum() / y1) / (n0.sum() / y0)), np.log((x1.sum() / n1.sum()) / (x0.sum() / n0.sum()))])
        pt = lg(n0, x0, n1, x1); rng = np.random.default_rng(SEED); bs = np.array([lg(n0[i], x0[i], n1[i], x1[i]) for i in (rng.integers(0, len(n0), len(n0)) for _ in range(NB))]); bs = bs[~np.isnan(bs[:, 0])]
        lo, hi = np.percentile(bs, 2.5, axis=0), np.percentile(bs, 97.5, axis=0)
        for j, nm in enumerate(["신규 단독 저자 수(연) 변화", "  = 신규 저자 수(연) 변화", "  + 단독 비중 변화"]): rows.append(dict(표본=SN[sn], 항목=nm, ln변화=pt[j], 배율=np.exp(pt[j]), lo=np.exp(lo[j]), hi=np.exp(hi[j])))
    T = pd.DataFrame(rows); print("\n[H1c] 로그 분해 (마지막 블록 vs 기준 블록, 연 환산, 기업 재표집). 배율<1이면 감소"); print(T.round(3).to_string(index=False)); save(T, "H1c_log_decomposition")
    # (d) 기업 내·기업 간 + 기여 기업
    rows = []
    for sn, N in NEWA.items():
        firms, n0, x0, n1, x1 = firm_arr(N, REF, [b1], "solo"); pt, lo, hi, contrib = kit_boot(firms, n0, x0, n1, x1)
        rows.append(dict(표본=SN[sn], 총변화pp=100 * pt[0], 기업간_구성pp=100 * pt[1], 구성_lo=100 * lo[1], 구성_hi=100 * hi[1], 기업내pp=100 * pt[2], 내_lo=100 * lo[2], 내_hi=100 * hi[2]))
        c = contrib.sort_values(); print(f"   [H1d] {SN[sn]} 단독 비중 하락 기여 상위 기업(pp): {(100*c.head(5)).round(2).to_dict()} | 상승 기여: {(100*c.tail(2)).round(2).to_dict()}")
    T = pd.DataFrame(rows); print("\n[H1d] 단독 비중 변화의 기업 내(같은 기업의 신규 저자 구성 변화) vs 기업 간(기업별 신규 저자 비중 이동) 분해"); print(T.round(2).to_string(index=False)); save(T, "H1d_within_between")
safe(H1)

# ============================ HET: 보류 표본이 정말 다른가 (합동 상호작용) ============================
def HET():
    N = pd.concat([v.assign(sample=k, firm2=k + "|" + v.firm) for k, v in NEWA.items()], ignore_index=True)
    d = N[N.blk.isin(REF + [b1])].copy(); d["late"] = (d.blk == b1).astype(int); d["firm"] = d.firm2
    m = cl_ols("solo ~ late*C(sample) + C(firm2)", d, key="firm2"); rows = []
    for n in [n for n in m.params.index if n.startswith("late")]: rows.append(dict(모형="H1 신규 단독 참여", 항=n, 계수pp=100 * m.params[n], lo=100 * (m.params[n] - 1.96 * m.bse[n]), hi=100 * (m.params[n] + 1.96 * m.bse[n]), p=m.pvalues[n]))
    P = pd.concat([v.assign(sample=k, firm2=k + "|" + v.firm) for k, v in FR.items()], ignore_index=True); P = P[P.blk.isin([b0, b1])].copy(); P["post"] = (P.blk == b1).astype(int)
    m2 = cl_ols("foruniv ~ post*C(sample) + C(firm2)", P, key="firm2")
    for n in [n for n in m2.params.index if n.startswith("post")]: rows.append(dict(모형="H5 해외대학 포함", 항=n, 계수pp=100 * m2.params[n], lo=100 * (m2.params[n] - 1.96 * m2.bse[n]), hi=100 * (m2.params[n] + 1.96 * m2.bse[n]), p=m2.pvalues[n]))
    T = pd.DataFrame(rows); print("[HET] 합동 회귀(표본별 기업 FE). 기준=코호트(coh). 'late:C(sample)[T.hold]'가 보류 표본이 코호트와 다른 정도(pp). 구간이 0을 포함하면 '다르다는 증거 없음'"); print(T.round(3).to_string(index=False)); save(T, "HET_pooled_interaction")
safe(HET)

# ============================ 요약: 가설군 F7 (사후 이질성 탐색) ============================
def SUMMARY():
    R = pd.DataFrame(HRES); R["q_BH"] = np.nan
    for (mo, sm_), ix in R[R.층 != "전체"].groupby(["모듈", "표본"]).groups.items():
        p = R.loc[ix, "p"].fillna(1).clip(1e-300, 1).values
        if len(p): R.loc[ix, "q_BH"] = multipletests(p, method="fdr_bh")[1]
    save(R, "F7_heterogeneity_registry"); print("\n[요약] 모듈x표본별: 층 수, 추정이 기대 방향인 비율(H1·H2 음/양 구분), q<0.1인 층 수 (사후 탐색이므로 방향 일관성 중심으로 읽기)")
    rows = []
    for (mo, sm_), g in R[R.층 != "전체"].groupby(["모듈", "표본"]):
        expect_neg = mo.startswith("H1"); good = (g.est < 0) if expect_neg else (g.est > 0)
        rows.append(dict(모듈=mo, 표본=sm_, 층수=len(g), 기대방향_비율=good.mean(), 구간이0제외_기대방향=((g.hi < 0) if expect_neg else (g.lo > 0)).mean(), q_lt_0점1=int((g.q_BH < 0.1).sum())))
    print(pd.DataFrame(rows).round(3).to_string(index=False))
    print("  가설군 F7은 사전등록과 별개입니다. 보류 표본의 사전등록 판정(H1·H2·H5 비지지)은 이 결과로 바꾸지 마세요.")
safe(SUMMARY)
print(f"\n[완료] 표는 {FIN}/ 에 저장. 붙여 주실 것: [0], [H5b], [H5c], [H5d], [H5e], [H2a], [H2b] 및 요약, [H2d], [H1b], [H1c], [H1d], [HET], [요약]")

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import os, json, datetime, warnings, traceback
import numpy as np, pandas as pd, statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 300)

need = ["FR", "NEWA", "SN", "REF", "BL", "b0", "b1", "LAB", "OUT", "GR", "TOPCTRY", "HRES", "cl_ols", "lpm_post", "h1_est", "author_table", "strata_list", "MIN_FIRMS"]
miss = [v for v in need if v not in globals()]; assert not miss, f"이전 셀(코드 ⑲) 변수가 없습니다: {miss}"
FIN = f"{OUT}/final"; os.makedirs(FIN, exist_ok=True); NB2 = 1000; SEED2 = 1
def save(df, name): df.to_csv(f"{FIN}/{name}.csv", index=False, encoding="utf-8-sig")
def safe(f):
    try: f()
    except Exception as e:
        print(f"\n[{f.__name__} 실패] {type(e).__name__}: {str(e)[:250]}"); traceback.print_exc(limit=2)
def pct(v): return np.percentile(v, [2.5, 97.5])

# ============ X1. H5 표본 간 차이: 기업 단위 부트스트랩 ============
def post_diff(d, y):
    d = d[d.blk.isin([b0, b1])]
    g = d.groupby("blk")[y].mean()
    return np.nan if (b0 not in g.index or b1 not in g.index) else 100 * (g[b1] - g[b0])
def firm_boot_diff(dA, dB, y):
    rng = np.random.default_rng(SEED2); gA = {f: x for f, x in dA.groupby("firm")}; gB = {f: x for f, x in dB.groupby("firm")}; fa, fb = list(gA), list(gB); bs = []
    for _ in range(NB2):
        a = pd.concat([gA[fa[i]] for i in rng.integers(0, len(fa), len(fa))]); b = pd.concat([gB[fb[i]] for i in rng.integers(0, len(fb), len(fb))])
        t = post_diff(b, y) - post_diff(a, y)
        if not np.isnan(t): bs.append(t)
    return np.array(bs)
def X1():
    rows = []
    for y, lab in [("foruniv", "H5 해외대학 포함 비중(pp)")]:
        base = post_diff(FR["coh"], y)
        for k in ["ext", "hold"]:
            pt = post_diff(FR[k], y) - base; bs = firm_boot_diff(FR["coh"], FR[k], y); lo, hi = pct(bs)
            rows.append(dict(지표=lab, 비교=f"{SN[k]} − 코호트", 코호트Δ=base, 비교표본Δ=post_diff(FR[k], y), 차이=pt, lo=lo, hi=hi, 해석="0 포함: 다르다는 증거 없음" if lo <= 0 <= hi else "0 제외: 표본 간 차이 있음"))
    for k in ["ext", "hold"]:
        for nm, col in [("마지막 외부 Δ", "last_ext"), ("제1 외부 Δ", "first_ext")]:
            dA, dB = FR["coh"][FR["coh"].foruniv == 1], FR[k][FR[k].foruniv == 1]; base = post_diff(dA, col)
            pt = post_diff(dB, col) - base; bs = firm_boot_diff(dA, dB, col); lo, hi = pct(bs)
            rows.append(dict(지표=f"H2 {nm}(pp)", 비교=f"{SN[k]} − 코호트", 코호트Δ=base, 비교표본Δ=post_diff(dB, col), 차이=pt, lo=lo, hi=hi, 해석="0 포함: 다르다는 증거 없음" if lo <= 0 <= hi else "0 제외: 표본 간 차이 있음"))
    T = pd.DataFrame(rows); print("[X1] 표본 간 차이 (기업 단위 부트스트랩 1000회; 비교 표본 기업군 재표집과 코호트 기업 재표집을 독립 추출)"); print(T.round(2).to_string(index=False)); save(T, "X1_between_sample_diff")
safe(X1)

# ============ X2. 국가 간 이질성: 논문 단위 군집 ============
def X2():
    rows = []
    for sn, d in FR.items():
        d = d[d.blk.isin([b0, b1])].reset_index(drop=True); d["pid"] = d.w.astype(str)
        for lab, groups in [("국가 그룹", GR), ("상위 개별국가", TOPCTRY)]:
            parts = []
            for g in groups:
                col = ("c_" + g) if g in GR else ("k_" + g); t = d[["pid", "firm", "blk"]].copy(); t["y"] = d[col].values; t["grp"] = g; parts.append(t)
            S = pd.concat(parts, ignore_index=True); S["post"] = (S.blk == b1).astype(int)
            for cl_key, tag in [("pid", "논문 군집"), ("firm", "기업 군집")]:
                try:
                    mod = smf.ols("y ~ post*C(grp) + C(firm)", data=S); m = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(S[cl_key])[0]})
                    names = [n for n in m.params.index if n.startswith("post:C(grp)")]; R = np.zeros((len(names), len(m.params))); idx = list(m.params.index)
                    for i, n in enumerate(names): R[i, idx.index(n)] = 1
                    rows.append(dict(표본=SN[sn], 대상=lab, 군집=tag, wald_p=float(m.wald_test(R, scalar=True).pvalue)))
                except Exception as e: rows.append(dict(표본=SN[sn], 대상=lab, 군집=tag, wald_p=np.nan))
    T = pd.DataFrame(rows); print("\n[X2] 국가 간 증가폭 이질성 Wald p (논문 군집은 같은 논문의 국가 중복을 처리, 두 군집 모두 보고)"); print(T.round(3).to_string(index=False)); save(T, "X2_country_wald")
safe(X2)

# ============ X3. 파나소닉 군 제외 H1, 최대 기업 제외 ============
def X3():
    rows = []
    for sn in ["coh", "ext", "hold"]:
        N = NEWA[sn]; top = N.groupby("firm").size().sort_values(ascending=False).head(3).index.tolist()
        variants = [("전체", N)] + [(f"{f} 제외", N[N.firm != f]) for f in top]
        if sn == "ext": variants.append(("Panasonic群 제외", N[N.firm != "Panasonic群"])); variants.append(("Panasonic群·Denso 제외", N[~N.firm.isin(["Panasonic群", "Denso"])]))
        if sn == "coh": variants.append(("도요타·닛산 제외", N[~N.firm.isin(["トヨタ自動車", "日産自動車"])]))
        for lab, sub in variants:
            r = h1_est(sub, REF)
            if r: rows.append(dict(표본=SN[sn], 변형=lab, **{k: r[k] for k in ["est", "lo", "hi", "p", "n", "firms"]}))
    T = pd.DataFrame(rows); print("\n[X3] H1(신규 기업 저자 중 단독 비중 변화, pp, 기업 FE) 최대 기업 제외 민감도"); print(T.round(3).to_string(index=False)); save(T, "X3_h1_leave_out")
safe(X3)

# ============ X4. H1 층별 표 출력 ============
def X4():
    R = pd.DataFrame(HRES); R = R[R.모듈 == "H1a 층별"]
    if R.empty: print("\n[X4] HRES에 H1a 층별 결과가 없습니다(코드 ⑲ H1 모듈 먼저)"); return
    print("\n[X4] H1 층별 계수 (pp)"); print(R[["표본", "층", "est", "lo", "hi", "p", "n", "firms"]].round(3).to_string(index=False)); save(R, "X4_h1_strata")
    for s, g in R[R.층 != "전체"].groupby("표본"):
        print(f"   {s}: 층 {len(g)}개 | 음수 {int((g.est < 0).sum())}개 | 구간 0 제외 {int((g.hi < 0).sum())}개")
safe(X4)

# ============ X5. 최종 레지스트리 ============
PRE = [("H1 신규 단독 참여", "보류41~80", -8.333, 0.068), ("H2 마지막−제1저자", "보류41~80", 19.009, 0.062), ("H3 신규 파트너→진입 OR", "보류41~80", 2.580, 1e-6),
       ("H4 core HHI 변화(기준<0.198)", "보류41~80", 0.018, 0.002), ("H5 해외대학 포함 비중", "보류41~80", 0.493, 0.859)]
def X5():
    P = pd.DataFrame(PRE, columns=["가설", "표본", "추정", "p"]); P["가설군"] = "확증(사전 고정, 보류 표본)"; P["p_adj"] = multipletests(P.p, method="holm")[1]; P["보정"] = "Holm(5개)"
    P["판정(사전기준)"] = ["비지지", "비지지", "지지", "지지(느슨한 기준)", "비지지"]
    E = pd.DataFrame(HRES); E = E[E.층.notna()].copy(); E = E[E.p.notna()]; E["가설군"] = "F7 사후 이질성 탐색"; E["보정"] = "BH(모듈·표본별)"
    E["p_adj"] = np.nan
    for _, ix in E.groupby(["모듈", "표본"]).groups.items(): E.loc[ix, "p_adj"] = multipletests(E.loc[ix, "p"].clip(1e-300, 1), method="fdr_bh")[1]
    out = pd.concat([P.rename(columns={"추정": "est"}).assign(층="전체"), E.rename(columns={"모듈": "가설"})[["가설", "표본", "층", "est", "p", "p_adj", "가설군", "보정"]]], ignore_index=True)
    save(out, "X5_final_registry"); print(f"\n[X5] 레지스트리 저장: 확증 {len(P)}개(Holm), 사후 탐색 {len(E)}개(BH). 논문에는 확증 5개 표 + 탐색은 방향 일관성 요약만 싣고, 모형 장부(약 500개)는 부록 파일로 공개")
    print(P[["가설", "추정" if "추정" in P else "추정", "p", "p_adj", "판정(사전기준)"]].round(4).to_string(index=False))
safe(X5)

# ============ X6. 제출 전 체크리스트 ============
def X6():
    pre = f"{FIN}/PREREG_holdout_v1.json"; ok = os.path.exists(pre)
    h = json.load(open(pre, encoding="utf-8"))["sha256"] if ok else "없음"
    items = [
        ("자동 확인", f"PREREG 파일 존재: {ok} (sha256 {h[:12]}...)", ""),
        ("사람", "PREREG_holdout_v1.json을 OSF 또는 날짜가 남는 git 커밋으로 보관", "보류 표본 결과를 본 뒤 가설을 바꾸면 안 됨"),
        ("사람", "OpenAlex 계정에서 옛 API 키 폐기 확인, .gitignore에 .env, oa_out 추가", "git 이력에 키가 있으면 폐기가 이력 정리보다 우선"),
        ("사람", "시드 문헌 4편(Arora et al., Wuchty et al., Ahmadpoor & Jones, Wu et al.)을 직접 읽고 C_claim_literature_matrix.csv를 채움", "서지(Arora 연도 등)는 출판사 페이지로 확인"),
        ("사람", "평가자 2가 gx_review_rater2_blank.csv를 기준 먼저 작성 후 분류 → gx_review_rater2.csv 저장 → 코드 ⑱ RATER2 재실행", "평가자 1명이라는 한계 해소"),
        ("사람", "저자 Jiang에게 대형 출원인 누락 사유 문의(A_email_to_author.txt), 답이 없으면 '사유 미확인'으로 서술", ""),
        ("사람", "논문 본문: 위치는 '고정 코호트 + 비교·보류 표본', 특허는 처치가 아닌 표본 틀, GX는 분석 개념", "정책 효과로 읽히지 않게"),
        ("사람", "방법 절에 쓸 정제 이력: 공개 2006~2015, 고유 출원번호 19,802건(원본 20,523행), 분할계수 민감도, 출원 2016+ 연속성 98.6%", ""),
        ("사람", "보류 표본 5개 중 2개 지지·3개 비지지를 그대로 보고, H2·H5는 이질성 결과로 서술", "기준 사후 변경 금지"),
        ("사람", "쓰지 말 것 목록 재확인: 충원 붕괴, 도요타 단독 집중 일반화, 기업 주도권 상실, 특허-core 정렬, 2016년 전후 등", ""),
        ("사람", "그림 3장(신규 단독 저자 로그 분해, 저자 위치 4칸, core HHI 코호트 대 외부)을 A_figures.png에서 개별 패널로 정리", ""),
        ("사람", "다중검정: 확증 5개(Holm)와 사후 탐색(BH, 모형 장부 약 500개)을 구분해 기술", "")]
    T = pd.DataFrame(items, columns=["구분", "할 일", "비고"]); save(T, "X6_submission_checklist")
    print("\n[X6] 제출 전 체크리스트(파일로도 저장)"); print(T.to_string(index=False))
safe(X6)
print(f"\n[완료] 표는 {FIN}/ 에 저장. 붙여 주실 것: [X1], [X2], [X3], [X4], [X5]")

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import os, warnings, traceback
import numpy as np, pandas as pd, statsmodels.formula.api as smf
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

need = ["FR", "SN", "BL", "b0", "b1", "LAB", "OUT", "cl_ols"]
miss = [v for v in need if v not in globals()]; assert not miss, f"이전 셀 변수 없음: {miss}"
FIN = f"{OUT}/final"; os.makedirs(FIN, exist_ok=True); NB3 = 1000; RNG = np.random.default_rng(7)
def save(df, name): df.to_csv(f"{FIN}/{name}.csv", index=False, encoding="utf-8-sig")
def safe(f):
    try: f()
    except Exception as e:
        print(f"\n[{f.__name__} 실패] {type(e).__name__}: {str(e)[:250]}"); traceback.print_exc(limit=2)

def firm_table(d):
    """기업별: 첫/끝 블록 논문 수와 해외대학 포함 논문 수"""
    t = d[d.blk.isin([b0, b1])].groupby(["firm", "blk"]).agg(n=("foruniv", "size"), x=("foruniv", "sum")).unstack(fill_value=0)
    T = pd.DataFrame({"n0": t["n"].get(b0, 0), "x0": t["x"].get(b0, 0), "n1": t["n"].get(b1, 0), "x1": t["x"].get(b1, 0)}).fillna(0)
    T["size"] = T.n0 + T.n1
    T["r0"] = np.where(T.n0 > 0, T.x0 / T.n0, np.nan); T["r1"] = np.where(T.n1 > 0, T.x1 / T.n1, np.nan)
    return T
FT = {k: firm_table(v) for k, v in FR.items()}
def pooled_delta(T, w=None):
    """논문 가중 풀링 변화(pp). w: 기업별 가중치(기본 1)"""
    w = np.ones(len(T)) if w is None else np.asarray(w, float)
    n0, x0, n1, x1 = (T[c].values * w for c in ["n0", "x0", "n1", "x1"])
    return 100 * (x1.sum() / n1.sum() - x0.sum() / n0.sum()) if n0.sum() > 0 and n1.sum() > 0 else np.nan
def fe_delta(d):
    d = d[d.blk.isin([b0, b1])].copy(); d["post"] = (d.blk == b1).astype(int)
    if d.firm.nunique() < 5: return np.nan
    return 100 * cl_ols("foruniv ~ post + C(firm)", d).params["post"]

# ---------- 0. 기술통계: 세 표본이 무엇이 다른가 ----------
def Y0():
    rows = []
    for k, T in FT.items():
        both = T[(T.n0 > 0) & (T.n1 > 0)]
        rows.append(dict(표본=SN[k], 기업수=len(T), 양쪽활동기업=len(both), 기업당논문_중앙값=T["size"].median(), 첫블록_해외비중=T.x0.sum() / max(T.n0.sum(), 1), 끝블록_해외비중=T.x1.sum() / max(T.n1.sum(), 1),
                         기업별_첫비중_중앙값=T.r0.median(), 해외경험기업비중_첫=(T.x0 > 0).mean(), 해외경험기업비중_끝=(T.x1 > 0).mean(), 풀링변화pp=pooled_delta(T), FE변화pp=fe_delta(FR[k])))
    R = pd.DataFrame(rows); print("[Y0] 표본 비교"); print(R.round(3).to_string(index=False)); save(R, "Y0_sample_profile")
safe(Y0)

# ---------- 1. 규모 맞춤: 코호트 기업을 보류의 규모 분포로 재가중 ----------
def Y1():
    C_, H_ = FT["coh"], FT["hold"]; qs = np.quantile(H_["size"], [0, .25, .5, .75, 1.0]); qs[0] = 0; qs[-1] = max(qs[-1], C_["size"].max()) + 1
    C_ = C_.assign(bin=pd.cut(C_["size"], qs, labels=False, include_lowest=True)); H_ = H_.assign(bin=pd.cut(H_["size"], qs, labels=False, include_lowest=True))
    share = H_.bin.value_counts(normalize=True); cshare = C_.bin.value_counts(normalize=True)
    w = C_.bin.map(lambda b: share.get(b, 0) / cshare.get(b, np.nan)).fillna(0).values
    rows = [dict(구분="코호트(원래)", 변화pp=pooled_delta(C_)), dict(구분="코호트(보류 규모분포로 재가중)", 변화pp=pooled_delta(C_, w)), dict(구분="보류", 변화pp=pooled_delta(H_))]
    for b in sorted(C_.bin.dropna().unique()):
        rows.append(dict(구분=f"  규모구간{int(b)} 코호트(기업 {int((C_.bin==b).sum())})", 변화pp=pooled_delta(C_[C_.bin == b])))
        if (H_.bin == b).any(): rows.append(dict(구분=f"  규모구간{int(b)} 보류(기업 {int((H_.bin==b).sum())})", 변화pp=pooled_delta(H_[H_.bin == b])))
    R = pd.DataFrame(rows); print("\n[Y1] 규모 맞춤. 구간은 보류 기업 규모(첫+끝 논문 수)의 사분위. 같은 규모끼리 비교해도 차이가 남는가"); print(R.round(2).to_string(index=False)); save(R, "Y1_size_matching")
safe(Y1)

# ---------- 2. 기저 수준 층화 ----------
def Y2():
    rows = []
    for k in ["coh", "ext", "hold"]:
        T = FT[k]; T = T[(T.n0 >= 3) & (T.n1 >= 3)].copy()
        if len(T) < 6: continue
        T["bin"] = pd.cut(T.r0, [-0.001, 0.0001, 0.1, 1.01], labels=["기저 0", "기저 0~10%", "기저 10%+"])
        for b, g in T.groupby("bin"):
            if len(g) >= 3: rows.append(dict(표본=SN[k], 기저=b, 기업수=len(g), 첫비중=g.x0.sum() / g.n0.sum(), 끝비중=g.x1.sum() / g.n1.sum(), 변화pp=pooled_delta(g)))
    R = pd.DataFrame(rows); print("\n[Y2] 첫 블록 해외비중(기업별 기저)으로 나눈 변화. 기저를 맞추면 표본 간 차이가 줄어드는가 (논문 3편 이상인 기업만, 기저 층이 결과와 일부 순환적이므로 방향만 읽기)"); print(R.round(3).to_string(index=False)); save(R, "Y2_baseline_strata")
safe(Y2)

# ---------- 3. 동일 크기 재표집: 코호트에서 보류와 같은 수의 기업을 뽑으면? ----------
def Y3():
    C_ = FT["coh"]; hn = len(FT["hold"]); hd = pooled_delta(FT["hold"]); firms = list(C_.index); out = {"무작위": [], "규모비례": []}
    p = (C_["size"] / C_["size"].sum()).values
    for _ in range(NB3):
        out["무작위"].append(pooled_delta(C_.loc[RNG.choice(firms, hn, replace=False)]))
        out["규모비례"].append(pooled_delta(C_.loc[RNG.choice(firms, hn, replace=False, p=p)]))
    rows = []
    for k, v in out.items():
        v = np.array([x for x in v if not np.isnan(x)]); rows.append(dict(추출=k, 코호트_재표집_평균=v.mean(), p2점5=np.percentile(v, 2.5), p97점5=np.percentile(v, 97.5), 보류_실제=hd, 보류가_분포_하회비율=(v <= hd).mean()))
    R = pd.DataFrame(rows); print(f"\n[Y3] 코호트에서 기업 {hn}곳을 뽑아 풀링 변화를 계산한 분포(논문 크기는 그대로, 기업 수만 보류와 같게). 보류 실제값이 분포 하단 밖이면 '기업 수가 적어서'만으로는 설명되지 않음")
    print(R.round(3).to_string(index=False)); save(R, "Y3_equal_size_resample")
safe(Y3)

# ---------- 4. 코호트 대기업 제외 ----------
def Y4():
    d = FR["coh"]; T = FT["coh"]; top = T["size"].sort_values(ascending=False).index.tolist(); rows = [dict(구분="코호트 전체", 기업=len(T), 풀링pp=pooled_delta(T), FEpp=fe_delta(d))]
    for k in [1, 2, 3, 5, 10]:
        ex = top[:k]; rows.append(dict(구분=f"상위 {k}개 기업 제외({', '.join(ex[:3])}{'...' if k>3 else ''})", 기업=len(T) - k, 풀링pp=pooled_delta(T.drop(index=ex)), FEpp=fe_delta(d[~d.firm.isin(ex)])))
    ms = T[T["size"] <= FT["hold"]["size"].quantile(0.9)]; rows.append(dict(구분="코호트 중 보류 규모 상한(보류 90분위) 이하 기업만", 기업=len(ms), 풀링pp=pooled_delta(ms), FEpp=fe_delta(d[d.firm.isin(ms.index)])))
    rows.append(dict(구분="보류 전체", 기업=len(FT["hold"]), 풀링pp=pooled_delta(FT["hold"]), FEpp=fe_delta(FR["hold"])))
    R = pd.DataFrame(rows); print("\n[Y4] 코호트 대기업을 빼면 증가가 보류에 가까워지는가"); print(R.round(2).to_string(index=False)); save(R, "Y4_cohort_trim")
safe(Y4)

# ---------- 5. 균형 패널 + 보류 기업 이탈 ----------
def Y5():
    rows = []
    for k in ["coh", "ext", "hold"]:
        T = FT[k]; both = T[(T.n0 >= 3) & (T.n1 >= 3)]; d = FR[k][FR[k].firm.isin(both.index)]
        gone = T[(T.n0 > 0) & (T.n1 == 0)]; new = T[(T.n0 == 0) & (T.n1 > 0)]
        rows.append(dict(표본=SN[k], 전체기업=len(T), 균형패널기업=len(both), 끝블록_소멸기업=len(gone), 첫블록_없던신규기업=len(new), 균형_풀링pp=pooled_delta(both), 균형_FEpp=fe_delta(d),
                         소멸기업_첫블록_해외비중=gone.x0.sum() / max(gone.n0.sum(), 1), 신규기업_끝블록_해외비중=new.x1.sum() / max(new.n1.sum(), 1)))
    R = pd.DataFrame(rows); print("\n[Y5] 양쪽 블록에 3편 이상 있는 균형 패널과 기업 출입. 보류의 비증가가 활동 기업의 이탈·진입 때문인지 확인"); print(R.round(3).to_string(index=False)); save(R, "Y5_balanced_panel")
safe(Y5)

# ---------- 6. 합동 모형: 규모·기저 통제 후 보류 차이 ----------
def Y6():
    rows = []; parts = []
    for k, d in FR.items():
        T = FT[k]; d = d[d.blk.isin([b0, b1])].copy(); d["post"] = (d.blk == b1).astype(int); d["sample"] = k; d["firm2"] = k + "|" + d.firm
        d["lsize"] = np.log1p(d.firm.map(T["size"])); d["base"] = d.firm.map(T["r0"].fillna(0)); parts.append(d)
    P = pd.concat(parts, ignore_index=True)
    for lab, f in [("통제 없음", "foruniv ~ post*C(sample) + C(firm2)"), ("+규모 상호작용", "foruniv ~ post*C(sample) + post:lsize + C(firm2)"), ("+기저 상호작용", "foruniv ~ post*C(sample) + post:lsize + post:base + C(firm2)")]:
        m = cl_ols(f, P, key="firm2"); n = [x for x in m.params.index if x.startswith("post:C(sample)")]
        for x in n: rows.append(dict(모형=lab, 항=x, 계수pp=100 * m.params[x], lo=100 * (m.params[x] - 1.96 * m.bse[x]), hi=100 * (m.params[x] + 1.96 * m.bse[x])))
    R = pd.DataFrame(rows); print("\n[Y6] 합동 회귀: post×표본 계수가 코호트 대비 차이. 규모·기저 통제 후 줄어드는 정도로 원인 기여를 읽음 (구간은 기업 군집, 기저는 결과 변수와 일부 순환)"); print(R.round(2).to_string(index=False)); save(R, "Y6_pooled_controls")
safe(Y6)

# ---------- 7. 읽기 요약 ----------
def Y7():
    print("\n[Y7] 판정 틀 (실행 전 고정한 기준)")
    print(" - Y1: 재가중 코호트 변화가 보류(-0.4)에 가까워지면 규모가 원인, 여전히 +8pp 이상이면 규모는 원인 아님")
    print(" - Y3: 보류 실제값이 코호트 재표집 하위 2.5% 밖이면 기업 수만으로는 설명 안 됨")
    print(" - Y4: 상위 기업을 빼도 코호트 증가가 +6pp 이상이면 대기업 구성은 원인 아님")
    print(" - Y6: 통제로 post×보류 계수가 -11.6에서 -5 이내로 줄면 규모·기저가 원인의 대부분")
    print(" - 어느 것도 원인이 아니면 '코호트 대기업의 해외 협력 확대가 코호트 특이 현상'으로 서술하고 원인은 미확인으로 둠")
safe(Y7)
print(f"\n[완료] 표는 {FIN}/ 에 저장. 붙여 주실 것: [Y0]~[Y6]")

In [ ]:
#

In [ ]:
globals().pop("C", None)
from patsy.builtins import C
import os, warnings, traceback
import numpy as np, pandas as pd, statsmodels.formula.api as smf
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)

need = ["FR", "SN", "BL", "b0", "b1", "LAB", "OUT", "cl_ols"]
miss = [v for v in need if v not in globals()]; assert not miss, f"이전 셀 변수 없음: {miss}"
FIN = f"{OUT}/final"; os.makedirs(FIN, exist_ok=True); NBZ = 1000; RZ = np.random.default_rng(11)
def save(df, name): df.to_csv(f"{FIN}/{name}.csv", index=False, encoding="utf-8-sig")
def safe(f):
    try: f()
    except Exception as e:
        print(f"\n[{f.__name__} 실패] {type(e).__name__}: {str(e)[:250]}"); traceback.print_exc(limit=2)

D2 = {k: v[v.blk.isin([b0, b1])].copy() for k, v in FR.items()}
for k, d in D2.items(): d["post"] = (d.blk == b1).astype(int)

def cell_table(d):
    """분야x블록: 논문 수, 해외대학 포함 수"""
    g = d.groupby(["dom", "blk"]).agg(n=("foruniv", "size"), x=("foruniv", "sum")).unstack(fill_value=0)
    T = pd.DataFrame({"n0": g["n"].get(b0, 0), "x0": g["x"].get(b0, 0), "n1": g["n"].get(b1, 0), "x1": g["x"].get(b1, 0)}).fillna(0)
    return T
CT = {k: cell_table(d) for k, d in D2.items()}

# ---------- Z0. 분야 구성 ----------
def Z0():
    rows = []
    for k, T in CT.items():
        for dom, r in T.iterrows(): rows.append(dict(표본=SN[k], 분야=dom, 첫블록_논문비중=r.n0 / T.n0.sum(), 끝블록_논문비중=r.n1 / T.n1.sum(), 첫_해외비중=r.x0 / r.n0 if r.n0 else np.nan, 끝_해외비중=r.x1 / r.n1 if r.n1 else np.nan))
    R = pd.DataFrame(rows); print("[Z0] 분야 구성과 분야별 해외대학 포함 비중"); print(R.round(3).to_string(index=False)); save(R, "Z0_domain_composition")
    P = R.pivot(index="분야", columns="표본", values="끝블록_논문비중"); print("\n 끝 블록 분야 비중 차(보류−코호트, pp):"); print((100 * (P["보류41~80"] - P["코호트"])).round(1).sort_values().to_string())
safe(Z0)

# ---------- Z1. 분야 내 증가 ----------
def Z1():
    rows = []
    for k, T in CT.items():
        for dom, r in T.iterrows():
            if r.n0 >= 15 and r.n1 >= 15: rows.append(dict(표본=SN[k], 분야=dom, n0=int(r.n0), n1=int(r.n1), 변화pp=100 * (r.x1 / r.n1 - r.x0 / r.n0)))
    R = pd.DataFrame(rows); P = R.pivot(index="분야", columns="표본", values="변화pp")
    print("\n[Z1] 분야 내 해외대학 포함 비중 변화(pp, 두 블록 모두 15편 이상인 분야)"); print(P.round(1).to_string()); save(R, "Z1_within_domain")
    if {"코호트", "보류41~80"} <= set(P.columns):
        d = P.dropna(subset=["코호트", "보류41~80"]); print(f"\n 공통 분야 {len(d)}개 중 보류가 코호트보다 낮은 분야: {int((d['보류41~80'] < d['코호트']).sum())}개 | 평균 차이 {(d['보류41~80'] - d['코호트']).mean():.1f}pp")
safe(Z1)

# ---------- Z2. 구성 효과 vs 분야 내 효과 (코호트를 보류의 분야 구성으로) ----------
def decomp(Tc, Th, blk_w):
    """Tc, Th: 분야 표. 코호트 분야별 변화를 보류의 분야 가중(끝 블록 논문 비중 또는 평균 비중)으로 합성"""
    doms = Tc.index.intersection(Th.index); Tc, Th = Tc.loc[doms], Th.loc[doms]
    ok = (Tc.n0 > 0) & (Tc.n1 > 0) & (Th.n0 > 0) & (Th.n1 > 0); Tc, Th = Tc[ok], Th[ok]
    dc = Tc.x1 / Tc.n1 - Tc.x0 / Tc.n0; dh = Th.x1 / Th.n1 - Th.x0 / Th.n0
    wc = (Tc.n0 + Tc.n1) / (Tc.n0 + Tc.n1).sum(); wh = (Th.n0 + Th.n1) / (Th.n0 + Th.n1).sum()
    total = (wh * dh).sum() - (wc * dc).sum()
    comp = ((wh - wc) * (dc + dh) / 2).sum(); within = (((wh + wc) / 2) * (dh - dc)).sum()
    return 100 * total, 100 * comp, 100 * within, len(doms)
def Z2():
    Tc, Th = CT["coh"], CT["hold"]; pt = decomp(Tc, Th, None)
    # 부트스트랩: 두 표본의 기업을 각각 재표집
    gc = {f: x for f, x in D2["coh"].groupby("firm")}; gh = {f: x for f, x in D2["hold"].groupby("firm")}; fc, fh = list(gc), list(gh); bs = []
    for _ in range(NBZ):
        a = pd.concat([gc[fc[i]] for i in RZ.integers(0, len(fc), len(fc))]); b = pd.concat([gh[fh[i]] for i in RZ.integers(0, len(fh), len(fh))])
        t = decomp(cell_table(a), cell_table(b), None)
        if not np.isnan(t[0]): bs.append(t[:3])
    bs = np.array(bs); lo, hi = np.percentile(bs, 2.5, axis=0), np.percentile(bs, 97.5, axis=0)
    R = pd.DataFrame({"항목": ["총차이(보류Δ−코호트Δ)", "분야 구성 효과", "분야 내 효과"], "pp": pt[:3], "lo": lo, "hi": hi}); print(f"\n[Z2] 분야 구성 분해 (공통 분야 {pt[3]}개, 기업 재표집 {len(bs)}회, 두 표본 평균 분야 비중 사용)"); print(R.round(2).to_string(index=False)); save(R, "Z2_composition_decomposition")
    share = pt[1] / pt[0] if pt[0] != 0 else np.nan; print(f" 구성 효과 비중 = {share:.2f} (읽기 기준: 0.5 이상이면 구성이 주된 원인, 0.25 미만이면 원인 아님)")
safe(Z2)

# ---------- Z3. 합동 회귀: 분야 통제 ----------
def Z3():
    parts = []
    for k, d in D2.items():
        d = d.copy(); d["sample"] = k; d["firm2"] = k + "|" + d.firm; T = FT[k]; d["lsize"] = np.log1p(d.firm.map(T["size"])); parts.append(d)
    P = pd.concat(parts, ignore_index=True); rows = []
    specs = [("통제 없음", "foruniv ~ post*C(sample) + C(firm2)"), ("+규모", "foruniv ~ post*C(sample) + post:lsize + C(firm2)"),
             ("+분야 고정효과", "foruniv ~ post*C(sample) + post:lsize + C(dom) + C(firm2)"), ("+분야x post", "foruniv ~ post*C(sample) + post:lsize + post:C(dom) + C(firm2)"),
             ("+분야x post+core", "foruniv ~ post*C(sample) + post:lsize + post:C(dom) + post:C(cls) + C(firm2)")]
    for lab, f in specs:
        try:
            m = cl_ols(f, P, key="firm2")
            for x in [x for x in m.params.index if x.startswith("post:C(sample)")]: rows.append(dict(모형=lab, 항=x.replace("post:C(sample)", ""), 계수pp=100 * m.params[x], lo=100 * (m.params[x] - 1.96 * m.bse[x]), hi=100 * (m.params[x] + 1.96 * m.bse[x])))
        except Exception as e: rows.append(dict(모형=lab, 항="실패", 계수pp=np.nan, lo=np.nan, hi=np.nan))
    R = pd.DataFrame(rows); print("\n[Z3] post×표본 계수(코호트 대비). 분야 통제로 보류 계수가 0에 가까워지면 구성이 원인 (구간은 기업 군집)"); print(R.round(2).to_string(index=False)); save(R, "Z3_pooled_domain_controls")
safe(Z3)

# ---------- Z4. 같은 분야 내 논문 단위 비교 ----------
def Z4():
    rows = []
    for dom in sorted(set(D2["coh"].dom) & set(D2["hold"].dom)):
        c, h = D2["coh"][D2["coh"].dom == dom], D2["hold"][D2["hold"].dom == dom]
        if min(len(c[c.post == 0]), len(c[c.post == 1]), len(h[h.post == 0]), len(h[h.post == 1])) < 15: continue
        dc = 100 * (c[c.post == 1].foruniv.mean() - c[c.post == 0].foruniv.mean()); dh = 100 * (h[h.post == 1].foruniv.mean() - h[h.post == 0].foruniv.mean())
        rows.append(dict(분야=dom, 코호트Δ=dc, 보류Δ=dh, 차이=dh - dc, 코호트_n=len(c), 보류_n=len(h)))
    R = pd.DataFrame(rows); print("\n[Z4] 분야별 논문 단위 증가 비교"); print(R.round(1).to_string(index=False)); save(R, "Z4_domain_pairs")
    if len(R): print(f" 보류가 낮은 분야 {int((R.차이 < 0).sum())}/{len(R)} | 가중평균 차이 {np.average(R.차이, weights=R.보류_n):.1f}pp")
safe(Z4)

# ---------- Z5. 코호트 업종별 증가 (코호트에만 업종 정보) ----------
def Z5():
    if "FM" not in globals(): print("\n[Z5] FM(firm_master_window) 없음: 건너뜀"); return
    ind = FM["industry"] if "industry" in FM.columns else None
    if ind is None: print("\n[Z5] industry 열 없음"); return
    d = D2["coh"].copy(); d["ind"] = d.firm.map(ind); g = d.groupby(["ind", "post"]).foruniv.agg(["size", "mean"]).unstack()
    R = pd.DataFrame({"n0": g[("size", 0)], "n1": g[("size", 1)], "첫비중": g[("mean", 0)], "끝비중": g[("mean", 1)]}).dropna(); R["변화pp"] = 100 * (R.끝비중 - R.첫비중)
    R = R[(R.n0 >= 15) & (R.n1 >= 15)].sort_values("변화pp"); print("\n[Z5] 코호트 업종별 해외대학 포함 비중 변화(두 블록 15편 이상)"); print(R.round(3).to_string()); save(R.reset_index(), "Z5_cohort_industry")
    print(" 읽기: 업종 간 편차가 크고 보류 기업군이 증가가 낮은 업종에 몰려 있다면 업종 구성이 후보. 외부·보류 기업군에는 업종 매핑이 없어 직접 맞춤은 불가")
safe(Z5)

print(f"\n[완료] 표는 {FIN}/ 에 저장. 붙여 주실 것: [Z0]~[Z5]")

In [ ]:
#